# Small-Cap Swing Trading Engine v2 — NSE / Groww API

**A production-shaped screener + trade-recommendation + full-lifecycle engine for Indian small caps.**

---

## What this notebook is

An end-to-end research-and-execution scaffold:

```
 Groww instruments ─┐
                    ├─► Universe ─► Data Hub ─► Feature Engine ─► Strategy Bank ─┐
 Daily OHLCV       ─┘      (small-cap    (SQLite     (60+ vectorised     (4 edges)  │
                            filters)      cache)      indicators)                    │
                                                                                     ▼
  Telegram  ◄──────  Orchestrator  ◄──  Lifecycle FSM  ◄── Trade Planner ◄── Composite Score
  Gemini    ◄──────       │                  │                   ▲               + Regime filter
                          │                  │                   │               + Risk sizing
                          └──► Backtester ───┴───► Broker (Paper | Groww) ────────┘
```

## Design principles

1. **The broker is an interface, not a dependency.** `BrokerAdapter` has two implementations —
   `PaperBroker` (simulated fills, default) and `GrowwBroker` (live). Nothing downstream knows
   which one it is talking to. You can run the entire notebook with zero credentials.
2. **Signals are generated on closed bars only.** No look-ahead. Signals form at the close of day
   *t*; orders are worked on day *t+1*. The backtester enforces the same rule.
3. **The LLM never invents a price.** Gemini sits *beside* the quant core as a veto/annotation
   layer with a hard-bounded score adjustment (±10 pts). It cannot create a trade, size one,
   or move a stop. It fails open (neutral) on any error.
4. **Every trade is a state machine**, persisted in SQLite, resumable after a crash.
5. **Costs are modelled honestly** — STT, exchange charges, stamp duty, GST, DP charges,
   plus slippage that scales with the position's share of ADV. Small-cap edges die on costs.

## Groww API facts this code is built against

| Item | Detail |
|---|---|
| Package | `pip install growwapi pyotp` |
| Auth | TOTP-only: `GrowwAPI.get_access_token(api_key=..., secret=...)`, token expires **daily ~6 AM IST** |
| Orders | Requires a **registered static IP** (SEBI rule) or `place_order` is rejected |
| Market data | Requires the paid Trading-API subscription |
| Historical | The 1-day interval is **capped at 180 days per request** — longer ranges are rejected outright, so requests are chunked at 175 days |
| Key methods | `get_all_instruments`, `get_historical_candles`, `get_quote`, `get_ltp`, `place_order`, `modify_order`, `cancel_order`, `get_order_detail`, `get_holdings_for_user`, `get_positions_for_user` |

> **The history-depth constraint is the single most important architectural fact here.** A swing
> system needs 250+ daily bars per name for 200-DMA, 52-week-high and relative-strength context,
> and years of it for backtesting. So `DataHub` is a *chained* provider: local SQLite cache →
> Groww → a configurable long-history fallback (yfinance / your own bhavcopy CSVs) → synthetic.
> Groww stays the source of truth for quotes, orders and positions.

---

### ⚠️ Read this before you risk money

I am not a financial adviser and this is not investment advice. This is engineering scaffolding
and a research tool. It ships in `PAPER` mode with `dry_run=True` deliberately. Indian small caps
are illiquid, circuit-prone and subject to ASM/GSM surveillance framework restrictions; a
backtest on them is optimistic by construction (survivorship bias, no delisting/suspension data,
understated impact cost). Forward-test on paper for at least one full quarter, validate every
number independently, understand the tax treatment of your trades, and size positions on the
assumption that the model is wrong. Automated order placement carries regulatory obligations —
check SEBI's current algo-trading rules for retail participants before enabling `LIVE`.

> **v2 defaults:** `RUN_MODE='SMOKE'`, ₹50,000 equity, long-only CNC, `dry_run=True`, approval required. Research conclusions must come from purged walk-forward/OOS evidence; the final holdout stays locked until design choices are frozen.


## This revision — benchmark patch + end-to-end logging

**The crash.** Step 2 died with `IndexError: single positional indexer is out-of-bounds` inside
`assess_regime()`. The benchmark frame was empty because:

1. `CFG.benchmark = "NIFTY_SMLCAP_250"` was sent to Groww as `NSE-NIFTY_SMLCAP_250`, which is not a
   Groww instrument (indices use `<EXCHANGE>-<trading_symbol>` from the instrument master, e.g. `NSE-NIFTY`);
2. with `allow_synthetic_fallback=False`, `_fetch` swallowed the error and returned an empty frame;
3. the failure put the symbol into the persisted `DEAD_SYMBOLS` memo, so every later run skipped it silently;
4. `_candles_to_df` also rejected 5-column (volume-less) candle payloads — which is what index candles look like.

**The fix.** The benchmark is now *resolved* from the instrument master (`find_index_instruments`),
never dead-listed (and scrubbed from the memo on load), fetched via its exact `groww_symbol`,
validated for length and recency, and falls back to yfinance and finally to an equal-weight
universe proxy — each step logged. `compute_panel` refuses to continue with a bad benchmark, and
`assess_regime` explains the problem instead of throwing an `IndexError`. `preflight()` now checks
the benchmark too, so this class of problem surfaces in seconds rather than after a 30-minute build.

**Logging.** In Colab, `logging.basicConfig()` was a silent no-op, so none of the engine's log
lines were ever shown. Logging is now forced to stdout (inline under each cell) *and* to
`swing_engine.log`. Every cell logs `▶ START` / `✔ DONE (duration)` / `✖ FAILED (exception)`
automatically, and each stage logs what it did: gate-by-gate universe counts, cache vs provider
activity, benchmark resolution, feature/RS coverage, strategy hits, filter drop-outs, sizing
decisions, lifecycle events, backtest progress. Set `LOG_LEVEL=DEBUG` for per-request detail.

**Secrets.** API keys and tokens are no longer hard-coded. Put `GROWW_ACCESS_TOKEN` (or
`GROWW_API_KEY` + `GROWW_API_SECRET`), `TELEGRAM_BOT_TOKEN`, `TELEGRAM_CHAT_ID` and `GEMINI_API_KEY`
in Colab's **Secrets** panel (🔑 in the left sidebar, enable notebook access) or in environment variables.

**Also fixed.** The Step-5 lifecycle walkthrough crashed in `LIVE` mode (it called paper-only
methods on the Groww broker) and wrote a demo trade into the real DB — it now runs in a sandbox.
Walk-forward no longer crashes when the configured backtest window has no data.


## 0 · Install

Uncomment and run once. `growwapi`, `pyotp`, `google-genai` and `yfinance` are all optional —
the notebook degrades gracefully without them.

In [95]:
# [00] Install
# Colab-safe research/runtime dependencies. Re-running is harmless.
!pip install -q "protobuf<8.0.0" "pandas>=2.1,<3" "numpy>=1.26,<3" "scipy>=1.11,<2" \
    "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4" "joblib>=1.3,<2" \
    growwapi pyotp requests python-dotenv google-genai "yfinance>=0.2.40" pyarrow


07:48:52 | INFO    | engine.cell       | ▶ START  [00] Install
07:48:54 | INFO    | engine.cell       | ✔ DONE   [00] Install in 1.8s


## 1 · Imports & environment

In [96]:
# [01] Imports, logging & environment
from __future__ import annotations

import base64
import functools
import json
import logging
import math
import os
import random
import re
import sqlite3
import sys
import threading
import time
import uuid
import hashlib
import itertools
import copy
import warnings
from pathlib import Path
from collections import defaultdict, deque
from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
from datetime import datetime, date, time as dtime, timedelta, timezone
from enum import Enum
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


IST = timezone(timedelta(hours=5, minutes=30))

# =============================================================================
# COLAB / DRIVE PERSISTENCE
# -----------------------------------------------------------------------------
# Research caches, SQLite state and logs live on Drive when Colab can mount it.
# A local fallback keeps SMOKE usable outside Colab.
# =============================================================================
def setup_persistence_root() -> Path:
    env_root = os.getenv("SWING_DATA_ROOT", "").strip()
    if env_root:
        p = Path(env_root).expanduser()
        p.mkdir(parents=True, exist_ok=True)
        return p
    try:
        from google.colab import drive  # type: ignore
        mount = Path("/content/drive")
        if not (mount / "MyDrive").exists():
            drive.mount(str(mount), force_remount=False)
        p = mount / "MyDrive" / "smallcap_swing_engine_v2"
        p.mkdir(parents=True, exist_ok=True)
        return p
    except Exception:
        p = Path.cwd() / ".swing_engine_v2"
        p.mkdir(parents=True, exist_ok=True)
        return p

DATA_ROOT = setup_persistence_root()
CACHE_ROOT = DATA_ROOT / "cache"
REPORT_ROOT = DATA_ROOT / "reports"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)


def now_ist() -> datetime:
    return datetime.now(IST)


# ---- optional dependencies -------------------------------------------------
def _try(name: str):
    try:
        return __import__(name)
    except Exception:
        return None


try:
    from IPython.display import display  # noqa: F401
except Exception:  # running as a plain .py module
    display = print  # type: ignore[assignment]

growwapi = _try("growwapi")
pyotp = _try("pyotp")
requests = _try("requests")
yf = _try("yfinance")

HAS_GROWW = growwapi is not None
HAS_GEMINI = _try("google") is not None and _try("google.genai") is not None

# =============================================================================
# LOGGING
# -----------------------------------------------------------------------------
# `logging.basicConfig()` is a silent no-op when the root logger already has a handler — and in
# Colab it does. That is why the original notebook never showed a single log line (look at the
# universe-build output: 30+ minutes of work and only `print` output). `force=True` replaces the
# existing handlers, and writing to stdout makes the lines appear inline under each cell.
#
# Every line also goes to LOG_FILE, so a long run can be inspected afterwards:  !tail -100 swing_engine.log
# Set LOG_LEVEL=DEBUG (env var, or change the default below) to see per-request detail.
# =============================================================================
LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO").upper()
LOG_FILE = os.getenv("LOG_FILE", str(DATA_ROOT / "swing_engine_v2.log"))
LOG_FORMAT = "%(asctime)s | %(levelname)-7s | %(name)-17s | %(message)s"

_handlers: List[logging.Handler] = [logging.StreamHandler(sys.stdout)]
try:
    _handlers.append(logging.FileHandler(LOG_FILE, encoding="utf-8"))
except Exception as _e:  # noqa: BLE001 — read-only FS etc.; console logging still works
    print(f"(file logging disabled: {_e})")

logging.basicConfig(level=getattr(logging, LOG_LEVEL, logging.INFO), format=LOG_FORMAT,
                    datefmt="%H:%M:%S", handlers=_handlers, force=True)
for _noisy in ("urllib3", "requests", "httpx", "httpcore", "yfinance", "peewee", "matplotlib",
               "google", "asyncio"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

# One logger per area, so every line says *where* it came from.
log = logging.getLogger("engine")                    # general / orchestration
LOG_CFG = logging.getLogger("engine.config")
LOG_INFRA = logging.getLogger("engine.infra")
LOG_STORE = logging.getLogger("engine.store")
LOG_BROKER = logging.getLogger("engine.broker")
LOG_DATA = logging.getLogger("engine.data")
LOG_BENCH = logging.getLogger("engine.benchmark")
LOG_UNIV = logging.getLogger("engine.universe")
LOG_FEAT = logging.getLogger("engine.features")
LOG_STRAT = logging.getLogger("engine.strategy")
LOG_REGIME = logging.getLogger("engine.regime")
LOG_SCREEN = logging.getLogger("engine.screener")
LOG_PLAN = logging.getLogger("engine.plan")
LOG_LC = logging.getLogger("engine.lifecycle")
LOG_BT = logging.getLogger("engine.backtest")
LOG_LLM = logging.getLogger("engine.llm")
LOG_TG = logging.getLogger("engine.telegram")
LOG_EVT = logging.getLogger("engine.events")
LOG_CELL = logging.getLogger("engine.cell")


@contextmanager
def log_step(name: str, logger: Optional[logging.Logger] = None, level: int = logging.INFO):
    """Log the start, the end and the duration of a block — and the exception, if it dies."""
    lg = logger or log
    t0 = time.perf_counter()
    lg.log(level, "→ %s ...", name)
    try:
        yield
    except Exception as e:  # noqa: BLE001
        lg.error("✖ %s FAILED after %.2fs — %s: %s", name, time.perf_counter() - t0,
                 type(e).__name__, str(e)[:300])
        raise
    lg.log(level, "← %s done in %.2fs", name, time.perf_counter() - t0)


def timed(logger: Optional[logging.Logger] = None, level: int = logging.INFO, name: str = ""):
    """Decorator form of log_step."""
    def deco(fn: Callable):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            with log_step(name or fn.__name__, logger, level):
                return fn(*args, **kwargs)
        return wrapper
    return deco


def describe_frame(df: Optional[pd.DataFrame], cols: Sequence[str] = ("close",)) -> str:
    """One-line summary of a bar frame for log messages."""
    if df is None:
        return "None"
    if len(df) == 0:
        return "EMPTY (0 rows)"
    try:
        span = f"{pd.Timestamp(df.index.min()).date()} → {pd.Timestamp(df.index.max()).date()}"
    except Exception:  # noqa: BLE001
        span = "non-date index"
    extra = []
    for c in cols:
        if c in df.columns:
            v = df[c].iloc[-1]
            extra.append(f"last_{c}={v:,.2f}" if isinstance(v, (int, float, np.floating)) and
                         np.isfinite(v) else f"last_{c}={v}")
    return f"{len(df)} rows · {span}" + (f" · {' '.join(extra)}" if extra else "")


def mask_secret(s: str, keep: int = 4) -> str:
    if not s:
        return "<missing>"
    return f"{s[:keep]}…{s[-keep:]} ({len(s)} chars)" if len(s) > 2 * keep else "****"


# =============================================================================
# SECRETS — never hard-code them in the notebook.
# Lookup order: environment variable → Colab "Secrets" panel (🔑 icon in the left sidebar).
# =============================================================================
def _secret(name: str, default: str = "") -> str:
    v = os.getenv(name, "")
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(name) or ""
    except Exception:  # noqa: BLE001 — not Colab, secret not defined, or access not granted
        v = ""
    return v or default


# =============================================================================
# CELL TRACKER — every cell logs START / DONE (with duration) / FAILED (with the exception).
# The label comes from the first line of the cell: `# [NN] Title`.
# =============================================================================
_CELL_TAG = re.compile(r"^\s*#\s*\[(?P<tag>[^\]]+)\]\s*(?P<title>.*)$")


class CellTracker:
    def __init__(self) -> None:
        self.t0: Optional[float] = None
        self.label = "<unknown>"
        self.count = 0

    def _label_for(self, raw: str) -> str:
        first = next((ln for ln in (raw or "").splitlines() if ln.strip()), "")
        m = _CELL_TAG.match(first)
        if m:
            return f"[{m['tag']}] {m['title'].strip()}"
        return (first.strip()[:70] or "<empty cell>")

    def pre(self, info: Any = None) -> None:
        self.count += 1
        self.label = self._label_for(getattr(info, "raw_cell", "") or "")
        self.t0 = time.perf_counter()
        LOG_CELL.info("▶ START  %s", self.label)

    def post(self, result: Any = None) -> None:
        if self.t0 is None:           # the cell that registered the hooks
            return
        dt = time.perf_counter() - self.t0
        err = getattr(result, "error_in_exec", None) or getattr(result, "error_before_exec", None)
        if err is not None:
            LOG_CELL.error("✖ FAILED %s after %.1fs — %s: %s", self.label, dt,
                           type(err).__name__, str(err)[:300])
        else:
            LOG_CELL.info("✔ DONE   %s in %.1fs", self.label, dt)
        self.t0 = None


CELL_TRACKER = CellTracker()


def _install_cell_hooks() -> bool:
    try:
        ip = get_ipython()  # type: ignore[name-defined]  # noqa: F821
    except NameError:
        return False
    if ip is None:
        return False

    def _pre(info=None):
        CELL_TRACKER.pre(info)

    def _post(result=None):
        CELL_TRACKER.post(result)

    _pre._swing_hook = True   # type: ignore[attr-defined]
    _post._swing_hook = True  # type: ignore[attr-defined]
    for ev in ("pre_run_cell", "post_run_cell"):          # re-running this cell must not stack hooks
        for cb in list(ip.events.callbacks.get(ev, [])):
            if getattr(cb, "_swing_hook", False):
                ip.events.unregister(ev, cb)
    ip.events.register("pre_run_cell", _pre)
    ip.events.register("post_run_cell", _post)
    return True


_hooks = _install_cell_hooks()

log.info("logging ready | level=%s | file=%s | cell tracking=%s", LOG_LEVEL, LOG_FILE,
         "ON" if _hooks else "off (not running under IPython)")
log.info("python %s | pandas %s | numpy %s", sys.version.split()[0], pd.__version__, np.__version__)
log.info("optional deps | growwapi=%s pyotp=%s requests=%s yfinance=%s google-genai=%s",
         HAS_GROWW, pyotp is not None, requests is not None, yf is not None, HAS_GEMINI)
log.info("now (IST) %s", f"{now_ist():%Y-%m-%d %H:%M:%S}")

07:48:54 | INFO    | engine.cell       | ▶ START  [01] Imports, logging & environment
07:48:57 | INFO    | engine            | logging ready | level=INFO | file=/content/.swing_engine_v2/swing_engine_v2.log | cell tracking=ON
07:48:57 | INFO    | engine            | python 3.13.15 | pandas 2.2.3 | numpy 2.1.3
07:48:57 | INFO    | engine            | optional deps | growwapi=True pyotp=True requests=True yfinance=True google-genai=True
07:48:57 | INFO    | engine            | now (IST) 2026-09-12 13:18:57


## 2 · Configuration

One dataclass holds every knob. Nothing else in the notebook reads `os.environ` directly except
the credential fields below, so the whole system is reproducible from a single object.

**The numbers below are defaults, not recommendations.** `risk_per_trade_pct`, `max_positions`
and `atr_stop_mult` are the three parameters that dominate the return distribution — change them
deliberately and re-run the walk-forward, never mid-campaign.

In [97]:
# [02] Configuration — one selector for SMOKE / RESEARCH / LIVE
# Change only this line for the normal workflow.
RUN_MODE = os.getenv("SWING_RUN_MODE", "LIVE").upper()   # SMOKE | RESEARCH | LIVE

@dataclass
class Config:
    # ---------------- three top-level run modes ----------------
    run_mode: str = RUN_MODE
    mode: str = "PAPER"                 # internal execution mode; derived from run_mode
    dry_run: bool = True
    data_source: str = "GROWW"               # derived unless SWING_DATA_SOURCE is set
    allow_synthetic_fallback: bool = False

    db_path: str = field(default_factory=lambda: str(DATA_ROOT / "swing_engine_v2.db"))
    csv_dir: str = field(default_factory=lambda: str(DATA_ROOT / "ohlcv_csv"))
    research_cache_dir: str = field(default_factory=lambda: str(CACHE_ROOT / "research"))
    report_dir: str = field(default_factory=lambda: str(REPORT_ROOT))

    # ---------------- credentials: Secrets/env only ----------------
    groww_api_key: str = field(default_factory=lambda: _secret("GROWW_API_KEY"))
    groww_api_secret: str = field(default_factory=lambda: _secret("GROWW_API_SECRET"))
    groww_access_token: str = field(default_factory=lambda: _secret("GROWW_ACCESS_TOKEN", ""))
    telegram_token: str = field(default_factory=lambda: _secret("TELEGRAM_BOT_TOKEN", ""))
    telegram_chat_id: str = field(default_factory=lambda: _secret("TELEGRAM_CHAT_ID", ""))
    gemini_api_key: str = field(default_factory=lambda: _secret("GEMINI_API_KEY", ""))
    gemini_model: str = field(default_factory=lambda: os.getenv("GEMINI_MODEL", "gemini-2.5-flash"))

    # ---------------- capital & risk ----------------
    equity: float = 50_000.0
    risk_per_trade_pct: float = 0.75
    max_positions: int = 8
    max_new_per_day: int = 3
    max_position_pct: float = 20.0
    max_portfolio_heat_pct: float = 4.0
    max_sector_positions: int = 2
    daily_loss_halt_pct: float = 2.5
    max_drawdown_halt_pct: float = 15.0
    drawdown_scale_start_pct: float = 7.5
    kelly_cap_fraction: float = 0.25

    # ---------------- universe ----------------
    mcap_min_cr: float = 500.0
    mcap_max_cr: float = 20_000.0
    price_min: float = 25.0
    price_max: float = 5_000.0
    min_median_turnover_cr: float = 3.0
    min_history_bars: int = 260
    warmup_bars: int = 250
    equity_series: Tuple[str, ...] = ("EQ",)
    prefilter_mode: str = "QUOTE"
    prefilter_days: int = 200
    deep_history_days: int = 365 * 9       # target 8+ years after warm-up where available
    max_universe: int = 250
    historical_workers: int = 1
    groww_max_days_per_request: int = 175
    exclusion_file: str = field(default_factory=lambda: str(DATA_ROOT / "exclusions.csv"))
    surveillance_manual_csv: str = field(default_factory=lambda: str(DATA_ROOT / "surveillance_manual.csv"))
    pit_membership_dir: str = field(default_factory=lambda: str(DATA_ROOT / "pit_constituents"))

    # ---------------- benchmark ----------------
    benchmark: str = "NIFTY_SMLCAP_250"
    benchmark_groww_symbol: str = ""
    benchmark_search_terms: Tuple[str, ...] = ("SMALLCAP 250", "SMLCAP 250", "SMLCAP250", "SMALLCAP250")
    benchmark_yf_tickers: Tuple[str, ...] = ("NIFTYSMLCAP250.NS",)
    benchmark_allow_universe_proxy: bool = True
    min_benchmark_bars: int = 210
    max_benchmark_staleness_days: int = 7
    allow_mcap_proxy: bool = True

    # ---------------- entry filters ----------------
    min_adx: float = 18.0
    max_atr_pct: float = 9.0
    min_atr_pct: float = 1.8
    min_rs_percentile: float = 60.0
    max_extension_atr: float = 4.0
    min_score: float = 62.0
    max_entry_gap_pct: float = 2.5
    max_stop_pct: float = 8.0             # v2 hard cap; 13.7% FINCABLES-style stop is rejected
    max_stop_atr: float = 3.2

    # ---------------- exits ----------------
    atr_stop_mult: float = 2.2
    t1_r_multiple: float = 1.5
    t1_exit_fraction: float = 0.5
    t2_r_multiple: float = 3.0
    breakeven_after_r: float = 1.0
    trail_activate_r: float = 1.5
    chandelier_mult: float = 3.0
    time_stop_bars: int = 12
    max_hold_bars: int = 30
    use_follow_through_exit: bool = True
    follow_through_bars: int = 5
    follow_through_min_r: float = 0.35
    keep_right_tail_after_r: float = 1.5

    # ---------------- optional candidate edges; OOS gate keeps them off ----------------
    enable_52w_momentum: bool = True
    enable_episodic_pivot: bool = True
    enable_mean_revert: bool = True
    use_meta_label: bool = False

    # ---------------- costs ----------------
    brokerage_pct: float = 0.10
    brokerage_cap: float = 20.0
    slippage_bps: float = 12.0
    max_adv_participation_pct: float = 3.0

    # ---------------- continuous regime ----------------
    regime_breadth_lookback: int = 50
    regime_risk_off_breadth: float = 35.0
    regime_risk_on_breadth: float = 55.0
    min_exposure_mult: float = 0.20
    max_exposure_mult: float = 1.00

    # ---------------- optional layers ----------------
    use_gemini: bool = True
    gemini_max_adjust: float = 10.0
    use_telegram: bool = True
    require_approval: bool = True

    # ---------------- research ----------------
    bt_start: str = "2018-01-01"
    bt_end: str = ""                     # blank = latest available bar
    holdout_months: int = 12
    wf_folds: int = 6
    wf_embargo_bars: int = 10
    wf_min_test_bars: int = 126
    min_oos_trades: int = 150
    objective_min_profit_factor: float = 1.4
    monte_carlo_sims: int = 3000
    random_seed: int = 42
    smoke_symbols: int = 24
    smoke_years: int = 4

    def __post_init__(self) -> None:
        self.run_mode = self.run_mode.upper().strip()
        explicit = os.getenv("SWING_DATA_SOURCE", "").upper().strip()
        if self.run_mode == "SMOKE":
            self.mode = "PAPER"
            self.data_source = explicit or "SYNTHETIC"
            self.allow_synthetic_fallback = True
            self.max_universe = min(self.max_universe, self.smoke_symbols)
            self.deep_history_days = 365 * self.smoke_years
            self.use_gemini = False
            self.use_telegram = False
        elif self.run_mode == "RESEARCH":
            self.mode = "PAPER"
            self.data_source = explicit or "YFINANCE"
            self.allow_synthetic_fallback = False
            self.use_gemini = False
            self.use_telegram = False
        elif self.run_mode == "LIVE":
            self.mode = "LIVE"
            self.data_source = explicit or "GROWW"
            self.allow_synthetic_fallback = False
            self.dry_run = True              # safe default, deliberately not environment-overridable
            self.require_approval = True
        Path(self.research_cache_dir).mkdir(parents=True, exist_ok=True)
        Path(self.report_dir).mkdir(parents=True, exist_ok=True)

    def validate(self) -> None:
        assert self.run_mode in ("SMOKE", "RESEARCH", "LIVE")
        assert self.mode in ("PAPER", "LIVE")
        assert self.data_source in ("SYNTHETIC", "GROWW", "YFINANCE", "CSV")
        assert 0 < self.risk_per_trade_pct <= 3
        assert 0 < self.max_drawdown_halt_pct <= 15
        assert 0 < self.max_stop_pct <= 15
        assert self.t1_r_multiple < self.t2_r_multiple
        if self.run_mode == "LIVE" and not self.dry_run:
            assert self.groww_access_token or (self.groww_api_key and self.groww_api_secret)
        LOG_CFG.info("Config OK | run_mode=%s execution=%s dry_run=%s source=%s equity=₹%s",
                     self.run_mode, self.mode, self.dry_run, self.data_source, f"{self.equity:,.0f}")
        LOG_CFG.info("research | history≈%.1fy · final holdout=%dm · wf=%d folds embargo=%d bars",
                     self.deep_history_days/365.25, self.holdout_months, self.wf_folds, self.wf_embargo_bars)
        LOG_CFG.info("credentials | groww token=%s · groww key=%s · telegram=%s · gemini=%s",
                     mask_secret(self.groww_access_token), mask_secret(self.groww_api_key),
                     "set" if self.telegram_token else "<missing>",
                     "set" if self.gemini_api_key else "<missing>")

    @property
    def history_days(self) -> int:
        return int(self.deep_history_days)

CFG = Config()
CFG.validate()
np.random.seed(CFG.random_seed)
random.seed(CFG.random_seed)

07:48:57 | INFO    | engine.cell       | ▶ START  [02] Configuration — one selector for SMOKE / RESEARCH / LIVE
07:49:10 | INFO    | engine.config     | Config OK | run_mode=LIVE execution=LIVE dry_run=True source=GROWW equity=₹50,000
07:49:10 | INFO    | engine.config     | research | history≈9.0y · final holdout=12m · wf=6 folds embargo=10 bars
07:49:10 | INFO    | engine.config     | credentials | groww token=eyJr…LCSA (1102 chars) · groww key=<missing> · telegram=set · gemini=set
07:49:10 | INFO    | engine.cell       | ✔ DONE   [02] Configuration — one selector for SMOKE / RESEARCH / LIVE in 12.8s


## 3 · Infrastructure — rate limiting, retries, market calendar

Groww applies rate limits **per API group** (orders / live-data / non-trading), not per endpoint,
so the limiter below is keyed by group. Every broker call goes through it. The retry wrapper uses
full jitter and never retries a non-idempotent order placement more than once.

In [98]:
# [03] Infrastructure — rate limiting, retries, market calendar
class RateLimiter:
    """Token bucket, thread-safe, one bucket per API group, **multiple windows per group**.

    One per-second rule is not enough. Brokers typically enforce a burst limit *and* a per-minute
    ceiling, and you discover the second one only after an hour of clean calls suddenly starts
    returning `Rate limit has breached`.
    """

    def __init__(self, limits: Dict[str, Any]):
        # group -> [(max_calls, per_seconds), ...]
        self.limits: Dict[str, List[Tuple[int, float]]] = {
            k: (list(v) if isinstance(v, list) else [v]) for k, v in limits.items()}
        self._hits: Dict[str, deque] = defaultdict(deque)
        self._cooldown: Dict[str, float] = {}
        self._lock = threading.Lock()

    def penalise(self, group: str, seconds: float) -> None:
        """Called when the API says we breached. Freezes the whole group, not just one thread.

        Without this, every worker retries independently, so a breach *multiplies* traffic at
        exactly the moment it needs to stop. That is how one 429 becomes a run where every
        symbol fails.
        """
        with self._lock:
            self._cooldown[group] = max(self._cooldown.get(group, 0.0),
                                        time.monotonic() + seconds)
        LOG_INFRA.warning("rate limit hit on '%s' — freezing that group for %.0fs", group, seconds)

    def acquire(self, group: str = "default") -> None:
        rules = self.limits.get(group) or self.limits.get("default", [(8, 1.0)])
        longest = max(w for _, w in rules)
        while True:
            with self._lock:
                cool = self._cooldown.get(group, 0.0) - time.monotonic()
            if cool > 0:
                LOG_INFRA.debug("'%s' group cooling down — %.1fs left", group, cool)
                time.sleep(min(cool, 5.0))
                continue
            with self._lock:
                now = time.monotonic()
                q = self._hits[group]
                while q and now - q[0] > longest:
                    q.popleft()
                wait = 0.0
                for max_calls, window in rules:
                    in_window = [t for t in q if now - t <= window]
                    if len(in_window) >= max_calls:
                        wait = max(wait, window - (now - in_window[0]) + 0.01)
                if wait <= 0:
                    q.append(now)
                    return
            if wait > 2.0:
                LOG_INFRA.debug("'%s' bucket full — waiting %.1fs", group, wait)
            time.sleep(min(wait, 5.0))


# Deliberately conservative. Historical is the tight group, and one symbol's full history is
# ~7 chunked requests (see below), so this bucket gets hit hard during a universe build.
# Groww publishes limits per *type*: Authentication 5/s & 30/min, Orders 10/s & 250/min,
# Live Data 10/s & 300/min, Non Trading 20/s & 500/min. **Historical is not in that table**, so its
# real ceiling is undocumented and observably tighter than 2/s. These values sit below anything
# published. If breaches continue, drop "historical" to [(1, 3.0), (15, 60.0)] and run
# `probe_historical()` (below) to measure your actual ceiling instead of guessing.
LIMITER = RateLimiter({
    "default":     [(8, 1.0)],
    "orders":      [(8, 1.0), (200, 60.0)],
    "live_data":   [(8, 1.0), (250, 60.0)],
    "historical":  [(1, 1.0), (50, 60.0)],
    "non_trading": [(8, 1.0), (400, 60.0)],
})

RATE_LIMIT_HINTS = ("rate limit", "too many requests", "429")
HARD_FAIL_HINTS = ("invalid", "unauthor", "forbidden", "rejected", "maximum of", "not found")


def retry(times: int = 3, base: float = 0.8, group: str = "default", idempotent: bool = True):
    """Exponential backoff with full jitter. Rate-limit errors get a long, escalating sleep
    rather than a 300ms jitter; hard failures short-circuit. Non-idempotent calls: one attempt.

    PATCH: `functools.wraps` sets `__wrapped__`, so `probe_historical()` can now genuinely bypass
    the limiter as its docstring promises; every retry/give-up decision is logged."""
    def deco(fn: Callable):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            attempts = times if idempotent else 1
            last = None
            for i in range(attempts):
                try:
                    LIMITER.acquire(group)
                    return fn(*args, **kwargs)
                except Exception as e:  # noqa: BLE001
                    last = e
                    msg = str(e).lower()
                    if any(k in msg for k in RATE_LIMIT_HINTS):
                        # Freeze the entire group. Retrying alone just adds traffic to a breach.
                        LIMITER.penalise(group, 20.0 * (2 ** i))
                        time.sleep(random.uniform(0, 2.0))
                        continue
                    if any(k in msg for k in HARD_FAIL_HINTS):
                        LOG_INFRA.debug("%s: hard failure, not retrying — %s", fn.__name__, str(e)[:160])
                        break  # don't hammer a hard failure
                    if i < attempts - 1:
                        sleep = random.uniform(0, base * (2 ** i))
                        LOG_INFRA.debug("%s: attempt %d/%d failed (%s) — retrying in %.2fs",
                                        fn.__name__, i + 1, attempts, str(e)[:120], sleep)
                        time.sleep(sleep)
            # DEBUG, not ERROR: callers decide how loud a failure is (a delisted small cap is
            # routine; a missing benchmark is fatal). The caller always logs the outcome.
            LOG_INFRA.log(logging.WARNING if group == "orders" else logging.DEBUG,
                          "%s failed after %d attempt(s): %s", fn.__name__, attempts, last)
            raise last  # type: ignore[misc]
        return wrapper
    return deco


# ---- NSE session windows (IST) --------------------------------------------
MARKET_OPEN = dtime(9, 15)
MARKET_CLOSE = dtime(15, 30)
PREMARKET_SCAN = dtime(8, 45)
EOD_SCAN = dtime(15, 45)

# Replace with the official NSE holiday list each January, or derive from the instruments feed.
NSE_HOLIDAYS_2026 = {
    "2026-01-26", "2026-03-04", "2026-03-26", "2026-04-01", "2026-04-03",
    "2026-05-01", "2026-08-15", "2026-09-14", "2026-10-02", "2026-10-21",
    "2026-11-09", "2026-12-25",
}


def is_trading_day(d: date, holidays: Iterable[str] = NSE_HOLIDAYS_2026) -> bool:
    return d.weekday() < 5 and d.isoformat() not in set(holidays)


def market_is_open(ts: Optional[datetime] = None) -> bool:
    ts = ts or now_ist()
    return is_trading_day(ts.date()) and MARKET_OPEN <= ts.time() <= MARKET_CLOSE


def next_trading_day(d: date) -> date:
    nd = d + timedelta(days=1)
    while not is_trading_day(nd):
        nd += timedelta(days=1)
    return nd


_today = now_ist()
LOG_INFRA.info("rate limits: %s", {k: v for k, v in LIMITER.limits.items()})
LOG_INFRA.info("calendar | today %s is %sa trading day · market %s · next session %s",
               _today.date(), "" if is_trading_day(_today.date()) else "NOT ",
               "OPEN" if market_is_open(_today) else "closed", next_trading_day(_today.date()))
if _today.year != 2026:
    LOG_INFRA.warning("holiday list is for 2026 but today is %d — update NSE_HOLIDAYS_2026", _today.year)


07:49:10 | INFO    | engine.cell       | ▶ START  [03] Infrastructure — rate limiting, retries, market calendar
07:49:10 | INFO    | engine.infra      | rate limits: {'default': [(8, 1.0)], 'orders': [(8, 1.0), (200, 60.0)], 'live_data': [(8, 1.0), (250, 60.0)], 'historical': [(1, 1.0), (50, 60.0)], 'non_trading': [(8, 1.0), (400, 60.0)]}
07:49:10 | INFO    | engine.infra      | calendar | today 2026-09-12 is NOT a trading day · market closed · next session 2026-09-15
07:49:10 | INFO    | engine.cell       | ✔ DONE   [03] Infrastructure — rate limiting, retries, market calendar in 0.0s


## 4 · Persistence

SQLite, because the entire state of the book must survive a kernel restart, a laptop reboot and a
network partition mid-order. Seven tables: `instruments`, `candles`, `signals`, `trades`,
`orders`, `events`, `equity`. `candles` is the incremental OHLCV cache keyed `(symbol, date)`.

In [99]:
# [04] Persistence
SCHEMA = """
CREATE TABLE IF NOT EXISTS instruments (
    symbol TEXT PRIMARY KEY, groww_symbol TEXT, exchange TEXT, segment TEXT,
    name TEXT, isin TEXT, series TEXT, instrument_type TEXT,
    exchange_token TEXT, tick_size REAL, lot_size INTEGER,
    sector TEXT, mcap_cr REAL, updated_at TEXT
);
CREATE TABLE IF NOT EXISTS candles (
    symbol TEXT, date TEXT, open REAL, high REAL, low REAL, close REAL, volume REAL,
    PRIMARY KEY (symbol, date)
);
CREATE INDEX IF NOT EXISTS ix_candles_sym ON candles(symbol);
CREATE TABLE IF NOT EXISTS signals (
    id TEXT PRIMARY KEY, scan_date TEXT, symbol TEXT, strategy TEXT, score REAL,
    entry_type TEXT, entry_trigger REAL, stop REAL, t1 REAL, t2 REAL,
    qty INTEGER, risk_amount REAL, r_per_share REAL, regime TEXT,
    features TEXT, llm TEXT, status TEXT, created_at TEXT
);
CREATE TABLE IF NOT EXISTS trades (
    id TEXT PRIMARY KEY, signal_id TEXT, symbol TEXT, strategy TEXT, state TEXT,
    side TEXT, planned_qty INTEGER, filled_qty INTEGER, open_qty INTEGER,
    avg_entry REAL, entry_trigger REAL, init_stop REAL, cur_stop REAL,
    t1 REAL, t2 REAL, r_per_share REAL, realised_pnl REAL, fees REAL,
    bars_held INTEGER, mfe_r REAL, mae_r REAL, exit_reason TEXT,
    entry_date TEXT, exit_date TEXT, avg_exit REAL, notes TEXT, updated_at TEXT
);
CREATE TABLE IF NOT EXISTS orders (
    id TEXT PRIMARY KEY, trade_id TEXT, broker_order_id TEXT, symbol TEXT,
    txn TEXT, order_type TEXT, qty INTEGER, price REAL, trigger_price REAL,
    status TEXT, filled_qty INTEGER, avg_price REAL, purpose TEXT,
    created_at TEXT, updated_at TEXT, raw TEXT
);
CREATE TABLE IF NOT EXISTS events (
    id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, level TEXT, kind TEXT,
    symbol TEXT, trade_id TEXT, message TEXT, payload TEXT
);
CREATE TABLE IF NOT EXISTS equity (
    date TEXT PRIMARY KEY, cash REAL, positions_value REAL, total REAL,
    open_risk REAL, n_positions INTEGER, regime TEXT
);
CREATE TABLE IF NOT EXISTS kv (k TEXT PRIMARY KEY, v TEXT);
CREATE TABLE IF NOT EXISTS journal (
    id TEXT PRIMARY KEY, ts TEXT, kind TEXT, symbol TEXT, strategy TEXT,
    planned_entry REAL, actual_entry REAL, planned_stop REAL, qty INTEGER,
    slippage_bps REAL, status TEXT, payload TEXT
);
CREATE TABLE IF NOT EXISTS research_runs (
    id TEXT PRIMARY KEY, ts TEXT, config_hash TEXT, mode TEXT, start TEXT, end TEXT,
    metrics TEXT, notes TEXT
);
"""


class Store:
    def __init__(self, path: str):
        self.path = path
        self._local = threading.local()
        with self.conn() as c:
            c.executescript(SCHEMA)

    def conn(self) -> sqlite3.Connection:
        if getattr(self._local, "c", None) is None:
            c = sqlite3.connect(self.path, timeout=30, check_same_thread=False)
            c.row_factory = sqlite3.Row
            c.execute("PRAGMA journal_mode=WAL")
            c.execute("PRAGMA synchronous=NORMAL")
            self._local.c = c
        return self._local.c

    # ---- generic helpers ----
    def exec(self, sql: str, params: Sequence = ()) -> None:
        with self.conn() as c:
            c.execute(sql, params)

    def query(self, sql: str, params: Sequence = ()) -> pd.DataFrame:
        return pd.read_sql_query(sql, self.conn(), params=list(params))

    def upsert(self, table: str, row: Dict[str, Any]) -> None:
        cols = list(row)
        ph = ",".join("?" * len(cols))
        sql = f"INSERT OR REPLACE INTO {table} ({','.join(cols)}) VALUES ({ph})"
        self.exec(sql, [row[c] for c in cols])

    # ---- candles ----
    def save_candles(self, symbol: str, df: pd.DataFrame) -> int:
        if df is None or df.empty:
            LOG_STORE.debug("save_candles %s: nothing to save", symbol)
            return 0
        d = df.copy()
        d.index = pd.to_datetime(d.index)
        d = d[["open", "high", "low", "close", "volume"]]
        rows = [(symbol, ts.strftime("%Y-%m-%d"), *[float(x) for x in r])
                for ts, r in zip(d.index, d.to_numpy())]
        with self.conn() as c:
            c.executemany(
                "INSERT OR REPLACE INTO candles(symbol,date,open,high,low,close,volume) "
                "VALUES (?,?,?,?,?,?,?)", rows)
        LOG_STORE.debug("save_candles %s: %d rows upserted", symbol, len(rows))
        return len(rows)

    def load_candles(self, symbol: str, start: Optional[str] = None) -> pd.DataFrame:
        sql = "SELECT date,open,high,low,close,volume FROM candles WHERE symbol=?"
        p: List[Any] = [symbol]
        if start:
            sql += " AND date>=?"
            p.append(start)
        df = self.query(sql + " ORDER BY date", p)
        if df.empty:
            return df
        df["date"] = pd.to_datetime(df["date"])
        return df.set_index("date")

    def last_candle_date(self, symbol: str) -> Optional[str]:
        df = self.query("SELECT MAX(date) d FROM candles WHERE symbol=?", [symbol])
        v = df.iloc[0]["d"]
        return None if pd.isna(v) else str(v)

    # ---- events ----
    def event(self, kind: str, message: str, level: str = "INFO",
              symbol: str = "", trade_id: str = "", payload: Any = None) -> None:
        self.exec(
            "INSERT INTO events(ts,level,kind,symbol,trade_id,message,payload) VALUES (?,?,?,?,?,?,?)",
            [now_ist().isoformat(), level, kind, symbol, trade_id, message,
             json.dumps(payload, default=str) if payload is not None else None])
        # PATCH: mirror every audit event to the console so the lifecycle is visible live, not
        # only after the fact in the `events` table. FETCH_FAIL is already logged by the fetcher.
        if kind not in EVENT_MIRROR_SKIP:
            LOG_EVT.log(getattr(logging, str(level).upper(), logging.INFO), "[%s] %s%s",
                        kind, f"{symbol} · " if symbol and symbol not in message else "", message)

    def stats(self) -> Dict[str, Any]:
        out: Dict[str, Any] = {}
        for t in ("candles", "instruments", "signals", "trades", "orders", "events"):
            try:
                out[t] = int(self.query(f"SELECT COUNT(*) n FROM {t}").iloc[0]["n"])
            except Exception:  # noqa: BLE001
                out[t] = "?"
        try:
            out["candle_symbols"] = int(self.query(
                "SELECT COUNT(DISTINCT symbol) n FROM candles").iloc[0]["n"])
            d = self.query("SELECT MIN(date) a, MAX(date) b FROM candles").iloc[0]
            out["candle_span"] = f"{d['a']} → {d['b']}"
        except Exception:  # noqa: BLE001
            pass
        return out

    def get_kv(self, k: str, default=None):
        df = self.query("SELECT v FROM kv WHERE k=?", [k])
        return json.loads(df.iloc[0]["v"]) if len(df) else default

    def set_kv(self, k: str, v) -> None:
        self.upsert("kv", {"k": k, "v": json.dumps(v, default=str)})


EVENT_MIRROR_SKIP = {"FETCH_FAIL"}

STORE = Store(CFG.db_path)
LOG_STORE.info("store ready → %s (%s)", os.path.abspath(CFG.db_path),
               f"{os.path.getsize(CFG.db_path) / 1e6:,.1f} MB" if os.path.exists(CFG.db_path) else "new")
LOG_STORE.info("db contents: %s", STORE.stats())
_halted = STORE.get_kv("halted", False)
if _halted:
    LOG_STORE.warning("kill switch is ENGAGED in the db (kv 'halted'=True) — no new entries will be "
                      "taken until you run STORE.set_kv('halted', False)")

07:49:10 | INFO    | engine.cell       | ▶ START  [04] Persistence
07:49:10 | INFO    | engine.store      | store ready → /content/.swing_engine_v2/swing_engine_v2.db (0.0 MB)
07:49:10 | INFO    | engine.store      | db contents: {'candles': 0, 'instruments': 0, 'signals': 0, 'trades': 0, 'orders': 0, 'events': 0, 'candle_symbols': 0, 'candle_span': 'None → None'}
07:49:10 | INFO    | engine.cell       | ✔ DONE   [04] Persistence in 0.0s


## 5 · Broker abstraction

`BrokerAdapter` is the only place that knows about Groww. Two implementations:

- **`PaperBroker`** — simulates a matching engine against daily bars: market orders fill at the
  next reference price plus slippage; buy-stop orders fill only if the bar's high crosses the
  trigger; stop-loss orders fill on the bar's low. It maintains its own positions and cash.
- **`GrowwBroker`** — real calls, with the TOTP login flow, a cached access token that it refreshes
  through the ~6 AM IST expiry, the rate limiter, and normalisation of Groww's response payloads
  into the same dicts the paper broker returns.

Because both return identical shapes, the lifecycle engine is written once.

In [100]:
# [05] Broker abstraction (Groww live + paper simulator)
class StrEnum(str, Enum):
    """`str(TState.OPEN)` must be 'OPEN', not 'TState.OPEN' — the DB stores these values."""
    def __str__(self) -> str:
        return str(self.value)


class Txn(StrEnum):
    BUY = "BUY"
    SELL = "SELL"


class OrdType(StrEnum):
    MARKET = "MARKET"
    LIMIT = "LIMIT"
    SL = "SL"            # stop-loss limit
    SL_M = "SL_M"        # stop-loss market


class OrdStatus(StrEnum):
    NEW = "NEW"
    PENDING = "PENDING"
    OPEN = "OPEN"
    FILLED = "FILLED"
    PARTIAL = "PARTIAL"
    CANCELLED = "CANCELLED"
    REJECTED = "REJECTED"
    TRIGGER_PENDING = "TRIGGER_PENDING"


GROWW_STATUS_MAP = {
    "NEW": OrdStatus.NEW, "ACKED": OrdStatus.OPEN, "APPROVED": OrdStatus.OPEN,
    "OPEN": OrdStatus.OPEN, "TRIGGER_PENDING": OrdStatus.TRIGGER_PENDING,
    "EXECUTED": OrdStatus.FILLED, "COMPLETE": OrdStatus.FILLED,
    "CANCELLED": OrdStatus.CANCELLED, "CANCELLATION_REQUESTED": OrdStatus.CANCELLED,
    "REJECTED": OrdStatus.REJECTED, "FAILED": OrdStatus.REJECTED,
}


class BrokerAdapter:
    """Interface. Every method returns plain dicts / DataFrames — no SDK objects leak out."""
    name = "base"

    def connect(self) -> None: ...
    def instruments(self) -> pd.DataFrame: raise NotImplementedError
    def daily_candles(self, symbol: str, start: date, end: date) -> pd.DataFrame: raise NotImplementedError
    def ltp(self, symbols: Sequence[str]) -> Dict[str, float]: raise NotImplementedError
    def quote(self, symbol: str) -> Dict[str, Any]: raise NotImplementedError
    def place_order(self, **kw) -> Dict[str, Any]: raise NotImplementedError
    def modify_order(self, order_id: str, **kw) -> Dict[str, Any]: raise NotImplementedError
    def cancel_order(self, order_id: str) -> Dict[str, Any]: raise NotImplementedError
    def order_detail(self, order_id: str) -> Dict[str, Any]: raise NotImplementedError
    def positions(self) -> pd.DataFrame: raise NotImplementedError
    def holdings(self) -> pd.DataFrame: raise NotImplementedError


class GrowwBroker(BrokerAdapter):
    name = "groww"

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.api = None
        self._token: str = ""
        self._token_exp: Optional[datetime] = None
        self._instruments: Optional[pd.DataFrame] = None
        self._lock = threading.Lock()
        # trading_symbol -> exact groww_symbol, for instruments whose groww symbol is not simply
        # "NSE-<trading_symbol>" (the benchmark index is registered here once resolved).
        self.symbol_overrides: Dict[str, str] = {}
        self._quote_keys_logged = False
        self._jwt_warned = False

    # ---------- auth ----------
    def _token_valid(self) -> bool:
        return bool(self._token) and self._token_exp is not None and now_ist() < self._token_exp

    def _expiry(self) -> datetime:
        """Groww access tokens die daily around 06:00 IST. Refresh a little before."""
        n = now_ist()
        exp = n.replace(hour=5, minute=45, second=0, microsecond=0)
        if n >= exp:
            exp += timedelta(days=1)
        return exp

    @staticmethod
    def _jwt_expiry(token: str) -> Optional[datetime]:
        """Read the `exp` claim of a JWT access token without verifying it (diagnostics only)."""
        try:
            part = token.split(".")[1]
            part += "=" * (-len(part) % 4)
            exp = json.loads(base64.urlsafe_b64decode(part)).get("exp")
            return datetime.fromtimestamp(int(exp), IST) if exp else None
        except Exception:  # noqa: BLE001
            return None

    def connect(self) -> None:
        if not HAS_GROWW:
            raise RuntimeError("growwapi not installed: pip install growwapi pyotp")
        with self._lock:
            if self._token_valid() and self.api is not None:
                return
            GrowwAPI = growwapi.GrowwAPI  # noqa: N806
            if self.cfg.groww_access_token:
                self._token = self.cfg.groww_access_token
                how = "access token"
                jwt_exp = self._jwt_expiry(self._token)
                if jwt_exp is not None and jwt_exp <= now_ist() and not self._jwt_warned:
                    self._jwt_warned = True
                    LOG_BROKER.error("Groww access token EXPIRED at %s — every request will fail "
                                     "(401/unauthorised). Generate a fresh token, or use the "
                                     "API key + secret (TOTP) flow.", f"{jwt_exp:%Y-%m-%d %H:%M IST}")
                elif jwt_exp is not None:
                    LOG_BROKER.info("access token expires %s (in %.1f h)",
                                    f"{jwt_exp:%Y-%m-%d %H:%M IST}",
                                    (jwt_exp - now_ist()).total_seconds() / 3600)
            elif self.cfg.groww_api_key and self.cfg.groww_api_secret:
                # TOTP flow — the SDK derives the 6-digit code from the secret internally.
                how = "api key + secret (TOTP)"
                LOG_BROKER.info("requesting a Groww access token via key + secret ...")
                self._token = GrowwAPI.get_access_token(
                    api_key=self.cfg.groww_api_key, secret=self.cfg.groww_api_secret)
            else:
                raise RuntimeError("no Groww credentials configured (set GROWW_ACCESS_TOKEN, or "
                                   "GROWW_API_KEY + GROWW_API_SECRET, in Colab Secrets / env)")
            self.api = GrowwAPI(self._token)
            self._token_exp = self._expiry()
            jwt_exp = self._jwt_expiry(self._token)
            if jwt_exp is not None and now_ist() < jwt_exp < self._token_exp:
                self._token_exp = jwt_exp
            LOG_BROKER.info("Groww connected via %s; will reconnect after %s", how,
                            f"{self._token_exp:%Y-%m-%d %H:%M IST}")

    def _api(self):
        if not self._token_valid() or self.api is None:
            self.connect()
        return self.api

    # ---------- reference data ----------
    @retry(group="non_trading")
    def instruments(self) -> pd.DataFrame:
        if self._instruments is not None:
            return self._instruments
        t0 = time.time()
        df = self._api().get_all_instruments()
        df = pd.DataFrame(df)
        df.columns = [c.lower() for c in df.columns]
        self._instruments = df
        LOG_BROKER.info("instrument master: %d rows in %.1fs", len(df), time.time() - t0)
        for c in ("segment", "instrument_type"):
            if c in df.columns:
                LOG_BROKER.info("  by %s: %s", c,
                                df[c].astype(str).value_counts().head(12).to_dict())
        return df

    def find_index_instruments(self, terms: Sequence[str], exact: Sequence[str] = ()) -> pd.DataFrame:
        """Search the instrument master for an index. Ranked: exact normalised name match first,
        then instrument_type IDX, then NSE. Also used to diagnose benchmark problems by hand."""
        df = self.instruments().copy()
        cols = ["exchange", "segment", "trading_symbol", "groww_symbol", "name", "instrument_type"]
        for c in cols:
            if c not in df.columns:
                df[c] = ""
            df[c] = df[c].fillna("").astype(str).str.strip()
        hay = (df["trading_symbol"] + " | " + df["name"] + " | " + df["groww_symbol"]).str.upper()
        pat = "|".join(re.escape(t.upper()) for t in terms if t)
        m = hay.str.contains(pat, regex=True, na=False) & df["segment"].str.upper().eq("CASH")
        out = df.loc[m, cols].copy()
        if out.empty:
            return out
        norm = lambda s: re.sub(r"[^A-Z0-9]", "", str(s).upper())  # noqa: E731
        targets = {norm(x) for x in exact if x}
        out["exact"] = out["trading_symbol"].map(norm).isin(targets) | out["name"].map(norm).isin(targets)
        out["is_index"] = out["instrument_type"].str.upper().isin(["IDX", "INDEX", "INDICES"])
        out["is_nse"] = out["exchange"].str.upper().eq("NSE")
        out["name_len"] = out["name"].str.len()
        out = out.sort_values(["exact", "is_index", "is_nse", "name_len"],
                              ascending=[False, False, False, True])
        return out.drop(columns=["name_len"]).reset_index(drop=True)

    def _groww_symbol(self, symbol: str, exchange: str = "NSE") -> str:
        return self.symbol_overrides.get(symbol) or f"{exchange}-{symbol}"

    # ---------- market data ----------
    @retry(group="historical")
    def _candle_window(self, symbol: str, start: date, end: date) -> pd.DataFrame:
        """One request. Must span <= 180 days or the API rejects it outright."""
        api = self._api()
        gsym = self._groww_symbol(symbol)
        LOG_BROKER.debug("candles %s (%s) %s → %s", symbol, gsym, start, end)
        try:
            resp = api.get_historical_candles(
                exchange=api.EXCHANGE_NSE,
                segment=api.SEGMENT_CASH,
                groww_symbol=gsym,
                start_time=f"{start:%Y-%m-%d} 09:15:00",
                end_time=f"{end:%Y-%m-%d} 15:30:00",
                candle_interval=getattr(api, "CANDLE_INTERVAL_DAY", 1440),
            )
        except AttributeError:                      # older SDKs
            LOG_BROKER.debug("SDK has no get_historical_candles — using the deprecated endpoint")
            resp = api.get_historical_candle_data(
                trading_symbol=gsym.split("-", 1)[-1], exchange=api.EXCHANGE_NSE,
                segment=api.SEGMENT_CASH,
                start_time=f"{start:%Y-%m-%d} 09:15:00",
                end_time=f"{end:%Y-%m-%d} 15:30:00", interval_in_minutes=1440)
        if isinstance(resp, dict) and "payload" in resp:
            resp = resp["payload"]
        out = self._candles_to_df(resp)
        if out.empty:
            LOG_BROKER.debug("candles %s %s → %s: empty window (raw type %s)", symbol, start, end,
                             type(resp).__name__)
        return out

    def daily_candles(self, symbol: str, start: date, end: date) -> pd.DataFrame:
        """Groww caps the 1-day interval at 180 days per request:
        `Interval 1day can only be queried for a maximum of 180 days`.
        So walk the range in windows and stitch. 1200 days -> 7 requests."""
        step = timedelta(days=int(self.cfg.groww_max_days_per_request))
        parts, cursor, n_req = [], start, 0
        while cursor <= end:
            stop = min(cursor + step - timedelta(days=1), end)
            df = self._candle_window(symbol, cursor, stop)
            if df is not None and not df.empty:
                parts.append(df)
            cursor = stop + timedelta(days=1)
            n_req += 1
        if not parts:
            LOG_BROKER.debug("daily_candles %s: 0 bars from %d request(s)", symbol, n_req)
            return pd.DataFrame(columns=["open", "high", "low", "close", "volume"])
        out = pd.concat(parts)
        out = out[~out.index.duplicated(keep="last")].sort_index()
        LOG_BROKER.debug("daily_candles %s: %s from %d request(s)", symbol, describe_frame(out), n_req)
        return out

    @staticmethod
    def _candles_to_df(resp: Any) -> pd.DataFrame:
        """Normalise whatever the API returns into a clean daily OHLCV frame.

        The timestamp column is not stable across SDK versions: epoch seconds, epoch millis, or an
        ISO string like `2026-02-23T00:00:00`. Assuming epoch is what produced
        `Unable to parse string "2026-02-23T00:00:00" at position 0`, so handle all three. Payloads
        also arrive as list-of-lists or list-of-dicts, optionally wrapped in `payload`/`candles`.
        """
        empty = pd.DataFrame(columns=["open", "high", "low", "close", "volume"])
        if isinstance(resp, dict):
            resp = resp.get("payload", resp)
            resp = resp.get("candles", resp) if isinstance(resp, dict) else resp
        if resp is None or len(resp) == 0:
            return empty

        df = pd.DataFrame(resp)
        if isinstance(resp[0], dict):                        # list-of-dicts payload
            cols = {c.lower(): c for c in df.columns}
            ts_col = next((cols[k] for k in ("timestamp", "time", "date", "start_time", "epoch")
                           if k in cols), df.columns[0])
            ren = {ts_col: "ts"}
            for k in ("open", "high", "low", "close", "volume"):
                if k in cols:
                    ren[cols[k]] = k
            df = df.rename(columns=ren)
        else:                                                # list-of-lists payload
            # PATCH: index candles have no volume, so they can arrive as 5 columns. The original
            # `< 6 -> empty` rule threw away every benchmark bar even when the symbol was right.
            if df.shape[1] < 5:
                LOG_BROKER.debug("candle payload has %d columns — expected ≥5", df.shape[1])
                return empty
            if df.shape[1] == 5:
                df.columns = ["ts", "open", "high", "low", "close"]
            else:
                df = df.iloc[:, :6]
                df.columns = ["ts", "open", "high", "low", "close", "volume"]
        if "ts" not in df.columns or "close" not in df.columns:
            LOG_BROKER.debug("candle payload missing ts/close; columns=%s", list(df.columns)[:10])
            return empty
        for k in ("open", "high", "low"):                    # tolerate close-only index feeds
            if k not in df.columns:
                df[k] = df["close"]
        if "volume" not in df.columns:
            df["volume"] = 0.0
        df["volume"] = pd.to_numeric(df["volume"], errors="coerce").fillna(0.0)

        ts = df["ts"]
        ts_num = pd.to_numeric(ts, errors="coerce")
        if ts_num.notna().all():                             # epoch seconds or milliseconds
            unit = "ms" if float(ts_num.max()) > 1e12 else "s"
            dt = pd.to_datetime(ts_num, unit=unit, utc=True).dt.tz_convert(IST).dt.tz_localize(None)
        else:                                                # ISO strings, tz-aware or naive
            dt = pd.to_datetime(ts, errors="coerce", format="mixed")
            if getattr(dt.dt, "tz", None) is not None:
                dt = dt.dt.tz_convert(IST).dt.tz_localize(None)
        df["date"] = dt.dt.normalize()
        df = df.dropna(subset=["date"]).drop(columns=["ts"]).set_index("date")
        df = df[["open", "high", "low", "close", "volume"]].apply(pd.to_numeric, errors="coerce")
        df = df.dropna(subset=["close"])
        return df[~df.index.duplicated(keep="last")].sort_index()

    @retry(group="live_data")
    def ltp(self, symbols: Sequence[str]) -> Dict[str, float]:
        api, out = self._api(), {}
        syms = list(symbols)
        for i in range(0, len(syms), 50):                     # API caps at 50 per call
            chunk = tuple(f"NSE_{s}" for s in syms[i:i + 50])
            r = api.get_ltp(segment=api.SEGMENT_CASH, exchange_trading_symbols=chunk)
            for k, v in (r or {}).items():
                out[k.replace("NSE_", "")] = float(v)
        return out

    @retry(group="live_data")
    def quote(self, symbol: str) -> Dict[str, Any]:
        api = self._api()
        q = api.get_quote(exchange=api.EXCHANGE_NSE, segment=api.SEGMENT_CASH, trading_symbol=symbol) or {}
        if not self._quote_keys_logged and q:
            # Logged once per session: tells you which fields Groww actually returns. If
            # `market_cap` is absent or 0, the universe falls back to a turnover proxy for size.
            self._quote_keys_logged = True
            LOG_BROKER.info("quote payload keys (sample %s): %s", symbol, sorted(q.keys()))
            LOG_BROKER.info("quote sample %s: last_price=%s market_cap=%s volume=%s 52wH=%s",
                            symbol, q.get("last_price"), q.get("market_cap"), q.get("volume"),
                            q.get("week_52_high"))
        return {
            "symbol": symbol,
            "ltp": float(q.get("last_price", q.get("ltp", 0)) or 0),
            "open": float((q.get("ohlc") or {}).get("open", 0) or 0),
            "high": float((q.get("ohlc") or {}).get("high", 0) or 0),
            "low": float((q.get("ohlc") or {}).get("low", 0) or 0),
            "close": float((q.get("ohlc") or {}).get("close", 0) or 0),
            "volume": float(q.get("volume", 0) or 0),
            "upper_circuit": float(q.get("upper_circuit_limit", 0) or 0),
            "lower_circuit": float(q.get("lower_circuit_limit", 0) or 0),
            "bid": float(q.get("bid_price", 0) or 0),
            "ask": float(q.get("offer_price", q.get("ask_price", 0)) or 0),
            # The quote payload carries market cap and 52-week levels — a whole universe screen
            # without touching the historical API. Market cap comes in rupees; convert to crore.
            "market_cap_cr": float(q.get("market_cap", 0) or 0) / 1e7,
            "week_52_high": float(q.get("week_52_high", 0) or 0),
            "week_52_low": float(q.get("week_52_low", 0) or 0),
            "raw": q,
        }

    # ---------- orders ----------
    @retry(times=1, group="orders", idempotent=False)
    def place_order(self, symbol: str, txn: str, qty: int, order_type: str,
                    price: float = 0.0, trigger_price: float = 0.0,
                    ref_id: str = "", product: str = "CNC", **_) -> Dict[str, Any]:
        api = self._api()
        ot = {
            OrdType.MARKET: api.ORDER_TYPE_MARKET,
            OrdType.LIMIT: api.ORDER_TYPE_LIMIT,
            OrdType.SL: api.ORDER_TYPE_STOP_LOSS,
            OrdType.SL_M: api.ORDER_TYPE_STOP_LOSS_MARKET,
        }[OrdType(order_type)]
        payload = dict(
            trading_symbol=symbol,
            quantity=int(qty),
            validity=api.VALIDITY_DAY,
            exchange=api.EXCHANGE_NSE,
            segment=api.SEGMENT_CASH,
            product=api.PRODUCT_CNC if product == "CNC" else api.PRODUCT_MIS,
            order_type=ot,
            transaction_type=api.TRANSACTION_TYPE_BUY if txn == Txn.BUY else api.TRANSACTION_TYPE_SELL,
        )
        if price:
            payload["price"] = round(float(price), 2)
        if trigger_price:
            payload["trigger_price"] = round(float(trigger_price), 2)
        if ref_id:
            payload["order_reference_id"] = ref_id[:20]
        if self.cfg.dry_run:
            LOG_BROKER.warning("[DRY-RUN] would place: %s", payload)
            return {"order_id": f"DRY-{uuid.uuid4().hex[:12]}", "status": OrdStatus.NEW, "raw": payload}
        LOG_BROKER.warning("PLACING LIVE ORDER: %s", payload)
        r = api.place_order(**payload) or {}
        LOG_BROKER.info("order response: %s", r)
        return {"order_id": r.get("groww_order_id", ""),
                "status": GROWW_STATUS_MAP.get(str(r.get("order_status", "")).upper(), OrdStatus.NEW),
                "raw": r}

    @retry(times=1, group="orders", idempotent=False)
    def modify_order(self, order_id: str, qty: Optional[int] = None, price: Optional[float] = None,
                     trigger_price: Optional[float] = None, order_type: Optional[str] = None) -> Dict[str, Any]:
        api = self._api()
        kw: Dict[str, Any] = {"groww_order_id": order_id, "segment": api.SEGMENT_CASH}
        if qty is not None:
            kw["quantity"] = int(qty)
        if price is not None:
            kw["price"] = round(float(price), 2)
        if trigger_price is not None:
            kw["trigger_price"] = round(float(trigger_price), 2)
        if order_type is not None:
            kw["order_type"] = order_type
        if self.cfg.dry_run:
            LOG_BROKER.warning("[DRY-RUN] would modify: %s", kw)
            return {"order_id": order_id, "status": OrdStatus.OPEN, "raw": kw}
        LOG_BROKER.warning("MODIFYING LIVE ORDER: %s", kw)
        return {"order_id": order_id, "raw": api.modify_order(**kw)}

    @retry(times=1, group="orders", idempotent=False)
    def cancel_order(self, order_id: str) -> Dict[str, Any]:
        api = self._api()
        if self.cfg.dry_run:
            LOG_BROKER.warning("[DRY-RUN] would cancel %s", order_id)
            return {"order_id": order_id, "status": OrdStatus.CANCELLED}
        LOG_BROKER.warning("CANCELLING LIVE ORDER %s", order_id)
        return {"order_id": order_id, "raw": api.cancel_order(groww_order_id=order_id, segment=api.SEGMENT_CASH)}

    @retry(group="orders")
    def order_detail(self, order_id: str) -> Dict[str, Any]:
        api = self._api()
        r = api.get_order_detail(groww_order_id=order_id, segment=api.SEGMENT_CASH) or {}
        return {
            "order_id": order_id,
            "status": GROWW_STATUS_MAP.get(str(r.get("order_status", "")).upper(), OrdStatus.PENDING),
            "filled_qty": int(r.get("filled_quantity", 0) or 0),
            "avg_price": float(r.get("average_fill_price", 0) or 0),
            "raw": r,
        }

    @retry(group="non_trading")
    def positions(self) -> pd.DataFrame:
        api = self._api()
        r = api.get_positions_for_user(segment=api.SEGMENT_CASH) or {}
        return pd.DataFrame(r.get("positions", r) or [])

    @retry(group="non_trading")
    def holdings(self) -> pd.DataFrame:
        r = self._api().get_holdings_for_user(timeout=5) or {}
        return pd.DataFrame(r.get("holdings", r) or [])


class PaperBroker(BrokerAdapter):
    """Deterministic simulator. Fills are resolved against the reference bar you feed it."""
    name = "paper"

    def __init__(self, cfg: Config, store: Store):
        self.cfg, self.store = cfg, store
        self.cash = cfg.equity
        self.book: Dict[str, Dict[str, Any]] = {}
        self.open_orders: Dict[str, Dict[str, Any]] = {}
        self.marks: Dict[str, float] = {}
        self.bars: Dict[str, Dict[str, float]] = {}   # symbol -> today's OHLC, set by the engine

    def connect(self) -> None:
        LOG_BROKER.info("PaperBroker ready | cash=₹%s", f"{self.cash:,.0f}")

    def set_bar(self, symbol: str, bar: Dict[str, float]) -> None:
        self.bars[symbol] = bar
        self.marks[symbol] = bar.get("close", self.marks.get(symbol, 0.0))

    def ltp(self, symbols: Sequence[str]) -> Dict[str, float]:
        return {s: self.marks.get(s, 0.0) for s in symbols}

    def quote(self, symbol: str) -> Dict[str, Any]:
        b = self.bars.get(symbol, {})
        return {"symbol": symbol, "ltp": self.marks.get(symbol, 0.0), **b}

    def place_order(self, symbol: str, txn: str, qty: int, order_type: str,
                    price: float = 0.0, trigger_price: float = 0.0, ref_id: str = "", **_):
        oid = f"PAP-{uuid.uuid4().hex[:12]}"
        LOG_BROKER.info("[paper] order %s: %s %s x%d %s px=%.2f trg=%.2f", oid, txn, symbol,
                        int(qty), order_type, price or 0.0, trigger_price or 0.0)
        self.open_orders[oid] = dict(order_id=oid, symbol=symbol, txn=txn, qty=int(qty),
                                     order_type=order_type, price=price, trigger_price=trigger_price,
                                     status=OrdStatus.OPEN, filled_qty=0, avg_price=0.0, ref_id=ref_id)
        return {"order_id": oid, "status": OrdStatus.OPEN, "raw": self.open_orders[oid]}

    def modify_order(self, order_id: str, qty=None, price=None, trigger_price=None, order_type=None):
        o = self.open_orders.get(order_id)
        if not o:
            return {"order_id": order_id, "status": OrdStatus.REJECTED}
        if qty is not None:
            o["qty"] = int(qty)
        if price is not None:
            o["price"] = price
        if trigger_price is not None:
            o["trigger_price"] = trigger_price
        if order_type is not None:
            o["order_type"] = order_type
        return {"order_id": order_id, "status": OrdStatus.OPEN, "raw": o}

    def cancel_order(self, order_id: str):
        o = self.open_orders.pop(order_id, None)
        if o:
            o["status"] = OrdStatus.CANCELLED
        return {"order_id": order_id, "status": OrdStatus.CANCELLED}

    def order_detail(self, order_id: str):
        o = self.open_orders.get(order_id, {"status": OrdStatus.FILLED, "filled_qty": 0, "avg_price": 0.0})
        return {"order_id": order_id, "status": o.get("status"),
                "filled_qty": o.get("filled_qty", 0), "avg_price": o.get("avg_price", 0.0), "raw": o}

    # ---- the simulated matching engine ----
    def match(self, symbol: str, bar: Dict[str, float], slip_bps: float) -> List[Dict[str, Any]]:
        """Resolve resting orders for `symbol` against one daily bar. Returns fill dicts."""
        fills = []
        o_, h, l, c = bar["open"], bar["high"], bar["low"], bar["close"]
        for oid, o in list(self.open_orders.items()):
            if o["symbol"] != symbol or o["status"] != OrdStatus.OPEN:
                continue
            slip = slip_bps / 10_000.0
            px = None
            if o["order_type"] == OrdType.MARKET:
                px = o_ * (1 + slip) if o["txn"] == Txn.BUY else o_ * (1 - slip)
            elif o["order_type"] == OrdType.LIMIT:
                if o["txn"] == Txn.BUY and l <= o["price"]:
                    px = min(o["price"], o_)
                elif o["txn"] == Txn.SELL and h >= o["price"]:
                    px = max(o["price"], o_)
            elif o["order_type"] in (OrdType.SL, OrdType.SL_M):
                trg = o["trigger_price"]
                if o["txn"] == Txn.BUY and h >= trg:      # buy-stop breakout entry
                    px = max(trg, o_) * (1 + slip)
                elif o["txn"] == Txn.SELL and l <= trg:   # protective stop
                    px = min(trg, o_) * (1 - slip)
            if px is None:
                continue
            qty = o["qty"]
            if o["txn"] == Txn.BUY:
                cost = px * qty
                if cost > self.cash:
                    qty = int(self.cash // px)
                    if qty <= 0:
                        o["status"] = OrdStatus.REJECTED
                        self.open_orders.pop(oid, None)
                        continue
                    cost = px * qty
                self.cash -= cost
                p = self.book.setdefault(symbol, {"qty": 0, "avg": 0.0})
                p["avg"] = (p["avg"] * p["qty"] + px * qty) / (p["qty"] + qty)
                p["qty"] += qty
            else:
                p = self.book.get(symbol, {"qty": 0, "avg": 0.0})
                qty = min(qty, p["qty"])
                if qty <= 0:
                    self.open_orders.pop(oid, None)
                    continue
                self.cash += px * qty
                p["qty"] -= qty
                if p["qty"] == 0:
                    self.book.pop(symbol, None)
            o.update(status=OrdStatus.FILLED, filled_qty=qty, avg_price=px)
            self.open_orders.pop(oid, None)
            fills.append({"order_id": oid, "symbol": symbol, "txn": o["txn"], "qty": qty,
                          "price": px, "purpose": o.get("ref_id", "")})
            LOG_BROKER.info("[paper] FILL %s %s x%d @ %.2f · cash now ₹%s", o["txn"], symbol, qty,
                            px, f"{self.cash:,.0f}")
        return fills

    def positions(self) -> pd.DataFrame:
        return pd.DataFrame([{"symbol": s, **v, "ltp": self.marks.get(s, 0.0)}
                             for s, v in self.book.items()])

    def holdings(self) -> pd.DataFrame:
        return self.positions()


def make_broker(cfg: Config, store: Store) -> BrokerAdapter:
    if cfg.mode == "LIVE":
        b = GrowwBroker(cfg)
    else:
        b = PaperBroker(cfg, store)
    with log_step(f"connect {b.name} broker", LOG_BROKER):
        b.connect()
    return b


BROKER = make_broker(CFG, STORE)
LOG_BROKER.info("broker=%s · mode=%s · dry_run=%s%s", BROKER.name, CFG.mode, CFG.dry_run,
                "  ⚠ LIVE ORDERS ENABLED" if CFG.mode == "LIVE" and not CFG.dry_run else "")

07:49:10 | INFO    | engine.cell       | ▶ START  [05] Broker abstraction (Groww live + paper simulator)
07:49:10 | INFO    | engine.broker     | → connect groww broker ...
07:49:10 | INFO    | engine.broker     | access token expires 2026-09-13 06:00 IST (in 16.7 h)
Ready to Groww!
07:49:11 | INFO    | engine.broker     | Groww connected via access token; will reconnect after 2026-09-13 05:45 IST
07:49:11 | INFO    | engine.broker     | ← connect groww broker done in 0.97s
07:49:11 | INFO    | engine.broker     | broker=groww · mode=LIVE · dry_run=True
07:49:11 | INFO    | engine.cell       | ✔ DONE   [05] Broker abstraction (Groww live + paper simulator) in 1.0s


## 6 · Data Hub

A chained provider with an incremental SQLite cache in front of it:

```
get_daily(symbol) ──► cache hit? ──► fetch only the missing tail ──► write back ──► return
                           │
                           └─► provider chain: GROWW → YFINANCE → CSV → SYNTHETIC
```

The tail-fetch matters: refreshing 400 small caps daily is 400 one-bar requests, not 400
three-year requests. With a 3 req/s historical limit that is a couple of minutes instead of hours.

Two rules this layer enforces:

1. **Requests are chunked to 175 days** — Groww rejects any 1-day-interval request spanning more
   than 180 days.
2. **A failed live fetch never returns synthetic data** unless you explicitly opt in with
   `allow_synthetic_fallback=True`. Silently substituting fabricated bars for a name whose fetch
   failed produces a universe that looks perfectly normal and is partly fiction. Failures return
   empty, get logged to the `events` table, and repeat offenders are memoised in `DEAD_SYMBOLS`
   so you stop paying API quota for them.

**Benchmark handling (patched).** The benchmark is special-cased: it is resolved from the Groww
instrument master to its exact `groww_symbol`, it can never enter `DEAD_SYMBOLS`, and it is
validated (`min_benchmark_bars`, `max_benchmark_staleness_days`) before use. Source chain:
Groww → yfinance (`benchmark_yf_tickers`) → equal-weight universe proxy
(`benchmark_allow_universe_proxy`). `HUB.bench_source` records which one was used.

In [101]:
# [06] Data Hub — providers, cache, benchmark resolution
class SyntheticMarket:
    """Reproducible fake NSE small-cap tape. Lets the whole notebook run with no keys.

    Not a toy random walk: names get persistent trend regimes, fat-tailed returns, volume that
    expands on breakout days, occasional gaps and circuit-like limit days.
    """

    SECTORS = ["Chemicals", "Capital Goods", "IT Services", "Pharma", "Auto Anc",
               "Textiles", "Realty", "Financials", "FMCG", "Metals"]

    def __init__(self, n_symbols: int = 60, years: int = 5, seed: int = 7):
        self.rng = np.random.default_rng(seed)
        self.n, self.years = n_symbols, years
        self.symbols = [f"SC{i:03d}" for i in range(n_symbols)]
        self.dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=252 * years)
        self.meta = pd.DataFrame({
            "symbol": self.symbols,
            "name": [f"Smallcap Industries {i}" for i in range(n_symbols)],
            "sector": [self.SECTORS[i % len(self.SECTORS)] for i in range(n_symbols)],
            "mcap_cr": self.rng.uniform(600, 18_000, n_symbols).round(0),
            "series": "EQ", "instrument_type": "EQ", "exchange": "NSE", "segment": "CASH",
        })
        self._cache: Dict[str, pd.DataFrame] = {}
        self._bench: Optional[pd.DataFrame] = None

    def _path(self, seed_shift: int, drift_scale: float = 1.0) -> pd.DataFrame:
        rng = np.random.default_rng(1000 + seed_shift)
        n = len(self.dates)
        # regime-switching drift: bull / chop / bear blocks
        drift = np.zeros(n)
        i = 0
        while i < n:
            blk = int(rng.integers(70, 260))                   # trends persist for months
            state = rng.choice([0.0022, 0.0002, -0.0016], p=[0.44, 0.31, 0.25])
            drift[i:i + blk] = state * drift_scale
            i += blk
        vol = np.abs(rng.normal(0.021, 0.006)) * drift_scale
        shock = rng.standard_t(4, n) / np.sqrt(2.0)          # fat tails
        ret = drift + vol * shock
        close = 100 * np.exp(np.cumsum(ret))
        close *= float(rng.uniform(0.4, 9.0))                 # price level spread
        rng2 = np.random.default_rng(2000 + seed_shift)
        rng_hl = np.abs(rng2.normal(0.012, 0.006, n)) + 0.004
        open_ = close * (1 + rng2.normal(0, 0.004, n))
        high = np.maximum(open_, close) * (1 + rng_hl)
        low = np.minimum(open_, close) * (1 - rng_hl)
        base_vol = float(rng2.uniform(4e4, 9e5))
        vmult = 1 + 3.5 * np.clip(ret / (vol + 1e-9), 0, None) ** 1.5
        volume = base_vol * vmult * np.exp(rng2.normal(0, 0.35, n))
        return pd.DataFrame({"open": open_, "high": high, "low": low,
                             "close": close, "volume": volume.round()}, index=self.dates)

    def daily(self, symbol: str) -> pd.DataFrame:
        if symbol not in self._cache:
            idx = self.symbols.index(symbol) if symbol in self.symbols else abs(hash(symbol)) % 997
            self._cache[symbol] = self._path(idx)
        return self._cache[symbol]

    def benchmark(self) -> pd.DataFrame:
        if self._bench is None:
            closes = pd.concat([self.daily(s)["close"] / self.daily(s)["close"].iloc[0]
                                for s in self.symbols], axis=1)
            idx = closes.mean(axis=1) * 1000
            self._bench = pd.DataFrame({"open": idx, "high": idx * 1.004,
                                        "low": idx * 0.996, "close": idx, "volume": 0.0})
        return self._bench


SYNTH = SyntheticMarket(n_symbols=max(CFG.smoke_symbols, 24), years=max(CFG.smoke_years, 4), seed=CFG.random_seed)
LOG_DATA.info("synthetic market ready (%d symbols) — used only when data_source=SYNTHETIC "
              "or allow_synthetic_fallback=True", SYNTH.n)

OHLCV = ["open", "high", "low", "close", "volume"]


def _empty_bars() -> pd.DataFrame:
    return pd.DataFrame(columns=OHLCV)


# Symbols the provider has told us do not exist / have no data. Persisted so a restart doesn't
# re-spend quota rediscovering them.
DEAD_SYMBOLS: set = set(STORE.get_kv("dead_symbols", []) or [])

# Earliest date we have already *asked* for, per symbol. Without this the two-stage universe build
# would re-request deep history every run for names that simply listed recently.
HISTORY_FLOOR: Dict[str, str] = dict(STORE.get_kv("history_floor", {}) or {})

# PATCH: the benchmark must never be dead-listed. The first failed "NIFTY_SMLCAP_250" request put
# it in DEAD_SYMBOLS, and from then on every run skipped it *silently* — even after a fix.
_PROTECTED_AT_LOAD = {CFG.benchmark, "NIFTY_SMLCAP_250"}
_resurrected = sorted(DEAD_SYMBOLS & _PROTECTED_AT_LOAD)
if _resurrected:
    DEAD_SYMBOLS.difference_update(_resurrected)
    STORE.set_kv("dead_symbols", sorted(DEAD_SYMBOLS))
    LOG_DATA.warning("removed benchmark name(s) %s from the persisted DEAD_SYMBOLS memo — this is "
                     "what made the benchmark come back empty on every run", _resurrected)
LOG_DATA.info("dead-symbol memo: %d names%s", len(DEAD_SYMBOLS),
              f" (e.g. {sorted(DEAD_SYMBOLS)[:8]})" if DEAD_SYMBOLS else "")
LOG_DATA.info("history-floor memo: %d names", len(HISTORY_FLOOR))


class DataHub:
    def __init__(self, cfg: Config, broker: BrokerAdapter, store: Store, synth: SyntheticMarket):
        self.cfg, self.broker, self.store, self.synth = cfg, broker, store, synth
        self.protected: set = {cfg.benchmark}          # never dead-listed
        self.bench_source: str = "unresolved"          # where the benchmark bars came from
        self.bench_symbol: Optional[str] = None
        self.stats: Dict[str, int] = defaultdict(int)
        self._stats_lock = threading.Lock()

    def _count(self, key: str, n: int = 1) -> None:
        with self._stats_lock:
            self.stats[key] += n

    # ---------- providers ----------
    @staticmethod
    def _yf_download(ticker: str, start: date, end: date) -> pd.DataFrame:
        if yf is None:
            raise RuntimeError("yfinance not installed")
        d = yf.download(ticker, start=start, end=end + timedelta(days=1),
                        progress=False, auto_adjust=True, actions=True)
        if d is None or d.empty:
            return _empty_bars()
        if isinstance(d.columns, pd.MultiIndex):
            d.columns = d.columns.get_level_values(0)
        d = d.rename(columns=str.lower)
        if "volume" not in d.columns:
            d["volume"] = 0.0
        d = d[OHLCV]
        d.index = pd.to_datetime(d.index).tz_localize(None).normalize()
        return d.dropna(subset=["close"])

    def _fetch(self, symbol: str, start: date, end: date) -> pd.DataFrame:
        empty = _empty_bars()
        protected = symbol in self.protected
        if symbol in DEAD_SYMBOLS and not protected:
            LOG_DATA.debug("skip %s — memoised in DEAD_SYMBOLS", symbol)
            self._count("dead_skipped")
            return empty
        src = self.cfg.data_source
        self._count("provider_calls")
        if src == "SYNTHETIC":
            d = self.synth.daily(symbol)
            return d.loc[(d.index >= pd.Timestamp(start)) & (d.index <= pd.Timestamp(end))]
        try:
            if src == "GROWW":
                d = self.broker.daily_candles(symbol, start, end)
            elif src == "YFINANCE":
                d = self._yf_download(f"{symbol}.NS", start, end)
            elif src == "CSV":
                p = os.path.join(self.cfg.csv_dir, f"{symbol}.csv")
                if not os.path.exists(p):
                    LOG_DATA.debug("CSV missing for %s: %s", symbol, p)
                    return empty
                d = pd.read_csv(p, parse_dates=["date"]).set_index("date")
                d.columns = [c.lower() for c in d.columns]
                d = d[OHLCV]
            else:
                LOG_DATA.error("unknown data_source %r", src)
                return empty
            LOG_DATA.debug("fetched %s (%s) %s → %s: %s", symbol, src, start, end, describe_frame(d))
            if d is None or d.empty:
                self._count("provider_empty")
            return d if d is not None else empty
        except Exception as e:  # noqa: BLE001
            msg = str(e)
            self._count("provider_errors")
            if any(k in msg.lower() for k in ("not found", "invalid", "no data")) and not protected:
                DEAD_SYMBOLS.add(symbol)              # don't pay for this symbol again
                LOG_DATA.info("memoised %s as dead (provider said: %s)", symbol, msg[:80])
            (LOG_BENCH if protected else LOG_DATA).warning(
                "fetch failed %s (%s) %s → %s: %s", symbol, src, start, end, msg[:200])
            self.store.event("FETCH_FAIL", msg[:300], level="WARN", symbol=symbol)
            if self.cfg.allow_synthetic_fallback:
                LOG_DATA.warning("SYNTHETIC FALLBACK for %s — these bars are FABRICATED", symbol)
                d = self.synth.daily(symbol)
                return d.loc[(d.index >= pd.Timestamp(start)) & (d.index <= pd.Timestamp(end))]
        return empty

    # ---------- public ----------
    def get_daily(self, symbol: str, lookback_days: int = 900, refresh: bool = True) -> pd.DataFrame:
        """Cache-first, with *two* gap checks. Checking only the tail is a trap: after the stage-1
        screen the cache holds 200 days, and a later 1,200-day request would be served from that
        cache and silently come back 200 days short."""
        end = now_ist().date()
        start = end - timedelta(days=lookback_days)
        cached = self.store.load_candles(symbol, start.isoformat())
        parts = [cached] if not cached.empty else []

        if cached.empty:
            LOG_DATA.debug("%s: cache empty → full fetch %s → %s", symbol, start, end)
            fresh = self._fetch(symbol, start, end)
            HISTORY_FLOOR[symbol] = min(HISTORY_FLOOR.get(symbol, "9999"), start.isoformat())
            if fresh is None or fresh.empty:
                self._count("empty_result")
                return cached
            self.store.save_candles(symbol, fresh)
            self._count("full_fetch")
            return fresh.sort_index()

        # --- head gap: cache doesn't reach far enough back ---
        touched = False
        head_have = cached.index.min().date()
        asked_before = HISTORY_FLOOR.get(symbol, "9999")
        if head_have > start + timedelta(days=7) and asked_before > start.isoformat():
            LOG_DATA.debug("%s: head gap — cache starts %s, need %s", symbol, head_have, start)
            head = self._fetch(symbol, start, head_have - timedelta(days=1))
            HISTORY_FLOOR[symbol] = start.isoformat()
            self._count("head_fill")
            touched = True
            if head is not None and not head.empty:
                self.store.save_candles(symbol, head)
                parts.append(head)

        # --- tail gap: cache is stale ---
        last = cached.index.max().date()
        if refresh and last < end - timedelta(days=1):
            LOG_DATA.debug("%s: tail gap — cache ends %s, today %s", symbol, last, end)
            tail = self._fetch(symbol, last - timedelta(days=3), end)   # overlap repairs revisions
            self._count("tail_fill")
            touched = True
            if tail is not None and not tail.empty:
                self.store.save_candles(symbol, tail)
                parts.append(tail)
        if not touched:
            self._count("cache_only")

        if len(parts) == 1:
            return parts[0].sort_index()
        merged = pd.concat(parts)                    # all parts non-empty: no pandas FutureWarning
        return merged[~merged.index.duplicated(keep="last")].sort_index()

    def bulk(self, symbols: Sequence[str], lookback_days: int = 900, workers: int = 3,
             refresh: bool = True, label: str = "fetch") -> Dict[str, pd.DataFrame]:
        """Parallel fetch with progress and an ETA. Safe to interrupt: each symbol is committed to
        SQLite as it lands, so a re-run resumes from the cache instead of starting over."""
        out: Dict[str, pd.DataFrame] = {}
        syms = list(symbols)
        n = max(len(syms), 1)
        t0 = time.time()
        before = dict(self.stats)
        LOG_DATA.info("%s: %d symbols · %d days lookback · refresh=%s · workers=%d",
                      label, len(syms), lookback_days, refresh, workers)

        def _report(done: int) -> None:
            if done % 50 == 0 or done == n:
                el = time.time() - t0
                LOG_DATA.info("%s %d/%d (%.0f%%) · %.0fs elapsed · ~%.0fs left",
                              label, done, n, done / n * 100, el, el / max(done, 1) * (n - done))

        if workers <= 1 or self.cfg.data_source == "SYNTHETIC":
            for i, sym in enumerate(syms, 1):
                try:
                    out[sym] = self.get_daily(sym, lookback_days, refresh)
                except Exception as e:  # noqa: BLE001
                    LOG_DATA.warning("bulk fetch failed %s: %s", sym, str(e)[:120])
                    out[sym] = _empty_bars()
                _report(i)
        else:
            from concurrent.futures import ThreadPoolExecutor, as_completed
            with ThreadPoolExecutor(max_workers=workers) as ex:
                futs = {ex.submit(self.get_daily, sym, lookback_days, refresh): sym for sym in syms}
                for i, f in enumerate(as_completed(futs), 1):
                    sym = futs[f]
                    try:
                        out[sym] = f.result()
                    except Exception as e:  # noqa: BLE001
                        LOG_DATA.warning("bulk fetch failed %s: %s", sym, str(e)[:120])
                        out[sym] = _empty_bars()
                    _report(i)

        ok = [s for s, d in out.items() if d is not None and not d.empty]
        empty = [s for s in syms if s not in ok]
        delta = {k: v - before.get(k, 0) for k, v in self.stats.items() if v - before.get(k, 0)}
        lens = [len(out[s]) for s in ok]
        LOG_DATA.info("%s complete: %d/%d symbols returned data in %.0fs · bars min/median/max = %s",
                      label, len(ok), len(syms), time.time() - t0,
                      f"{min(lens)}/{int(np.median(lens))}/{max(lens)}" if lens else "n/a")
        LOG_DATA.info("%s cache/provider activity: %s", label, delta or "none")
        if empty:
            LOG_DATA.warning("%s: %d symbols came back EMPTY (e.g. %s)", label, len(empty), empty[:10])
        STORE.set_kv("dead_symbols", sorted(DEAD_SYMBOLS))
        STORE.set_kv("history_floor", HISTORY_FLOOR)
        return out

    # ---------- benchmark (PATCHED) ----------
    def resolve_benchmark_symbol(self) -> Optional[Tuple[str, str, str]]:
        """-> (trading_symbol, groww_symbol, description) for the benchmark index, or None.

        Groww identifies an index as "<EXCHANGE>-<TRADING_SYMBOL>" in the CASH segment (e.g.
        NSE-NIFTY), and the exact trading symbol lives in the instrument master — so look it up
        there instead of guessing a string like "NIFTY_SMLCAP_250"."""
        cfg = self.cfg
        if cfg.benchmark_groww_symbol:
            gs = cfg.benchmark_groww_symbol.strip()
            ts = gs.split("-", 1)[-1]
            LOG_BENCH.info("using benchmark_groww_symbol override: %s", gs)
            return ts, gs, "config override"
        if not isinstance(self.broker, GrowwBroker):
            LOG_BENCH.info("broker is %s — no Groww instrument master to resolve against",
                           self.broker.name)
            return None
        try:
            hits = self.broker.find_index_instruments(
                cfg.benchmark_search_terms,
                exact=(cfg.benchmark, "NIFTY SMALLCAP 250", "NIFTY SMLCAP 250"))
        except Exception as e:  # noqa: BLE001
            LOG_BENCH.error("instrument-master lookup failed: %s: %s", type(e).__name__, str(e)[:200])
            return None
        if hits.empty:
            LOG_BENCH.error("no CASH-segment instrument matches %s — set "
                            "CFG.benchmark_groww_symbol by hand", list(cfg.benchmark_search_terms))
            return None
        LOG_BENCH.info("instrument-master matches for the benchmark (best first):\n%s",
                       hits.head(8).to_string(index=False))
        best = hits.iloc[0]
        if not bool(best["is_index"]):
            LOG_BENCH.warning("best match %s has instrument_type=%r — not an index; it will be "
                              "used as a PROXY", best["trading_symbol"], best["instrument_type"])
        if not bool(best["exact"]):
            LOG_BENCH.warning("no exact 'Nifty Smallcap 250' match; closest is %r (%s) — check "
                              "it is the index you intend", best["name"], best["trading_symbol"])
        ts = str(best["trading_symbol"])
        gs = str(best["groww_symbol"]) or f"{best['exchange']}-{ts}"
        return ts, gs, f"{best['name'] or ts} [{best['instrument_type'] or '?'}]"

    def _validate_bench(self, df: Optional[pd.DataFrame]) -> Tuple[bool, str]:
        if df is None or df.empty:
            return False, "0 bars"
        if "close" not in df.columns or df["close"].dropna().empty:
            return False, "no close prices"
        n = int(df["close"].notna().sum())
        age = (now_ist().date() - pd.Timestamp(df.index.max()).date()).days
        desc = describe_frame(df)
        if n < self.cfg.min_benchmark_bars:
            return False, f"only {n} bars (< min_benchmark_bars={self.cfg.min_benchmark_bars}) · {desc}"
        if age > self.cfg.max_benchmark_staleness_days:
            LOG_BENCH.warning("benchmark last bar is %d days old (%s) — regime will be stale",
                              age, pd.Timestamp(df.index.max()).date())
        return True, desc

    def _last_fetch_error(self, symbol: str) -> str:
        try:
            q = self.store.query("SELECT message FROM events WHERE kind='FETCH_FAIL' AND symbol=? "
                                 "ORDER BY id DESC LIMIT 1", [symbol])
            return str(q.iloc[0]["message"]) if len(q) else ""
        except Exception:  # noqa: BLE001
            return ""

    def benchmark(self, lookback_days: int = 900) -> pd.DataFrame:
        """Resolve → fetch → validate, with fallbacks. Returns an EMPTY frame only if every source
        failed; `compute_panel` then decides (universe proxy, or a loud error)."""
        cfg = self.cfg
        attempts: List[str] = []
        with log_step(f"benchmark {cfg.benchmark} ({lookback_days}d)", LOG_BENCH):
            if cfg.data_source == "SYNTHETIC":
                b = self.synth.benchmark()
                self.bench_source, self.bench_symbol = "SYNTHETIC", "SYNTH_INDEX"
                b = b.iloc[-min(len(b), lookback_days):]
                LOG_BENCH.info("synthetic benchmark: %s", describe_frame(b))
                return b

            # 1. Groww, via the instrument master
            if cfg.data_source == "GROWW":
                res = self.resolve_benchmark_symbol()
                if res is None:
                    attempts.append("GROWW: symbol could not be resolved")
                else:
                    ts, gs, desc = res
                    if isinstance(self.broker, GrowwBroker):
                        self.broker.symbol_overrides[ts] = gs
                    self.protected.add(ts)
                    if ts in DEAD_SYMBOLS:
                        DEAD_SYMBOLS.discard(ts)
                        LOG_BENCH.warning("%s was in DEAD_SYMBOLS — removed", ts)
                    LOG_BENCH.info("fetching benchmark via Groww: %s → %s (%s)", cfg.benchmark, gs, desc)
                    df = self.get_daily(ts, lookback_days, refresh=True)
                    ok, why = self._validate_bench(df)
                    attempts.append(f"GROWW {gs}: {why}")
                    if ok:
                        self.bench_source, self.bench_symbol = f"GROWW:{gs}", ts
                        LOG_BENCH.info("✔ benchmark from Groww: %s", why)
                        return df
                    err = self._last_fetch_error(ts)
                    LOG_BENCH.warning("Groww benchmark unusable (%s)%s", why,
                                      f" · last provider error: {err[:200]}" if err else "")

            # 2. CSV source: a <benchmark>.csv next to the stock files
            if cfg.data_source == "CSV":
                df = self.get_daily(cfg.benchmark, lookback_days, refresh=True)
                ok, why = self._validate_bench(df)
                attempts.append(f"CSV {cfg.benchmark}.csv: {why}")
                if ok:
                    self.bench_source, self.bench_symbol = f"CSV:{cfg.benchmark}", cfg.benchmark
                    return df

            # 3. yfinance (free, no Groww quota)
            end = now_ist().date()
            start = end - timedelta(days=lookback_days)
            for t in cfg.benchmark_yf_tickers:
                if yf is None:
                    attempts.append("YFINANCE: yfinance not installed")
                    break
                try:
                    LOG_BENCH.info("trying yfinance ticker %s ...", t)
                    df = self._yf_download(t, start, end)
                except Exception as e:  # noqa: BLE001
                    df = _empty_bars()
                    LOG_BENCH.warning("yfinance %s failed: %s", t, str(e)[:160])
                ok, why = self._validate_bench(df)
                attempts.append(f"YFINANCE {t}: {why}")
                if ok:
                    self.bench_source, self.bench_symbol = f"YFINANCE:{t}", t
                    LOG_BENCH.warning("✔ benchmark from yfinance %s (not Groww): %s", t, why)
                    return df

            self.bench_source, self.bench_symbol = "UNAVAILABLE", None
            LOG_BENCH.error("benchmark UNAVAILABLE from every source:\n  - %s", "\n  - ".join(attempts))
            return _empty_bars()

    def universe_proxy(self, raw: Dict[str, pd.DataFrame], min_bars: int) -> pd.DataFrame:
        """Last resort: an equal-weight index of the universe's own daily returns (real prices,
        not fabricated). Daily returns are clipped at ±20% so a split or bad print can't dominate."""
        closes = {s: d["close"] for s, d in raw.items() if d is not None and len(d) >= min_bars}
        if len(closes) < 10:
            LOG_BENCH.error("universe proxy needs ≥10 names with ≥%d bars; have %d", min_bars, len(closes))
            return _empty_bars()
        px = pd.DataFrame(closes).sort_index()
        rets = px.pct_change(fill_method=None).clip(-0.20, 0.20)
        cnt = rets.notna().sum(axis=1)
        ew = rets.mean(axis=1, skipna=True)[cnt >= max(5, int(0.3 * len(closes)))].fillna(0.0)
        idx = 1000.0 * (1.0 + ew).cumprod()
        out = pd.DataFrame({"open": idx, "high": idx, "low": idx, "close": idx, "volume": 0.0})
        self.bench_source, self.bench_symbol = f"UNIVERSE_EW_PROXY({len(closes)} names)", None
        LOG_BENCH.warning("using an EQUAL-WEIGHT UNIVERSE PROXY as the benchmark (%d names): %s — "
                          "relative strength is now 'vs this universe', not vs Nifty Smallcap 250",
                          len(closes), describe_frame(out))
        return out


HUB = DataHub(CFG, BROKER, STORE, SYNTH)
LOG_DATA.info("data hub ready | source=%s · synthetic fallback=%s", CFG.data_source,
              CFG.allow_synthetic_fallback)


07:49:11 | INFO    | engine.cell       | ▶ START  [06] Data Hub — providers, cache, benchmark resolution
07:49:11 | INFO    | engine.data       | synthetic market ready (24 symbols) — used only when data_source=SYNTHETIC or allow_synthetic_fallback=True
07:49:11 | INFO    | engine.data       | dead-symbol memo: 0 names
07:49:11 | INFO    | engine.data       | history-floor memo: 24 names
07:49:11 | INFO    | engine.data       | data hub ready | source=GROWW · synthetic fallback=False
07:49:11 | INFO    | engine.cell       | ✔ DONE   [06] Data Hub — providers, cache, benchmark resolution in 0.0s


## 7 · Universe construction — what counts as a tradeable small cap

Built in **two stages**, because the naive version is unaffordable. NSE CASH has ~2,500 tradeable
equities and the historical API is the scarcest resource you have, so the design goal is simply:
*spend as few historical requests as possible.*

- **Stage 1 — screen on quotes, not candles.** `get_quote` returns `market_cap`, `volume`,
  `last_price` and the 52-week range in a single call, and it lives on the **Live Data** budget
  (10/s, 300/min) rather than the historical one. That is ~2,500 cheap calls instead of ~2,500
  scarce ones, and it gives a *real* market cap instead of the turnover proxy.
- **Stage 2 — deep history for survivors only.** `deep_history_days` of bars for the few hundred
  names that passed, then the strict 20-day-median-turnover gate using real bars.

Request budget, roughly: 2,500 Live-Data calls (~10 min) + 250 × 4 historical calls (~20 min).
The previous candle-based screen cost ~5,300 historical calls; this costs ~1,000.

**On `deep_history_days = 700`.** That is ~480 bars: enough for a 200-EMA, a 52-week high and
relative strength, i.e. everything the *live* screen needs, at 4 chunked requests per name. It is
**not** enough for a meaningful backtest — you get under a year of usable window after warm-up.
For research depth, run `backfill(universe["symbol"].tolist(), years=4, source="YFINANCE")` once
(Part III). It costs zero Groww quota, writes to the same `candles` table, and the deep bars are
then picked up automatically by the panel builder.

The build is **safe to interrupt**: every symbol is committed to SQLite as it arrives, so
re-running resumes from the cache. Progress and an ETA are logged as it goes.

On the full ~2,500-name NSE equity list the first live build is 30–60 minutes. It is **safe to
interrupt**: every symbol is committed to SQLite the moment it arrives, so re-running resumes from
the cache rather than starting over. Progress and an ETA are logged every 50 symbols.

The gates, in order of cost:

1. **Structural** — a **whitelist**: NSE, CASH, `series == "EQ"`. This is the one that bit me in
   the first live run. NSE CASH lists thousands of NCDs, bonds, SGBs, ETFs, REITs and InvITs
   alongside equities — symbols like `1075SCL35` (a 10.75% NCD) and `10CAGL27` sail straight
   through a series *blacklist* because their series codes (`N1`, `NA`, `NB`, …) were never on it.
   So: whitelist the series, reject symbols starting with a digit, reject non-equity name tokens,
   and require an `INE`/`INF` ISIN prefix.
2. **Size** — market cap inside the small-cap band (`₹500cr – ₹20,000cr` by default). Below the
   floor you are in operator territory; above it you are trading mid caps with small-cap risk
   settings. Feed real market caps via `mcap.csv`; the fallback derives a rough turnover proxy.
3. **Liquidity** — 20-day *median* traded value (not mean — one block deal shouldn't qualify a
   name) above `min_median_turnover_cr`, plus a price floor. This is the gate that actually
   determines whether your backtest is fiction.
4. **Surveillance** — your own ASM / GSM / T2T exclusion list. NSE publishes these daily; keep
   `exclusions.csv` current or you will get filled into a 5% circuit and be unable to exit.

In [102]:
# [07] Universe construction
def load_exclusions(cfg: Config) -> set:
    ex = set()
    if not os.path.exists(cfg.exclusion_file):
        LOG_UNIV.warning("no surveillance exclusion file at %s — ASM/GSM/T2T names are NOT being "
                         "excluded", cfg.exclusion_file)
    if os.path.exists(cfg.exclusion_file):
        try:
            df = pd.read_csv(cfg.exclusion_file)
            col = "symbol" if "symbol" in df.columns else df.columns[0]
            ex = set(df[col].astype(str).str.upper().str.strip())
            LOG_UNIV.info("loaded %d surveillance exclusions from %s", len(ex), cfg.exclusion_file)
        except Exception as e:  # noqa: BLE001
            LOG_UNIV.warning("exclusion file unreadable: %s", e)
    return ex


def load_mcap_map(path: str = "./mcap.csv") -> Dict[str, float]:
    if os.path.exists(path):
        df = pd.read_csv(path)
        m = dict(zip(df.iloc[:, 0].astype(str).str.upper(), df.iloc[:, 1].astype(float)))
        LOG_UNIV.info("loaded %d market caps from %s", len(m), path)
        return m
    LOG_UNIV.info("no %s — market caps come from quotes (or a turnover proxy)", path)
    return {}


NON_EQUITY_TOKENS = ("NCD", "BOND", "DEBENTURE", "SGB", "GOI", "GSEC", "TBILL", "ETF",
                     "MFUNIT", "REIT", "INVIT", "PARTLY", "WARRANT", "RIGHTS")


def candidate_instruments(cfg: Config, broker: BrokerAdapter, synth: SyntheticMarket) -> pd.DataFrame:
    """Structural filter over the instrument master. Whitelist, never blacklist — every debenture
    that slips through costs API quota and pollutes the screen."""
    if cfg.data_source == "SYNTHETIC" or not isinstance(broker, GrowwBroker):
        LOG_UNIV.info("using the synthetic instrument list (%d names) — source=%s broker=%s",
                      len(synth.meta), cfg.data_source, broker.name)
        return synth.meta.copy()

    df = broker.instruments().copy()
    df.columns = [c.lower() for c in df.columns]
    for c in ("exchange", "segment", "series", "instrument_type", "trading_symbol", "name", "isin"):
        if c not in df.columns:
            df[c] = ""
        df[c] = df[c].astype(str).str.strip()

    gates = [
        ("exchange == NSE", df["exchange"].str.upper().eq("NSE")),
        ("segment == CASH", df["segment"].str.upper().eq("CASH")),
        (f"series in {list(cfg.equity_series)}",
         df["series"].str.upper().isin([x.upper() for x in cfg.equity_series])),
        ("instrument_type is equity", df["instrument_type"].str.upper().isin(["EQ", "EQUITY", "STOCK", ""])),
        ("symbol not digit-led", ~df["trading_symbol"].str.match(r"^\d")),   # 1075SCL35 … are NCDs
        ("symbol well-formed", df["trading_symbol"].str.match(r"^[A-Z][A-Z0-9&\-]{1,19}$")),
        ("name not non-equity", ~df["name"].str.upper().str.contains("|".join(NON_EQUITY_TOKENS), na=False)),
        ("symbol not non-equity",
         ~df["trading_symbol"].str.upper().str.contains("|".join(NON_EQUITY_TOKENS), na=False)),
    ]
    keep = pd.Series(True, index=df.index)
    for label, g in gates:                       # cumulative, so each line shows what *that* gate cut
        before = int(keep.sum())
        keep &= g.fillna(False).astype(bool)
        LOG_UNIV.info("  gate %-28s %6d → %6d", label, before, int(keep.sum()))
    if df["isin"].str.len().gt(0).any():
        before = int(keep.sum())
        keep &= (df["isin"].str.startswith("INE") | df["isin"].str.startswith("INF")
                 | df["isin"].eq(""))
        LOG_UNIV.info("  gate %-28s %6d → %6d", "ISIN INE/INF", before, int(keep.sum()))
    for c in ("buy_allowed", "sell_allowed"):
        if c in df.columns:
            before = int(keep.sum())
            keep &= pd.to_numeric(df[c], errors="coerce").fillna(1).gt(0)
            LOG_UNIV.info("  gate %-28s %6d → %6d", c, before, int(keep.sum()))

    out = (df.loc[keep, [c for c in ("trading_symbol", "groww_symbol", "name", "series",
                                     "instrument_type", "exchange", "segment", "isin",
                                     "exchange_token", "tick_size", "lot_size")
                         if c in df.columns]]
             .rename(columns={"trading_symbol": "symbol"})
             .drop_duplicates("symbol")
             .reset_index(drop=True))
    LOG_UNIV.info("instrument master: %d rows → %d equities (dropped %d non-equity rows)",
                  len(df), len(out), len(df) - len(out))
    return out


def prefilter_on_quotes(cfg: Config, broker: BrokerAdapter, symbols: Sequence[str]) -> pd.DataFrame:
    """One Live-Data quote per symbol: price, market cap, day volume, 52-week range.
    Costs nothing from the historical budget."""
    rows, n, t0 = [], len(symbols), time.time()
    failed: List[str] = []
    zero_px: List[str] = []
    errors: Dict[str, int] = defaultdict(int)
    LOG_UNIV.info("stage 1 (quotes): %d symbols · est. %.0f min at the live-data limit", n, n / 250)
    for i, sym in enumerate(symbols, 1):
        try:
            q = broker.quote(sym)
        except Exception as e:  # noqa: BLE001
            LOG_UNIV.debug("quote failed %s: %s", sym, str(e)[:80])
            failed.append(sym)
            errors[type(e).__name__ + ": " + str(e)[:60]] += 1
            if len(failed) == 25 and not rows:
                LOG_UNIV.error("the first 25 quotes all FAILED — most common error: %s",
                               max(errors, key=errors.get))
            continue
        px = float(q.get("ltp") or 0)
        if px <= 0:
            zero_px.append(sym)
            continue
        rows.append({"symbol": sym, "price": px,
                     "mcap_cr": float(q.get("market_cap_cr") or 0),
                     "day_turnover_cr": px * float(q.get("volume") or 0) / 1e7,
                     "day_volume": float(q.get("volume") or 0),
                     "week_52_high": float(q.get("week_52_high") or 0)})
        if i % 100 == 0 or i == n:
            el = time.time() - t0
            LOG_UNIV.info("quotes %d/%d (%.0f%%) · %.0fs elapsed · ~%.0fs left · ok=%d failed=%d",
                          i, n, i / n * 100, el, el / i * (n - i), len(rows), len(failed))
    out = pd.DataFrame(rows)
    LOG_UNIV.info("stage 1 quotes done: %d ok · %d failed · %d zero-price", len(rows), len(failed),
                  len(zero_px))
    if errors:
        LOG_UNIV.info("  quote errors by type: %s",
                      dict(sorted(errors.items(), key=lambda kv: -kv[1])[:5]))
    if not out.empty:
        has_mcap = int((out["mcap_cr"] > 0).sum())
        LOG_UNIV.info("  quotes carrying a market cap: %d/%d", has_mcap, len(out))
        if has_mcap < 0.5 * len(out):
            LOG_UNIV.warning("  most quotes have NO market cap → size will be a turnover proxy. "
                             "Supply ./mcap.csv (symbol,mcap_cr) for a real small-cap band.")
    return out


def build_universe(cfg: Config, hub: DataHub, broker: BrokerAdapter,
                   synth: SyntheticMarket, refresh: bool = True) -> pd.DataFrame:
    """Stage 1: cheap screen (quotes by default). Stage 2: deep history for survivors only."""
    t_start = time.time()
    LOG_UNIV.info("building universe | source=%s prefilter=%s", cfg.data_source, cfg.prefilter_mode)
    inst = candidate_instruments(cfg, broker, synth)
    excl = load_exclusions(cfg)
    n0 = len(inst)
    inst = inst[~inst["symbol"].str.upper().isin(excl)].copy()
    LOG_UNIV.info("structural filter → %d names (%d removed by exclusions)", len(inst), n0 - len(inst))
    if "mcap_cr" not in inst.columns:
        inst["mcap_cr"] = np.nan
    mcap_map = load_mcap_map()
    if mcap_map:
        inst["mcap_cr"] = inst["symbol"].str.upper().map(mcap_map).fillna(inst["mcap_cr"])
        LOG_UNIV.info("mcap.csv covers %d/%d candidates", int(inst["mcap_cr"].notna().sum()), len(inst))
    meta = inst.set_index("symbol")[["name"] + (["sector"] if "sector" in inst else [])].to_dict("index")

    use_quotes = cfg.prefilter_mode == "QUOTE" and cfg.data_source == "GROWW"

    # ---------- stage 1: cheap screen ----------
    if use_quotes:
        pre = prefilter_on_quotes(cfg, broker, inst["symbol"].tolist())
        if pre.empty:
            log.error("stage 1 returned nothing — check subscription, credentials and static IP")
            return pre
        known = inst.set_index("symbol")["mcap_cr"]
        pre["mcap_src"] = pre.apply(
            lambda r: "csv" if pd.notna(known.get(r["symbol"]))
            else ("quote" if r["mcap_cr"] > 0 else "PROXY"), axis=1)
        pre["mcap_cr"] = pre.apply(
            lambda r: known.get(r["symbol"]) if pd.notna(known.get(r["symbol"]))
            else (r["mcap_cr"] if r["mcap_cr"] > 0 else r["day_turnover_cr"] * 900.0), axis=1)
        LOG_UNIV.info("market-cap source: %s", pre["mcap_src"].value_counts().to_dict())
        if not cfg.allow_mcap_proxy:
            n_before = len(pre)
            pre = pre[pre["mcap_src"] != "PROXY"]
            LOG_UNIV.info("allow_mcap_proxy=False → dropped %d names without a real market cap",
                          n_before - len(pre))
        n1 = len(pre)
        g_px = pre["price"].between(cfg.price_min, cfg.price_max)
        g_mc = pre["mcap_cr"].between(cfg.mcap_min_cr, cfg.mcap_max_cr)
        g_to = pre["day_turnover_cr"].ge(cfg.min_median_turnover_cr * 0.5)
        LOG_UNIV.info("stage-1 gates on %d quoted names: price band passes %d · mcap band passes %d · "
                      "day turnover ≥ ₹%.1f cr passes %d · all three %d", n1, int(g_px.sum()),
                      int(g_mc.sum()), cfg.min_median_turnover_cr * 0.5, int(g_to.sum()),
                      int((g_px & g_mc & g_to).sum()))
        # Loose gate: one day of volume is noisy, so screen at half the threshold here and apply
        # the real 20-day-median test in stage 2 once actual bars exist.
        pre = pre[pre["price"].between(cfg.price_min, cfg.price_max)
                  & pre["mcap_cr"].between(cfg.mcap_min_cr, cfg.mcap_max_cr)
                  & pre["day_turnover_cr"].ge(cfg.min_median_turnover_cr * 0.5)]
        pre = pre.sort_values("day_turnover_cr", ascending=False).head(cfg.max_universe)
        if "mcap_src" in pre.columns:
            prox = pre[pre["mcap_src"] == "PROXY"]
            if len(prox):
                LOG_UNIV.warning("%d/%d stage-1 survivors have a PROXY market cap (turnover × 900) — "
                                 "large caps can slip into the 'small-cap' band this way, e.g. %s",
                                 len(prox), len(pre), prox["symbol"].head(8).tolist())
    else:
        short = hub.bulk(inst["symbol"].tolist(), lookback_days=cfg.prefilter_days,
                         refresh=refresh, workers=cfg.historical_workers, label="stage1")
        rows = []
        for sym, df in short.items():
            if df is None or len(df) < 40:
                continue
            px = float(df["close"].iloc[-1])
            med = float((df["close"] * df["volume"]).tail(20).median()) / 1e7
            mc = inst.set_index("symbol")["mcap_cr"].get(sym)
            rows.append({"symbol": sym, "price": px,
                         "mcap_cr": float(mc) if pd.notna(mc) else med * 900.0,
                         "day_turnover_cr": med, "day_volume": float(df["volume"].iloc[-1]),
                         "week_52_high": float(df["high"].max())})
        pre = pd.DataFrame(rows)
        if pre.empty:
            log.error("stage 1 returned nothing — check subscription, credentials and static IP")
            return pre
        pre = pre[pre["price"].between(cfg.price_min, cfg.price_max)
                  & pre["mcap_cr"].between(cfg.mcap_min_cr, cfg.mcap_max_cr)
                  & pre["day_turnover_cr"].ge(cfg.min_median_turnover_cr)]
        pre = pre.sort_values("day_turnover_cr", ascending=False).head(cfg.max_universe)

    pre = pre.reset_index(drop=True)
    LOG_UNIV.info("stage 1: %d names pass price/size/liquidity (capped at max_universe=%d) — "
                  "deep-fetching %d days of history", len(pre), cfg.max_universe, cfg.deep_history_days)

    # ---------- stage 2: deep history, survivors only ----------
    deep = hub.bulk(pre["symbol"].tolist(), lookback_days=cfg.deep_history_days,
                    refresh=refresh, workers=cfg.historical_workers, label="stage2")

    rows = []
    short_hist: List[str] = []
    thin: List[str] = []
    for _, r in pre.iterrows():
        df = deep.get(r["symbol"])
        if df is None or len(df) < cfg.min_history_bars:
            short_hist.append(f"{r['symbol']}({0 if df is None else len(df)})")
            continue
        med_turn_cr = float((df["close"] * df["volume"]).tail(20).median()) / 1e7   # strict gate
        if med_turn_cr < cfg.min_median_turnover_cr:
            thin.append(f"{r['symbol']}({med_turn_cr:.1f}cr)")
            continue
        m = meta.get(r["symbol"], {})
        rows.append({"symbol": r["symbol"], "name": m.get("name", r["symbol"]),
                     "sector": m.get("sector", "UNKNOWN"),
                     "price": float(df["close"].iloc[-1]), "mcap_cr": float(r["mcap_cr"]),
                     "median_turnover_cr": med_turn_cr,
                     "adv_shares": float(df["volume"].tail(20).median()), "bars": len(df)})

    u = pd.DataFrame(rows).sort_values("median_turnover_cr", ascending=False).reset_index(drop=True) \
        if rows else pd.DataFrame(columns=["symbol", "name", "sector", "price", "mcap_cr",
                                           "median_turnover_cr", "adv_shares", "bars"])
    STORE.set_kv("dead_symbols", sorted(DEAD_SYMBOLS))
    STORE.set_kv("history_floor", HISTORY_FLOOR)
    LOG_UNIV.info("stage 2 drops: %d with < %d bars (e.g. %s) · %d with 20d median turnover < ₹%.1f cr "
                  "(e.g. %s)", len(short_hist), cfg.min_history_bars, short_hist[:6], len(thin),
                  cfg.min_median_turnover_cr, thin[:6])
    LOG_UNIV.info("universe: %d tradeable small caps (≥%d bars, median turnover ≥ ₹%.1f cr) built in %.0fs",
                  len(u), cfg.min_history_bars, cfg.min_median_turnover_cr, time.time() - t_start)
    if len(u):
        LOG_UNIV.info("universe stats | price ₹%.0f–%.0f · mcap ₹%.0f–%.0f cr · median turnover "
                      "₹%.1f–%.1f cr · bars %d–%d", u["price"].min(), u["price"].max(),
                      u["mcap_cr"].min(), u["mcap_cr"].max(), u["median_turnover_cr"].min(),
                      u["median_turnover_cr"].max(), u["bars"].min(), u["bars"].max())
        if "sector" in u and u["sector"].eq("UNKNOWN").all():
            LOG_UNIV.warning("every sector is UNKNOWN → max_sector_positions=%d caps the WHOLE book "
                             "at %d positions. Add a sector column (e.g. via mcap.csv) to fix.",
                             cfg.max_sector_positions, cfg.max_sector_positions)
    else:
        LOG_UNIV.error("universe is EMPTY — nothing downstream can work")
    return u

07:49:11 | INFO    | engine.cell       | ▶ START  [07] Universe construction
07:49:11 | INFO    | engine.cell       | ✔ DONE   [07] Universe construction in 0.0s


### Preflight

Run this before a live universe build. It costs one instrument-master call and one full history
fetch, and it catches the three things that waste an hour of API quota: a bad filter letting
bonds through, credentials/subscription problems, and a chunking bug.

In [103]:
# [07b] Preflight & probes
def preflight(cfg: Config, broker: BrokerAdapter, hub: DataHub, check_benchmark: bool = True) -> Dict[str, Any]:
    """Cheap checks before an expensive universe build. PATCH: now also resolves and fetches the
    benchmark, so a broken index symbol fails *here* in seconds, not after a 30-minute build."""
    report: Dict[str, Any] = {}
    print(f"mode={cfg.mode}  source={cfg.data_source}  "
          f"synthetic_fallback={cfg.allow_synthetic_fallback}  dry_run={cfg.dry_run}")
    with log_step("preflight: instrument filter", LOG_UNIV):
        inst = candidate_instruments(cfg, broker, SYNTH)
    print(f"equity instruments after filter : {len(inst)}")
    print(f"sample                          : {inst['symbol'].head(8).tolist()}")
    digit_led = [x for x in inst["symbol"] if x[:1].isdigit()]
    print(f"symbols starting with a digit   : {len(digit_led)}  (must be 0)")
    print(f"dead-symbol memo                : {len(DEAD_SYMBOLS)} names")
    report.update(instruments=len(inst), digit_led=len(digit_led))

    test = inst["symbol"].iloc[0] if len(inst) else "RELIANCE"
    t0 = time.time()
    df = hub._fetch(test, date.today() - timedelta(days=cfg.deep_history_days), date.today())
    span = f"{df.index.min().date()} -> {df.index.max().date()}" if len(df) else "NO DATA"
    print(f"{test:<12} {len(df):>5} bars in {time.time() - t0:>5.1f}s   {span}")
    report["test_symbol_bars"] = len(df)
    if len(df) == 0:
        print("  ^ zero bars: check the subscription, then read the last FETCH_FAIL event:")
        print(STORE.query("SELECT symbol,message FROM events WHERE kind='FETCH_FAIL' "
                          "ORDER BY id DESC LIMIT 1").to_string(index=False))

    if check_benchmark:
        b = hub.benchmark(cfg.history_days)
        ok = len(b) >= cfg.min_benchmark_bars
        print(f"benchmark {cfg.benchmark:<18}: {len(b):>5} bars from {hub.bench_source}"
              f"   {'OK' if ok else 'PROBLEM — see the engine.benchmark log lines above'}")
        report.update(benchmark_bars=len(b), benchmark_source=hub.bench_source)
        if not ok and cfg.benchmark_allow_universe_proxy:
            print("  (the run will continue with an equal-weight universe proxy as the benchmark)")

    if cfg.data_source == "GROWW":
        chunks = math.ceil(cfg.deep_history_days / cfg.groww_max_days_per_request)
        hist = min(cfg.max_universe, len(inst)) * chunks
        cheap = len(inst) if cfg.prefilter_mode == "QUOTE" else 0
        print(f"budget: {cheap:,} Live-Data quotes + {hist:,} historical requests "
              f"({chunks} chunks x {min(cfg.max_universe, len(inst))} names)")
        print(f"        ~{cheap / 250:.0f} min + ~{hist / 50:.0f} min at the configured limits")
    LOG_UNIV.info("preflight report: %s", report)
    return report


def probe_historical(broker: BrokerAdapter, symbol: str = "RELIANCE",
                     attempts: int = 6, gap: float = 1.0) -> None:
    """Measure your actual historical ceiling instead of guessing at it.

    Fires `attempts` single-window requests `gap` seconds apart, bypassing the limiter, and reports
    which ones succeed. If everything fails even at gap=5, you are not being throttled per-second —
    you have exhausted a cumulative quota and need to wait for the window to reset.
    """
    if not isinstance(broker, GrowwBroker):
        print("probe_historical only applies to the live Groww broker")
        return
    end = date.today()
    ok = 0
    for i in range(1, attempts + 1):
        t0 = time.time()
        try:
            df = broker._candle_window.__wrapped__(broker, symbol, end - timedelta(days=30), end) \
                if hasattr(broker._candle_window, "__wrapped__") else \
                broker._candle_window(symbol, end - timedelta(days=30), end)
            ok += 1
            print(f"  {i}/{attempts}  OK      {len(df):>4} bars   {time.time() - t0:.1f}s")
        except Exception as e:  # noqa: BLE001
            print(f"  {i}/{attempts}  FAILED  {str(e)[:80]}")
        time.sleep(gap)
    print(f"{ok}/{attempts} succeeded at {gap:.1f}s spacing "
          f"({'raise' if ok == attempts else 'lower'} the historical limit accordingly)")

07:49:11 | INFO    | engine.cell       | ▶ START  [07b] Preflight & probes
07:49:11 | INFO    | engine.cell       | ✔ DONE   [07b] Preflight & probes in 0.0s


## 8 · Feature engine

Everything is vectorised pandas — no `for` loop touches a bar. Wilder smoothing (`ewm(alpha=1/n)`)
is used for RSI/ATR/ADX so the numbers match what your charting platform shows.

The features are grouped by what they are actually *for*:

| Group | Columns | Question it answers |
|---|---|---|
| Trend | `ema20/50/200`, `ema50_slope`, `above_*` | Is this thing going up? |
| Momentum | `ret_21/63/126`, `rsi14`, `adx` | How forcefully? |
| Volatility | `atr`, `atr_pct`, `bbw`, `contraction`, `nr7` | Is it coiling or already extended? |
| Volume | `vol_ratio`, `vol_z`, `dry_up`, `pocket_pivot` | Is institutional money showing up? |
| Structure | `hh20`, `dist_52w_high`, `base_len`, `base_depth` | Is there a real base to break out of? |
| Liquidity | `turnover_cr`, `adv20` | Can I actually get in and out? |
| Relative | `rs_line`, `rs_slope`, `rel_63` | Is it beating the small-cap index? |

In [104]:
# [08] Feature engine
def _wilder(s: pd.Series, n: int) -> pd.Series:
    return s.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()


def true_range(h: pd.Series, l: pd.Series, c: pd.Series) -> pd.Series:
    pc = c.shift(1)
    return pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)


def rsi(close: pd.Series, n: int = 14) -> pd.Series:
    d = close.diff()
    up = _wilder(d.clip(lower=0), n)
    dn = _wilder((-d).clip(lower=0), n)
    rs = up / dn.replace(0, np.nan)
    return (100 - 100 / (1 + rs)).fillna(50)


def adx(h: pd.Series, l: pd.Series, c: pd.Series, n: int = 14) -> Tuple[pd.Series, pd.Series, pd.Series]:
    up, dn = h.diff(), -l.diff()
    plus = pd.Series(np.where((up > dn) & (up > 0), up, 0.0), index=h.index)
    minus = pd.Series(np.where((dn > up) & (dn > 0), dn, 0.0), index=h.index)
    atr_ = _wilder(true_range(h, l, c), n).replace(0, np.nan)
    pdi = 100 * _wilder(plus, n) / atr_
    mdi = 100 * _wilder(minus, n) / atr_
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    return _wilder(dx, n), pdi, mdi


def compute_features(df: pd.DataFrame, bench: Optional[pd.Series] = None) -> pd.DataFrame:
    """One symbol in, ~45 feature columns out. Index must be a DatetimeIndex of daily bars."""
    if df is None or len(df) == 0:
        raise ValueError("compute_features() got an empty frame — nothing to compute")
    missing = [c for c in ("open", "high", "low", "close", "volume") if c not in df.columns]
    if missing:
        raise ValueError(f"compute_features() input is missing columns {missing}")
    f = df[["open", "high", "low", "close", "volume"]].copy().astype(float)
    o, h, l, c, v = f["open"], f["high"], f["low"], f["close"], f["volume"]

    # --- trend ---
    for n in (10, 20, 50, 200):
        f[f"ema{n}"] = c.ewm(span=n, adjust=False).mean()
    f["sma20"] = c.rolling(20).mean()
    f["ema50_slope"] = f["ema50"].pct_change(10) * 100
    f["ema200_slope"] = f["ema200"].pct_change(20) * 100
    f["above_20"] = c > f["ema20"]
    f["above_50"] = c > f["ema50"]
    f["above_200"] = c > f["ema200"]
    f["stack"] = (f["ema20"] > f["ema50"]) & (f["ema50"] > f["ema200"])

    # --- momentum ---
    for n in (5, 21, 63, 126, 252):
        f[f"ret_{n}"] = c.pct_change(n) * 100
    f["rsi14"] = rsi(c, 14)
    f["rsi2"] = rsi(c, 2)
    f["adx"], f["pdi"], f["mdi"] = adx(h, l, c, 14)

    # --- volatility ---
    f["atr"] = _wilder(true_range(h, l, c), 14)
    f["atr_pct"] = f["atr"] / c * 100
    f["atr_pct_50"] = f["atr_pct"].rolling(50).mean()
    f["contraction"] = f["atr_pct"].rolling(10).mean() / f["atr_pct_50"]
    sd = c.rolling(20).std()
    f["bbw"] = (4 * sd) / f["sma20"] * 100
    f["bbw_pct"] = f["bbw"].rolling(120).rank(pct=True) * 100
    rng = h - l
    f["nr7"] = rng == rng.rolling(7).min()
    f["inside"] = (h < h.shift(1)) & (l > l.shift(1))

    # --- volume ---
    f["vol_sma50"] = v.rolling(50).mean()
    f["vol_ratio"] = v / f["vol_sma50"]
    f["vol_z"] = (v - f["vol_sma50"]) / v.rolling(50).std()
    f["dry_up"] = v.rolling(5).mean() / f["vol_sma50"]        # <0.8 during a good pullback
    up_vol = v.where(c > c.shift(1), 0.0)
    dn_vol = v.where(c < c.shift(1), np.nan)
    f["pocket_pivot"] = up_vol > dn_vol.rolling(10).max()
    f["turnover_cr"] = c * v / 1e7
    f["adv20"] = v.rolling(20).median()
    f["turnover_med20_cr"] = f["turnover_cr"].rolling(20).median()

    # --- structure ---
    f["hh20"] = h.rolling(20).max().shift(1)
    f["hh55"] = h.rolling(55).max().shift(1)
    f["ll10"] = l.rolling(10).min().shift(1)
    f["ll20"] = l.rolling(20).min().shift(1)
    f["high_252"] = h.rolling(252, min_periods=60).max()
    f["low_252"] = l.rolling(252, min_periods=60).min()
    f["dist_52w_high"] = (c / f["high_252"] - 1) * 100        # 0 = at highs, negative = below
    f["off_52w_low"] = (c / f["low_252"] - 1) * 100
    f["base_len"] = (~(h >= f["high_252"].shift(1))).astype(int).groupby(
        (h >= f["high_252"].shift(1)).cumsum()).cumsum()
    f["base_depth"] = (h.rolling(40).max() - l.rolling(40).min()) / h.rolling(40).max() * 100
    f["ext_atr"] = (c - f["ema20"]) / f["atr"]                # extension from the mean, in ATRs
    f["gap_pct"] = (o / c.shift(1) - 1) * 100

    # --- relative strength ---
    if bench is not None and len(bench) > 10:
        b = bench.reindex(f.index).ffill()
        f["rs_line"] = c / b
        f["rs_slope"] = f["rs_line"].pct_change(21) * 100
        f["rs_new_high"] = f["rs_line"] >= f["rs_line"].rolling(63).max()
        f["rel_63"] = f["ret_63"] - (b.pct_change(63) * 100)
        f["bench_ret_21"] = b.pct_change(21) * 100
    else:
        for col in ("rs_line", "rs_slope", "rel_63", "bench_ret_21"):
            f[col] = np.nan
        f["rs_new_high"] = False

    f["quality"] = (f["ret_63"].rolling(63).mean() /
                    (f["ret_5"].rolling(63).std().replace(0, np.nan)))

    # Daily-price sanity flags used by research/live risk controls.
    prev = f["close"].shift(1)
    ret1 = (f["close"] / prev - 1) * 100
    flat_bar = ((f["open"] - f["high"]).abs() < 1e-9) & ((f["high"] - f["low"]).abs() < 1e-9) & ((f["low"] - f["close"]).abs() < 1e-9)
    near_band = pd.concat([(ret1.abs()-x).abs() for x in (2, 5, 10, 20)], axis=1).min(axis=1) < 0.35
    f["circuit_like"] = flat_bar & near_band
    f["lower_circuit_locked"] = f["circuit_like"] & (ret1 < 0)
    f["corp_action_suspect"] = (ret1.abs() > 35) & (f["volume"] > 0)
    return f
FEATURE_SOURCE_FOR_TEST = '# [08] Feature engine\ndef _wilder(s: pd.Series, n: int) -> pd.Series:\n    return s.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()\n\n\ndef true_range(h: pd.Series, l: pd.Series, c: pd.Series) -> pd.Series:\n    pc = c.shift(1)\n    return pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)\n\n\ndef rsi(close: pd.Series, n: int = 14) -> pd.Series:\n    d = close.diff()\n    up = _wilder(d.clip(lower=0), n)\n    dn = _wilder((-d).clip(lower=0), n)\n    rs = up / dn.replace(0, np.nan)\n    return (100 - 100 / (1 + rs)).fillna(50)\n\n\ndef adx(h: pd.Series, l: pd.Series, c: pd.Series, n: int = 14) -> Tuple[pd.Series, pd.Series, pd.Series]:\n    up, dn = h.diff(), -l.diff()\n    plus = pd.Series(np.where((up > dn) & (up > 0), up, 0.0), index=h.index)\n    minus = pd.Series(np.where((dn > up) & (dn > 0), dn, 0.0), index=h.index)\n    atr_ = _wilder(true_range(h, l, c), n).replace(0, np.nan)\n    pdi = 100 * _wilder(plus, n) / atr_\n    mdi = 100 * _wilder(minus, n) / atr_\n    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)\n    return _wilder(dx, n), pdi, mdi\n\n\ndef compute_features(df: pd.DataFrame, bench: Optional[pd.Series] = None) -> pd.DataFrame:\n    """One symbol in, ~45 feature columns out. Index must be a DatetimeIndex of daily bars."""\n    if df is None or len(df) == 0:\n        raise ValueError("compute_features() got an empty frame — nothing to compute")\n    missing = [c for c in ("open", "high", "low", "close", "volume") if c not in df.columns]\n    if missing:\n        raise ValueError(f"compute_features() input is missing columns {missing}")\n    f = df[["open", "high", "low", "close", "volume"]].copy().astype(float)\n    o, h, l, c, v = f["open"], f["high"], f["low"], f["close"], f["volume"]\n\n    # --- trend ---\n    for n in (10, 20, 50, 200):\n        f[f"ema{n}"] = c.ewm(span=n, adjust=False).mean()\n    f["sma20"] = c.rolling(20).mean()\n    f["ema50_slope"] = f["ema50"].pct_change(10) * 100\n    f["ema200_slope"] = f["ema200"].pct_change(20) * 100\n    f["above_20"] = c > f["ema20"]\n    f["above_50"] = c > f["ema50"]\n    f["above_200"] = c > f["ema200"]\n    f["stack"] = (f["ema20"] > f["ema50"]) & (f["ema50"] > f["ema200"])\n\n    # --- momentum ---\n    for n in (5, 21, 63, 126, 252):\n        f[f"ret_{n}"] = c.pct_change(n) * 100\n    f["rsi14"] = rsi(c, 14)\n    f["rsi2"] = rsi(c, 2)\n    f["adx"], f["pdi"], f["mdi"] = adx(h, l, c, 14)\n\n    # --- volatility ---\n    f["atr"] = _wilder(true_range(h, l, c), 14)\n    f["atr_pct"] = f["atr"] / c * 100\n    f["atr_pct_50"] = f["atr_pct"].rolling(50).mean()\n    f["contraction"] = f["atr_pct"].rolling(10).mean() / f["atr_pct_50"]\n    sd = c.rolling(20).std()\n    f["bbw"] = (4 * sd) / f["sma20"] * 100\n    f["bbw_pct"] = f["bbw"].rolling(120).rank(pct=True) * 100\n    rng = h - l\n    f["nr7"] = rng == rng.rolling(7).min()\n    f["inside"] = (h < h.shift(1)) & (l > l.shift(1))\n\n    # --- volume ---\n    f["vol_sma50"] = v.rolling(50).mean()\n    f["vol_ratio"] = v / f["vol_sma50"]\n    f["vol_z"] = (v - f["vol_sma50"]) / v.rolling(50).std()\n    f["dry_up"] = v.rolling(5).mean() / f["vol_sma50"]        # <0.8 during a good pullback\n    up_vol = v.where(c > c.shift(1), 0.0)\n    dn_vol = v.where(c < c.shift(1), np.nan)\n    f["pocket_pivot"] = up_vol > dn_vol.rolling(10).max()\n    f["turnover_cr"] = c * v / 1e7\n    f["adv20"] = v.rolling(20).median()\n    f["turnover_med20_cr"] = f["turnover_cr"].rolling(20).median()\n\n    # --- structure ---\n    f["hh20"] = h.rolling(20).max().shift(1)\n    f["hh55"] = h.rolling(55).max().shift(1)\n    f["ll10"] = l.rolling(10).min().shift(1)\n    f["ll20"] = l.rolling(20).min().shift(1)\n    f["high_252"] = h.rolling(252, min_periods=60).max()\n    f["low_252"] = l.rolling(252, min_periods=60).min()\n    f["dist_52w_high"] = (c / f["high_252"] - 1) * 100        # 0 = at highs, negative = below\n    f["off_52w_low"] = (c / f["low_252"] - 1) * 100\n    f["base_len"] = (~(h >= f["high_252"].shift(1))).astype(int).groupby(\n        (h >= f["high_252"].shift(1)).cumsum()).cumsum()\n    f["base_depth"] = (h.rolling(40).max() - l.rolling(40).min()) / h.rolling(40).max() * 100\n    f["ext_atr"] = (c - f["ema20"]) / f["atr"]                # extension from the mean, in ATRs\n    f["gap_pct"] = (o / c.shift(1) - 1) * 100\n\n    # --- relative strength ---\n    if bench is not None and len(bench) > 10:\n        b = bench.reindex(f.index).ffill()\n        f["rs_line"] = c / b\n        f["rs_slope"] = f["rs_line"].pct_change(21) * 100\n        f["rs_new_high"] = f["rs_line"] >= f["rs_line"].rolling(63).max()\n        f["rel_63"] = f["ret_63"] - (b.pct_change(63) * 100)\n        f["bench_ret_21"] = b.pct_change(21) * 100\n    else:\n        for col in ("rs_line", "rs_slope", "rel_63", "bench_ret_21"):\n            f[col] = np.nan\n        f["rs_new_high"] = False\n\n    f["quality"] = (f["ret_63"].rolling(63).mean() /\n                    (f["ret_5"].rolling(63).std().replace(0, np.nan)))\n\n    # Daily-price sanity flags used by research/live risk controls.\n    prev = f["close"].shift(1)\n    ret1 = (f["close"] / prev - 1) * 100\n    flat_bar = ((f["open"] - f["high"]).abs() < 1e-9) & ((f["high"] - f["low"]).abs() < 1e-9) & ((f["low"] - f["close"]).abs() < 1e-9)\n    near_band = pd.concat([(ret1.abs()-x).abs() for x in (2, 5, 10, 20)], axis=1).min(axis=1) < 0.35\n    f["circuit_like"] = flat_bar & near_band\n    f["lower_circuit_locked"] = f["circuit_like"] & (ret1 < 0)\n    f["corp_action_suspect"] = (ret1.abs() > 35) & (f["volume"] > 0)\n    return f'


07:49:11 | INFO    | engine.cell       | ▶ START  [08] Feature engine
07:49:11 | INFO    | engine.cell       | ✔ DONE   [08] Feature engine in 0.0s


## 9 · Strategy bank

Four uncorrelated small-cap swing edges. Each returns `(signal, entry_trigger, init_stop)` per
bar, computed **only from data available at that bar's close**.

| Key | Edge | Typical hold | Entry mechanic |
|---|---|---|---|
| `VCP_BREAKOUT` | Volatility contraction resolving upward on volume expansion | 8–20 d | Buy-stop above the pivot high |
| `TREND_PULLBACK` | Continuation after an orderly dip to the 20-EMA in a stacked uptrend | 5–15 d | Buy-stop above the reversal bar's high |
| `MEAN_REVERT` | Short-term washout inside a long-term uptrend (RSI-2 style) | 2–6 d | Limit at/just below the close |
| `DONCHIAN_55` | Classic channel breakout with an ADX trend filter | 15–40 d | Buy-stop at the 55-day high |

The mean-reversion sleeve exists mainly to smooth the equity curve — momentum breakouts fail
in clusters, and a small-cap book made of nothing but breakouts is a single-factor bet.

In [105]:
# [09] Strategy bank
def _pack(sig: pd.Series, entry: pd.Series, stop: pd.Series, name: str) -> pd.DataFrame:
    out = pd.DataFrame({"signal": sig.fillna(False).astype(bool),
                        "entry": entry, "stop": stop})
    out["strategy"] = name
    bad = (~np.isfinite(out["entry"])) | (~np.isfinite(out["stop"])) | (out["stop"] >= out["entry"])
    out.loc[bad, "signal"] = False
    return out


def strat_vcp_breakout(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    c, h, l = f["close"], f["high"], f["low"]
    sig = (
        f["stack"]
        & (f["ema50_slope"] > 0)
        & (c > f["hh20"])                              # breaking the 20-day pivot
        & (c.shift(1) <= f["hh20"])                    # first close through it
        & (f["vol_ratio"] > 1.5)
        & (f["contraction"] < 0.95)                    # coiled before the move
        & (f["dist_52w_high"] > -25)
        & (f["adx"] > cfg.min_adx)
        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)
        & (f["ext_atr"] < cfg.max_extension_atr)
        & (f["base_depth"] < 45)
    )
    entry = np.maximum(h, f["hh20"]) * 1.002           # buy-stop just above the breakout bar
    stop = np.minimum(c - cfg.atr_stop_mult * f["atr"], f["ll10"] * 0.995)
    return _pack(sig, entry, stop, "VCP_BREAKOUT")


def strat_trend_pullback(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    c, h, l = f["close"], f["high"], f["low"]
    touched = (l <= f["ema20"] * 1.01).rolling(4).max().astype(bool)
    reversal = (c > c.shift(1)) & (c > (h + l) / 2) & (l > l.shift(1) * 0.995)
    sig = (
        f["stack"] & (f["ema50_slope"] > 0)
        & touched & reversal
        & (c > f["ema50"])
        & f["rsi14"].shift(1).between(35, 58)          # a dip, not a collapse
        & (f["dry_up"] < 1.15)                          # volume dried up into the low
        & (f["adx"] > cfg.min_adx - 3)
        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)
        & (f["dist_52w_high"] > -30)
    )
    entry = h * 1.002
    stop = np.minimum(l * 0.99, c - cfg.atr_stop_mult * 0.9 * f["atr"])
    return _pack(sig, entry, stop, "TREND_PULLBACK")


def strat_mean_revert(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    c, l = f["close"], f["low"]
    sig = (
        f["above_200"] & (f["ema200_slope"] > 0)
        & (f["rsi2"] < 8)
        & (c < f["ema20"]) & (c > f["ema200"] * 1.02)
        & (f["ret_5"] < -4)
        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)
        & (f["turnover_med20_cr"] > cfg.min_median_turnover_cr)
    )
    entry = c * 1.003                                   # next-day open-ish, small buffer
    stop = np.minimum(l, c - 2.0 * f["atr"]) * 0.995
    return _pack(sig, entry, stop, "MEAN_REVERT")


def strat_donchian(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    c, h = f["close"], f["high"]
    sig = (
        (c > f["hh55"]) & (c.shift(1) <= f["hh55"])
        & (f["adx"] > 25) & (f["pdi"] > f["mdi"])
        & f["above_200"] & (f["ema200_slope"] > 0)
        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)
        & (f["ext_atr"] < cfg.max_extension_atr + 1)
    )
    entry = np.maximum(h, f["hh55"]) * 1.002
    stop = c - cfg.atr_stop_mult * 1.15 * f["atr"]
    return _pack(sig, entry, stop, "DONCHIAN_55")



def strat_52w_momentum(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Price leadership near 52-week highs; disabled until it earns its place OOS."""
    c, h, l = f["close"], f["high"], f["low"]
    sig = (
        f["above_200"] & (f["ema200_slope"] > 0)
        & (f["dist_52w_high"] > -4.0)
        & (f["ret_126"] > 12)
        & (f["vol_ratio"] > 1.15)
        & (f["adx"] > 20)
        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)
    )
    entry = h * 1.002
    stop = np.maximum(c - 2.2 * f["atr"], f["ema20"] - 0.5 * f["atr"])
    return _pack(sig, entry, stop, "MOMENTUM_52W")


def strat_episodic_pivot(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Large gap/volume thrust from a constructive trend; event calendar is intentionally not assumed."""
    c, h, l, o = f["close"], f["high"], f["low"], f["open"]
    gap = (o / f["close"].shift(1) - 1) * 100
    sig = (
        f["above_200"] & (f["ema200_slope"] > 0)
        & gap.between(3.0, 12.0)
        & (f["vol_ratio"] > 2.5)
        & (c > o)
        & (c > f["hh20"])
        & (f["ext_atr"] < cfg.max_extension_atr + 1.0)
        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)
    )
    entry = h * 1.002
    stop = np.maximum(l * 0.995, c - 2.4 * f["atr"])
    return _pack(sig, entry, stop, "EPISODIC_PIVOT")


STRATEGIES: Dict[str, Callable[[pd.DataFrame, Config], pd.DataFrame]] = {
    "VCP_BREAKOUT": strat_vcp_breakout,
    "TREND_PULLBACK": strat_trend_pullback,
    "DONCHIAN_55": strat_donchian,
}
if CFG.enable_mean_revert:
    STRATEGIES["MEAN_REVERT"] = strat_mean_revert
if CFG.enable_52w_momentum:
    STRATEGIES["MOMENTUM_52W"] = strat_52w_momentum
if CFG.enable_episodic_pivot:
    STRATEGIES["EPISODIC_PIVOT"] = strat_episodic_pivot

STRATEGY_PRIORITY = [
    x for x in ("EPISODIC_PIVOT", "VCP_BREAKOUT", "MOMENTUM_52W",
                "DONCHIAN_55", "TREND_PULLBACK", "MEAN_REVERT") if x in STRATEGIES
]
STRATEGY_WEIGHT = {
    "EPISODIC_PIVOT": 1.03, "VCP_BREAKOUT": 1.00, "MOMENTUM_52W": 0.98,
    "DONCHIAN_55": 0.95, "TREND_PULLBACK": 0.92, "MEAN_REVERT": 0.80,
}


def run_strategies(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Collapse the strategy bank into one signal frame (highest-priority strategy wins a bar)."""
    frames = {k: fn(f, cfg) for k, fn in STRATEGIES.items()}
    out = pd.DataFrame(index=f.index, columns=["signal", "entry", "stop", "strategy"])
    out["signal"] = False
    for name in reversed(STRATEGY_PRIORITY):            # later assignment wins -> reverse order
        fr = frames[name]
        m = fr["signal"].to_numpy()
        out.loc[m, ["signal", "entry", "stop", "strategy"]] = fr.loc[m, ["signal", "entry", "stop", "strategy"]]
    out["signal"] = out["signal"].fillna(False).astype(bool)
    out["n_confirm"] = sum(fr["signal"].astype(int) for fr in frames.values())
    return out


def strategy_hit_counts(panel: Dict[str, pd.DataFrame], cfg: Config, last_n: int = 1) -> Dict[str, int]:
    """How many names each strategy fires on over the last `last_n` bars — a quick sanity check
    that the bank is alive (all zeros for weeks usually means a broken feature, e.g. NaN RS)."""
    counts: Dict[str, int] = defaultdict(int)
    for f in panel.values():
        tail = f.tail(max(last_n + 260, 300))          # enough warm-up for rolling windows
        for name, fn in STRATEGIES.items():
            try:
                counts[name] += int(fn(tail, cfg)["signal"].tail(last_n).any())
            except Exception as e:  # noqa: BLE001
                LOG_STRAT.debug("%s failed on a frame: %s", name, e)
    return dict(counts)


LOG_STRAT.info("strategy bank: %s · priority %s", list(STRATEGIES), STRATEGY_PRIORITY)
STRATEGY_SOURCE_FOR_TEST = '# [09] Strategy bank\ndef _pack(sig: pd.Series, entry: pd.Series, stop: pd.Series, name: str) -> pd.DataFrame:\n    out = pd.DataFrame({"signal": sig.fillna(False).astype(bool),\n                        "entry": entry, "stop": stop})\n    out["strategy"] = name\n    bad = (~np.isfinite(out["entry"])) | (~np.isfinite(out["stop"])) | (out["stop"] >= out["entry"])\n    out.loc[bad, "signal"] = False\n    return out\n\n\ndef strat_vcp_breakout(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:\n    c, h, l = f["close"], f["high"], f["low"]\n    sig = (\n        f["stack"]\n        & (f["ema50_slope"] > 0)\n        & (c > f["hh20"])                              # breaking the 20-day pivot\n        & (c.shift(1) <= f["hh20"])                    # first close through it\n        & (f["vol_ratio"] > 1.5)\n        & (f["contraction"] < 0.95)                    # coiled before the move\n        & (f["dist_52w_high"] > -25)\n        & (f["adx"] > cfg.min_adx)\n        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)\n        & (f["ext_atr"] < cfg.max_extension_atr)\n        & (f["base_depth"] < 45)\n    )\n    entry = np.maximum(h, f["hh20"]) * 1.002           # buy-stop just above the breakout bar\n    stop = np.minimum(c - cfg.atr_stop_mult * f["atr"], f["ll10"] * 0.995)\n    return _pack(sig, entry, stop, "VCP_BREAKOUT")\n\n\ndef strat_trend_pullback(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:\n    c, h, l = f["close"], f["high"], f["low"]\n    touched = (l <= f["ema20"] * 1.01).rolling(4).max().astype(bool)\n    reversal = (c > c.shift(1)) & (c > (h + l) / 2) & (l > l.shift(1) * 0.995)\n    sig = (\n        f["stack"] & (f["ema50_slope"] > 0)\n        & touched & reversal\n        & (c > f["ema50"])\n        & f["rsi14"].shift(1).between(35, 58)          # a dip, not a collapse\n        & (f["dry_up"] < 1.15)                          # volume dried up into the low\n        & (f["adx"] > cfg.min_adx - 3)\n        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)\n        & (f["dist_52w_high"] > -30)\n    )\n    entry = h * 1.002\n    stop = np.minimum(l * 0.99, c - cfg.atr_stop_mult * 0.9 * f["atr"])\n    return _pack(sig, entry, stop, "TREND_PULLBACK")\n\n\ndef strat_mean_revert(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:\n    c, l = f["close"], f["low"]\n    sig = (\n        f["above_200"] & (f["ema200_slope"] > 0)\n        & (f["rsi2"] < 8)\n        & (c < f["ema20"]) & (c > f["ema200"] * 1.02)\n        & (f["ret_5"] < -4)\n        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)\n        & (f["turnover_med20_cr"] > cfg.min_median_turnover_cr)\n    )\n    entry = c * 1.003                                   # next-day open-ish, small buffer\n    stop = np.minimum(l, c - 2.0 * f["atr"]) * 0.995\n    return _pack(sig, entry, stop, "MEAN_REVERT")\n\n\ndef strat_donchian(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:\n    c, h = f["close"], f["high"]\n    sig = (\n        (c > f["hh55"]) & (c.shift(1) <= f["hh55"])\n        & (f["adx"] > 25) & (f["pdi"] > f["mdi"])\n        & f["above_200"] & (f["ema200_slope"] > 0)\n        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)\n        & (f["ext_atr"] < cfg.max_extension_atr + 1)\n    )\n    entry = np.maximum(h, f["hh55"]) * 1.002\n    stop = c - cfg.atr_stop_mult * 1.15 * f["atr"]\n    return _pack(sig, entry, stop, "DONCHIAN_55")\n\n\n\ndef strat_52w_momentum(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:\n    """Price leadership near 52-week highs; disabled until it earns its place OOS."""\n    c, h, l = f["close"], f["high"], f["low"]\n    sig = (\n        f["above_200"] & (f["ema200_slope"] > 0)\n        & (f["dist_52w_high"] > -4.0)\n        & (f["ret_126"] > 12)\n        & (f["vol_ratio"] > 1.15)\n        & (f["adx"] > 20)\n        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)\n    )\n    entry = h * 1.002\n    stop = np.maximum(c - 2.2 * f["atr"], f["ema20"] - 0.5 * f["atr"])\n    return _pack(sig, entry, stop, "MOMENTUM_52W")\n\n\ndef strat_episodic_pivot(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:\n    """Large gap/volume thrust from a constructive trend; event calendar is intentionally not assumed."""\n    c, h, l, o = f["close"], f["high"], f["low"], f["open"]\n    gap = (o / f["close"].shift(1) - 1) * 100\n    sig = (\n        f["above_200"] & (f["ema200_slope"] > 0)\n        & gap.between(3.0, 12.0)\n        & (f["vol_ratio"] > 2.5)\n        & (c > o)\n        & (c > f["hh20"])\n        & (f["ext_atr"] < cfg.max_extension_atr + 1.0)\n        & f["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct)\n    )\n    entry = h * 1.002\n    stop = np.maximum(l * 0.995, c - 2.4 * f["atr"])\n    return _pack(sig, entry, stop, "EPISODIC_PIVOT")\n\n\nSTRATEGIES: Dict[str, Callable[[pd.DataFrame, Config], pd.DataFrame]] = {\n    "VCP_BREAKOUT": strat_vcp_breakout,\n    "TREND_PULLBACK": strat_trend_pullback,\n    "DONCHIAN_55": strat_donchian,\n}\nif CFG.enable_mean_revert:\n    STRATEGIES["MEAN_REVERT"] = strat_mean_revert\nif CFG.enable_52w_momentum:\n    STRATEGIES["MOMENTUM_52W"] = strat_52w_momentum\nif CFG.enable_episodic_pivot:\n    STRATEGIES["EPISODIC_PIVOT"] = strat_episodic_pivot\n\nSTRATEGY_PRIORITY = [\n    x for x in ("EPISODIC_PIVOT", "VCP_BREAKOUT", "MOMENTUM_52W",\n                "DONCHIAN_55", "TREND_PULLBACK", "MEAN_REVERT") if x in STRATEGIES\n]\nSTRATEGY_WEIGHT = {\n    "EPISODIC_PIVOT": 1.03, "VCP_BREAKOUT": 1.00, "MOMENTUM_52W": 0.98,\n    "DONCHIAN_55": 0.95, "TREND_PULLBACK": 0.92, "MEAN_REVERT": 0.80,\n}\n\n\ndef run_strategies(f: pd.DataFrame, cfg: Config) -> pd.DataFrame:\n    """Collapse the strategy bank into one signal frame (highest-priority strategy wins a bar)."""\n    frames = {k: fn(f, cfg) for k, fn in STRATEGIES.items()}\n    out = pd.DataFrame(index=f.index, columns=["signal", "entry", "stop", "strategy"])\n    out["signal"] = False\n    for name in reversed(STRATEGY_PRIORITY):            # later assignment wins -> reverse order\n        fr = frames[name]\n        m = fr["signal"].to_numpy()\n        out.loc[m, ["signal", "entry", "stop", "strategy"]] = fr.loc[m, ["signal", "entry", "stop", "strategy"]]\n    out["signal"] = out["signal"].fillna(False).astype(bool)\n    out["n_confirm"] = sum(fr["signal"].astype(int) for fr in frames.values())\n    return out\n\n\ndef strategy_hit_counts(panel: Dict[str, pd.DataFrame], cfg: Config, last_n: int = 1) -> Dict[str, int]:\n    """How many names each strategy fires on over the last `last_n` bars — a quick sanity check\n    that the bank is alive (all zeros for weeks usually means a broken feature, e.g. NaN RS)."""\n    counts: Dict[str, int] = defaultdict(int)\n    for f in panel.values():\n        tail = f.tail(max(last_n + 260, 300))          # enough warm-up for rolling windows\n        for name, fn in STRATEGIES.items():\n            try:\n                counts[name] += int(fn(tail, cfg)["signal"].tail(last_n).any())\n            except Exception as e:  # noqa: BLE001\n                LOG_STRAT.debug("%s failed on a frame: %s", name, e)\n    return dict(counts)\n\n\nLOG_STRAT.info("strategy bank: %s · priority %s", list(STRATEGIES), STRATEGY_PRIORITY)'


07:49:11 | INFO    | engine.cell       | ▶ START  [09] Strategy bank
07:49:11 | INFO    | engine.strategy   | strategy bank: ['VCP_BREAKOUT', 'TREND_PULLBACK', 'DONCHIAN_55', 'MEAN_REVERT', 'MOMENTUM_52W', 'EPISODIC_PIVOT'] · priority ['EPISODIC_PIVOT', 'VCP_BREAKOUT', 'MOMENTUM_52W', 'DONCHIAN_55', 'TREND_PULLBACK', 'MEAN_REVERT']
07:49:11 | INFO    | engine.cell       | ✔ DONE   [09] Strategy bank in 0.0s


## 10 · Composite score

A signal is binary; a *ranking* is what turns 40 signals into the 3 you actually take. The score
is a weighted blend of cross-sectional percentile ranks (so it is scale-free and comparable across
a chemicals micro cap and a ₹15,000cr NBFC), then explicit penalties, then the strategy weight.

Percentile ranks are computed **within the same scan date only** — never across time. That is
what keeps the backtest honest: on 12-Mar-2023 the model can only know how a name ranked against
its peers on 12-Mar-2023.

In [106]:
# [10] Composite score
SCORE_WEIGHTS = {
    "rel_63":        0.20,   # relative strength vs the small-cap index
    "momentum":      0.16,   # blended 63/126-day return
    "trend":         0.15,   # EMA stack + slope
    "proximity":     0.12,   # closeness to the 52-week high
    "volume":        0.12,   # thrust / accumulation
    "tightness":     0.12,   # pre-breakout contraction
    "liquidity":     0.08,   # tradeability
    "consensus":     0.05,   # how many strategies agree
}


_SCORE_WARNED = False


def _pct(s: pd.Series) -> pd.Series:
    """Cross-sectional percentile 0-100; NaNs land at the median so they neither help nor hurt."""
    r = s.rank(pct=True) * 100
    return r.fillna(50.0)


def score_candidates(cand: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    if cand.empty:
        return cand.assign(score=[])
    d = cand.copy()
    comp = pd.DataFrame(index=d.index)
    comp["rel_63"] = _pct(d.get("rel_63", pd.Series(np.nan, index=d.index)))
    comp["momentum"] = 0.5 * _pct(d["ret_63"]) + 0.5 * _pct(d["ret_126"])
    comp["trend"] = 0.6 * _pct(d["ema50_slope"]) + 40 * d["stack"].astype(float)
    comp["proximity"] = _pct(d["dist_52w_high"])
    comp["volume"] = 0.6 * _pct(d["vol_z"]) + 40 * d["pocket_pivot"].astype(float)
    comp["tightness"] = 0.5 * _pct(-d["contraction"]) + 0.5 * _pct(-d["bbw_pct"])
    comp["liquidity"] = _pct(d["turnover_med20_cr"])
    comp["consensus"] = np.clip(d.get("n_confirm", 1).astype(float) * 33, 0, 100)

    raw = sum(comp[k] * w for k, w in SCORE_WEIGHTS.items())

    # ---- penalties (points off the raw 0-100 blend) ----
    pen = pd.Series(0.0, index=d.index)
    pen += np.clip(d["ext_atr"] - 2.5, 0, None) * 4.0            # chasing an extended move
    pen += np.clip(d["atr_pct"] - 6.0, 0, None) * 2.5            # too violent to size properly
    pen += np.clip(d["base_depth"] - 35, 0, None) * 0.25         # sloppy, deep base
    pen += np.where(d["gap_pct"] > 6, 8.0, 0.0)                  # already gapped away
    pen += np.where(~d["above_200"].astype(bool), 10.0, 0.0)     # counter-trend
    pen += np.clip(3.0 - d["turnover_med20_cr"], 0, None) * 3.0  # thin

    weight = d["strategy"].map(STRATEGY_WEIGHT).fillna(0.85)
    d["score"] = np.clip((raw - pen) * weight, 0, 100).round(1)
    for k in comp.columns:
        d[f"c_{k}"] = comp[k].round(1)
    d["penalty"] = pen.round(1)
    if (d["rel_63"].isna().all() if "rel_63" in d else True):
        # Warn once per session (the backtester scores every day); later occurrences go to DEBUG.
        lvl = logging.DEBUG if _SCORE_WARNED else logging.WARNING
        LOG_SCREEN.log(lvl, "score: rel_63 is NaN for every candidate on %s — the 20%% relative-"
                       "strength weight is neutralised (benchmark missing, or still in warm-up)",
                       str(pd.Timestamp(d["date"].iloc[0]).date()) if "date" in d else "this bar")
        globals()["_SCORE_WARNED"] = True
    LOG_SCREEN.debug("scored %d candidates · score range %.1f–%.1f · mean penalty %.1f",
                     len(d), d["score"].min(), d["score"].max(), d["penalty"].mean())
    return d.sort_values("score", ascending=False)

07:49:11 | INFO    | engine.cell       | ▶ START  [10] Composite score
07:49:11 | INFO    | engine.cell       | ✔ DONE   [10] Composite score in 0.0s


## 11 · Market regime filter

Small caps have a beta to sentiment far above 1 and their drawdowns are correlated to the point of
being a single position. Trading the same size in every environment is how a good system produces a
40% drawdown. Two inputs:

- **Trend of the benchmark** (Nifty Smallcap 250) vs its 50/200 EMA and its distance from highs.
- **Breadth** — the % of the *tradeable universe* above its own 50-EMA. Breadth turns before price.

Output is an exposure multiplier applied to position size and a cap on new entries per day.

If the benchmark is missing, `assess_regime` now raises a `RuntimeError` that says so (and where
to look), instead of the opaque `IndexError` from `bench_feat.iloc[-1]`.

In [107]:
# [11] Market regime filter — continuous exposure dial with compatibility states
@dataclass
class Regime:
    state: str
    breadth: float
    bench_above_50: bool
    bench_above_200: bool
    bench_dd: float
    exposure_mult: float
    max_new: int
    note: str = ""

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def assess_regime(bench_feat: pd.DataFrame, breadth_pct: float, cfg: Config,
                  verbose: bool = True) -> Regime:
    if bench_feat is None or len(bench_feat) == 0:
        raise RuntimeError(f"assess_regime(): benchmark {cfg.benchmark!r} is empty")
    need = [c for c in ("close", "ema50", "ema200") if c not in bench_feat.columns]
    if need:
        raise RuntimeError(f"assess_regime(): benchmark lacks {need}")
    valid = bench_feat.dropna(subset=["close", "ema50", "ema200"])
    if valid.empty:
        raise RuntimeError("assess_regime(): benchmark has no mature row")
    last = valid.iloc[-1]
    close = valid["close"].astype(float)
    a50 = bool(last["close"] > last["ema50"])
    a200 = bool(last["close"] > last["ema200"])
    dd = float(last["close"] / close.cummax().iloc[-1] - 1) * 100
    vol20 = float(close.pct_change().rolling(20).std().iloc[-1] * math.sqrt(252) * 100)
    if not np.isfinite(vol20):
        vol20 = 25.0

    breadth_component = np.clip((breadth_pct - 20.0) / 60.0, 0, 1)
    trend_component = 0.55 * float(a200) + 0.45 * float(a50)
    dd_component = np.clip((dd + 20.0) / 20.0, 0, 1)
    vol_component = np.clip((45.0 - vol20) / 30.0, 0, 1)
    raw = 0.36*breadth_component + 0.30*trend_component + 0.22*dd_component + 0.12*vol_component
    mult = float(np.clip(cfg.min_exposure_mult + raw*(cfg.max_exposure_mult-cfg.min_exposure_mult),
                         cfg.min_exposure_mult, cfg.max_exposure_mult))

    if mult >= 0.72 and breadth_pct >= cfg.regime_risk_on_breadth:
        state = "RISK_ON"
    elif mult <= 0.38 or breadth_pct <= cfg.regime_risk_off_breadth or (not a200 and dd < -12):
        state = "RISK_OFF"
    else:
        state = "NEUTRAL"
    mx = 0 if state == "RISK_OFF" else max(1, int(round(cfg.max_new_per_day * mult)))
    note = (f"breadth {breadth_pct:.0f}% · index {'>' if a50 else '<'}50EMA "
            f"{'>' if a200 else '<'}200EMA · DD {dd:.1f}% · vol20 {vol20:.1f}% · exposure {mult:.2f}")
    if verbose:
        LOG_REGIME.info("→ %s · continuous exposure ×%.2f · max %d new/day | %s", state, mult, mx, note)
    return Regime(state, round(breadth_pct, 1), a50, a200, round(dd, 2), round(mult, 3), mx, note)


07:49:11 | INFO    | engine.cell       | ▶ START  [11] Market regime filter — continuous exposure dial with compatibility states
07:49:11 | INFO    | engine.cell       | ✔ DONE   [11] Market regime filter — continuous exposure dial with compatibility states in 0.0s


## 12 · Cost & slippage model

NSE delivery (CNC) round-trip, as of the Groww retail schedule. Verify against your latest
contract note — these rates change with every budget.

| Component | Rate |
|---|---|
| Brokerage | 0.1% or ₹20 per order, whichever is lower |
| STT | 0.1% on buy **and** sell (delivery) |
| Exchange txn | ~0.00297% of turnover |
| SEBI | ₹10 per crore |
| Stamp duty | 0.015% on buy only |
| GST | 18% on (brokerage + txn + SEBI) |
| DP charge | ~₹18.5 + GST per scrip on sell |

**Slippage** is modelled as a base spread cost plus a square-root impact term scaled by your
participation in the 20-day median volume. On a name doing ₹4cr a day, a ₹60k order is nothing;
on a name doing ₹40 lakh a day it is the whole top of book, and the model charges you for it.

In [108]:
# [12] Cost & slippage model
def slippage_bps(order_shares: float, adv_shares: float, cfg: Config) -> float:
    if not adv_shares or adv_shares <= 0:
        return cfg.slippage_bps * 3
    participation = min(order_shares / adv_shares, 0.5)
    impact = 45.0 * math.sqrt(max(participation, 0.0))     # sqrt-law market impact
    return cfg.slippage_bps + impact


def transaction_costs(price: float, qty: int, side: str, cfg: Config) -> Dict[str, float]:
    turnover = price * qty
    if turnover <= 0:
        return {"total": 0.0}
    brokerage = min(turnover * cfg.brokerage_pct / 100, cfg.brokerage_cap)
    stt = turnover * 0.001                                  # both legs, delivery
    exch = turnover * 0.0000297
    sebi = turnover * 0.000001
    stamp = turnover * 0.00015 if side == Txn.BUY else 0.0
    dp = 0.0 if side == Txn.BUY else 18.5
    gst = 0.18 * (brokerage + exch + sebi + dp)
    total = brokerage + stt + exch + sebi + stamp + dp + gst
    return {"brokerage": brokerage, "stt": stt, "exchange": exch, "sebi": sebi,
            "stamp": stamp, "dp": dp, "gst": gst, "total": total,
            "bps": total / turnover * 10_000}


_c = transaction_costs(500.0, 200, Txn.BUY, CFG), transaction_costs(500.0, 200, Txn.SELL, CFG)
LOG_PLAN.info("cost model check | ₹1L round trip: buy ₹%.0f (%.1f bps) + sell ₹%.0f (%.1f bps) · "
              "base slippage %.0f bps", _c[0]["total"], _c[0]["bps"], _c[1]["total"], _c[1]["bps"],
              CFG.slippage_bps)

07:49:11 | INFO    | engine.cell       | ▶ START  [12] Cost & slippage model
07:49:11 | INFO    | engine.plan       | cost model check | ₹1L round trip: buy ₹142 (14.2 bps) + sell ₹149 (14.9 bps) · base slippage 12 bps
07:49:11 | INFO    | engine.cell       | ✔ DONE   [12] Cost & slippage model in 0.0s


## 13 · Position sizing & portfolio risk

Risk-first, never notional-first. The quantity falls out of *how far away the stop is*, then gets
clipped by four independent caps. If any cap binds hard enough that quantity rounds to zero, the
trade is skipped — a position too small to matter is a position that only pays brokerage.

```
qty = (equity × risk% × regime_mult) / (entry − stop)
      ↓ capped by max_position_pct of equity
      ↓ capped by max_adv_participation_pct of 20-day median volume
      ↓ capped by remaining portfolio heat
      ↓ capped by available cash
```

In [109]:
# [13] Position sizing & portfolio risk
@dataclass
class SizingResult:
    qty: int
    risk_amount: float
    notional: float
    r_per_share: float
    binding_cap: str
    ok: bool
    reason: str = ""


def size_position(entry: float, stop: float, equity: float, cash: float,
                  adv_shares: float, cfg: Config, regime_mult: float = 1.0,
                  open_risk: float = 0.0) -> SizingResult:
    r_per_share = entry - stop
    if r_per_share <= 0 or entry <= 0:
        return SizingResult(0, 0, 0, 0, "invalid", False, "stop is not below entry")

    stop_dist_pct = r_per_share / entry * 100
    if stop_dist_pct > cfg.max_stop_pct:
        return SizingResult(0, 0, 0, r_per_share, "stop_too_wide", False,
                            f"stop {stop_dist_pct:.1f}% away > v2 cap {cfg.max_stop_pct:.1f}%")

    risk_budget = equity * cfg.risk_per_trade_pct / 100 * regime_mult
    heat_left = equity * cfg.max_portfolio_heat_pct / 100 - open_risk
    if heat_left <= 0:
        return SizingResult(0, 0, 0, r_per_share, "portfolio_heat", False, "book is at max heat")
    risk_budget = min(risk_budget, heat_left)

    qty_risk = risk_budget / r_per_share
    qty_notional = (equity * cfg.max_position_pct / 100) / entry
    qty_adv = (adv_shares or 0) * cfg.max_adv_participation_pct / 100
    qty_cash = cash / entry

    caps = {"risk": qty_risk, "notional": qty_notional, "liquidity": qty_adv, "cash": qty_cash}
    binding = min(caps, key=lambda k: caps[k])
    qty = int(math.floor(min(caps.values())))

    LOG_PLAN.debug("sizing entry=%.2f stop=%.2f (%.1f%%) · caps risk=%.0f notional=%.0f "
                   "liquidity=%.0f cash=%.0f → %d (%s)", entry, stop, stop_dist_pct, qty_risk,
                   qty_notional, qty_adv, qty_cash, max(qty, 0), binding)
    if qty <= 0:
        return SizingResult(0, 0, 0, r_per_share, binding, False, f"{binding} cap allows 0 shares")
    return SizingResult(qty, qty * r_per_share, qty * entry, r_per_share, binding, True)

07:49:11 | INFO    | engine.cell       | ▶ START  [13] Position sizing & portfolio risk
07:49:11 | INFO    | engine.cell       | ✔ DONE   [13] Position sizing & portfolio risk in 0.0s


## 14 · Trade plan

The screener's output is not "buy XYZ". It is a complete, pre-committed plan: trigger, invalidation,
two targets, a trailing rule, a time stop and an expiry. Everything that will ever be decided about
this trade is decided *now*, before there is a P&L to make you stupid.

In [110]:
# [14] Trade plan
@dataclass
class TradePlan:
    id: str
    scan_date: str
    symbol: str
    name: str
    sector: str
    strategy: str
    score: float
    entry_type: str            # BUY_STOP | LIMIT
    entry_trigger: float
    valid_till: str
    init_stop: float
    r_per_share: float
    t1: float
    t2: float
    qty: int
    risk_amount: float
    notional: float
    risk_pct_equity: float
    stop_pct: float
    trail_rule: str
    time_stop_bars: int
    regime: str
    binding_cap: str
    features: Dict[str, float] = field(default_factory=dict)
    llm: Dict[str, Any] = field(default_factory=dict)
    thesis: str = ""

    def to_row(self) -> Dict[str, Any]:
        return {"id": self.id, "scan_date": self.scan_date, "symbol": self.symbol,
                "strategy": self.strategy, "score": self.score, "entry_type": self.entry_type,
                "entry_trigger": self.entry_trigger, "stop": self.init_stop, "t1": self.t1,
                "t2": self.t2, "qty": self.qty, "risk_amount": self.risk_amount,
                "r_per_share": self.r_per_share, "regime": self.regime,
                "features": json.dumps(self.features, default=str),
                "llm": json.dumps(self.llm, default=str),
                "status": "PROPOSED", "created_at": now_ist().isoformat()}

    def telegram(self) -> str:
        rr1 = (self.t1 - self.entry_trigger) / max(self.r_per_share, 1e-9)
        where = "above" if self.entry_type == "BUY_STOP" else "at/below"
        return (
            f"*{self.symbol}* · {self.strategy} · score *{self.score:.0f}*\n"
            f"`{self.entry_type}` {where} *₹{self.entry_trigger:,.2f}*  (valid till {self.valid_till})\n"
            f"SL *₹{self.init_stop:,.2f}*  ({self.stop_pct:.1f}%)   Qty *{self.qty}*\n"
            f"T1 ₹{self.t1:,.2f} ({rr1:.1f}R, book {int(CFG.t1_exit_fraction*100)}%) → "
            f"T2 ₹{self.t2:,.2f}\n"
            f"Risk ₹{self.risk_amount:,.0f} ({self.risk_pct_equity:.2f}% of book) · "
            f"cap: {self.binding_cap}\n"
            f"_{self.thesis}_"
        )


def build_trade_plan(row: pd.Series, cfg: Config, equity: float, cash: float,
                     regime: Regime, open_risk: float, scan_date: str) -> Optional[TradePlan]:
    entry, stop = float(row["entry"]), float(row["stop"])
    sizing = size_position(entry, stop, equity, cash, float(row.get("adv20", 0) or 0),
                           cfg, regime.exposure_mult, open_risk)
    if not sizing.ok:
        LOG_PLAN.info("✗ %s (%s, score %.1f) not sized: %s", row["symbol"], row.get("strategy"),
                      float(row.get("score", float("nan"))), sizing.reason)
        return None

    r = sizing.r_per_share
    t1 = entry + cfg.t1_r_multiple * r
    t2 = entry + cfg.t2_r_multiple * r
    # A measured-move sanity check: don't set T2 beyond a plausible extension of the base.
    if np.isfinite(row.get("base_depth", np.nan)):
        measured = entry * (1 + float(row["base_depth"]) / 100 * 0.9)
        t2 = min(t2, max(measured, t1 * 1.05))

    thesis = (f"{row['strategy'].replace('_',' ').title()} · RS {row.get('rel_63', 0):+.0f}% vs index · "
              f"{row.get('dist_52w_high', 0):+.0f}% from 52w high · ADX {row.get('adx', 0):.0f} · "
              f"ATR {row.get('atr_pct', 0):.1f}%")

    rs = row.get("rel_63", np.nan)
    thesis = thesis.replace(f"RS {row.get('rel_63', 0):+.0f}% vs index",
                            f"RS {rs:+.0f}% vs index" if pd.notna(rs) else "RS n/a (no benchmark)")
    LOG_PLAN.info("✓ %s %s score=%.1f · %s @ %.2f · SL %.2f (%.1f%%) · T1 %.2f · T2 %.2f · qty %d · "
                  "risk ₹%.0f · cap=%s", row["symbol"], row["strategy"], float(row["score"]),
                  "LIMIT" if row["strategy"] == "MEAN_REVERT" else "BUY_STOP", entry, stop,
                  r / entry * 100, t1, t2, sizing.qty, sizing.risk_amount, sizing.binding_cap)
    return TradePlan(
        id=f"SIG-{uuid.uuid4().hex[:10]}", scan_date=scan_date,
        symbol=row["symbol"], name=str(row.get("name", row["symbol"])),
        sector=str(row.get("sector", "UNKNOWN")), strategy=row["strategy"],
        score=float(row["score"]), entry_type="LIMIT" if row["strategy"] == "MEAN_REVERT" else "BUY_STOP",
        entry_trigger=round(entry, 2), valid_till=str(next_trading_day(pd.Timestamp(scan_date).date())),
        init_stop=round(stop, 2), r_per_share=round(r, 2),
        t1=round(t1, 2), t2=round(t2, 2), qty=sizing.qty,
        risk_amount=round(sizing.risk_amount, 0), notional=round(sizing.notional, 0),
        risk_pct_equity=round(sizing.risk_amount / equity * 100, 2),
        stop_pct=round(r / entry * 100, 2),
        trail_rule=f"chandelier {cfg.chandelier_mult}xATR after {cfg.trail_activate_r}R; "
                   f"BE after {cfg.breakeven_after_r}R",
        time_stop_bars=cfg.time_stop_bars, regime=regime.state, binding_cap=sizing.binding_cap,
        features={k: float(row[k]) for k in
                  ("score", "adx", "atr_pct", "rel_63", "ret_63", "dist_52w_high", "vol_ratio",
                   "contraction", "ext_atr", "turnover_med20_cr", "rsi14")
                  if k in row and pd.notna(row[k])},
        thesis=thesis,
    )

07:49:11 | INFO    | engine.cell       | ▶ START  [14] Trade plan
07:49:11 | INFO    | engine.cell       | ✔ DONE   [14] Trade plan in 0.0s


## 15 · The screener

Puts sections 6–14 together into one call. Returns the ranked candidate table, the regime, and the
feature panel (kept in memory so the lifecycle manager can reuse ATR without refetching).

In [111]:
# [15] The screener
def compute_panel(cfg: Config, hub: DataHub, universe: pd.DataFrame,
                  refresh: bool = True) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    """Fetch bars → benchmark → features. PATCHED: the benchmark is validated *before* anything
    uses it; if every source fails, fall back to a universe proxy or stop with a clear error."""
    if universe is None or universe.empty:
        raise RuntimeError("compute_panel(): the universe is empty — run the universe build first")
    syms = universe["symbol"].tolist()
    LOG_SCREEN.info("compute_panel: %d symbols · %d days · refresh=%s", len(syms), cfg.history_days, refresh)

    with log_step("fetch universe bars", LOG_SCREEN):
        raw = hub.bulk(syms, lookback_days=cfg.history_days, refresh=refresh,
                       workers=cfg.historical_workers, label="panel")

    # ---- benchmark: fetch, validate, fall back ----
    bench_df = hub.benchmark(cfg.history_days)
    if bench_df is None or len(bench_df) < cfg.min_benchmark_bars:
        n = 0 if bench_df is None else len(bench_df)
        if cfg.benchmark_allow_universe_proxy:
            LOG_BENCH.warning("benchmark %r returned %d bars (< %d) — building the universe proxy",
                              cfg.benchmark, n, cfg.min_benchmark_bars)
            bench_df = hub.universe_proxy(raw, cfg.min_history_bars)
        if bench_df is None or len(bench_df) < cfg.min_benchmark_bars:
            raise RuntimeError(
                f"benchmark {cfg.benchmark!r} returned {0 if bench_df is None else len(bench_df)} bars "
                f"(need ≥{cfg.min_benchmark_bars}). Check the engine.benchmark log lines, "
                "HUB.broker.find_index_instruments(CFG.benchmark_search_terms), DEAD_SYMBOLS and the "
                "FETCH_FAIL events.")
    bench_close = bench_df["close"]
    LOG_BENCH.info("benchmark in use: %s · %s", hub.bench_source, describe_frame(bench_df))

    # ---- features ----
    panel: Dict[str, pd.DataFrame] = {}
    short: List[str] = []
    failed: List[str] = []
    with log_step(f"compute features for {len(syms)} symbols", LOG_FEAT):
        for s in syms:
            df = raw.get(s)
            if df is None or len(df) < cfg.min_history_bars:
                short.append(f"{s}({0 if df is None else len(df)})")
                continue
            try:
                panel[s] = compute_features(df, bench_close)
            except Exception as e:  # noqa: BLE001
                failed.append(s)
                LOG_FEAT.warning("features failed for %s: %s: %s", s, type(e).__name__, str(e)[:160])
    bench_feat = compute_features(bench_df)

    LOG_FEAT.info("panel: %d/%d symbols have features · %d skipped for < %d bars%s · %d failed",
                  len(panel), len(syms), len(short), cfg.min_history_bars,
                  f" (e.g. {short[:6]})" if short else "", len(failed))
    if not panel:
        raise RuntimeError("compute_panel(): no symbol had enough history — nothing to screen")

    # ---- alignment / health checks ----
    last_dates = pd.Series({s: f.index[-1] for s, f in panel.items()})
    newest = last_dates.max()
    stale = last_dates[last_dates < newest - pd.Timedelta(days=5)]
    b_last = bench_feat.index[-1]
    LOG_FEAT.info("latest bar: universe %s · benchmark %s%s", pd.Timestamp(newest).date(),
                  pd.Timestamp(b_last).date(), f" · {len(stale)} symbols stale >5d" if len(stale) else "")
    if pd.Timestamp(b_last) < pd.Timestamp(newest) - pd.Timedelta(days=3):
        LOG_BENCH.warning("benchmark ends %s but stocks run to %s — RS/regime use stale index data",
                          pd.Timestamp(b_last).date(), pd.Timestamp(newest).date())
    rs_ok = sum(1 for f in panel.values() if pd.notna(f["rel_63"].iloc[-1]))
    LOG_FEAT.info("relative strength (rel_63) available on the latest bar for %d/%d symbols",
                  rs_ok, len(panel))
    if rs_ok < 0.8 * len(panel):
        LOG_FEAT.warning("RS is missing for %d symbols — benchmark dates don't cover their latest bars",
                         len(panel) - rs_ok)
    return panel, bench_feat


def breadth_pct(panel: Dict[str, pd.DataFrame], asof: Optional[pd.Timestamp] = None) -> float:
    vals = []
    for f in panel.values():
        row = f.iloc[-1] if asof is None else (f.loc[:asof].iloc[-1] if len(f.loc[:asof]) else None)
        if row is not None and np.isfinite(row.get("ema50", np.nan)):
            vals.append(float(row["close"] > row["ema50"]))
    if not vals:
        LOG_REGIME.warning("breadth: no symbol had a valid 50-EMA — defaulting to 50%")
        return 50.0
    out = 100 * float(np.mean(vals))
    if asof is None:
        LOG_REGIME.info("breadth: %d/%d names above their 50-EMA = %.1f%%",
                        int(sum(vals)), len(vals), out)
    return out


FEATURE_COLS = ["close", "open", "high", "low", "atr", "atr_pct", "adx", "rsi14", "rsi2",
                "ema20", "ema50", "ema200", "ema50_slope", "stack", "above_200", "vol_ratio",
                "vol_z", "pocket_pivot", "contraction", "bbw_pct", "dist_52w_high", "base_depth",
                "ext_atr", "gap_pct", "ret_21", "ret_63", "ret_126", "rel_63",
                "turnover_med20_cr", "adv20", "dry_up"]


def collect_candidates(panel: Dict[str, pd.DataFrame], universe: pd.DataFrame, cfg: Config,
                       asof: Optional[pd.Timestamp] = None) -> pd.DataFrame:
    meta = universe.set_index("symbol")[["name", "sector"]].to_dict("index")
    rows = []
    for sym, f in panel.items():
        ff = f if asof is None else f.loc[:asof]
        if len(ff) < cfg.min_history_bars:
            continue
        sig = run_strategies(ff, cfg)
        last = sig.iloc[-1]
        if not bool(last["signal"]):
            continue
        feat = ff.iloc[-1]
        row = {"symbol": sym, "date": ff.index[-1],
               "strategy": last["strategy"], "entry": float(last["entry"]),
               "stop": float(last["stop"]), "n_confirm": int(last["n_confirm"])}
        row.update({c: float(feat[c]) if not isinstance(feat[c], (bool, np.bool_)) else bool(feat[c])
                    for c in FEATURE_COLS if c in feat})
        row.update(meta.get(sym, {"name": sym, "sector": "UNKNOWN"}))
        rows.append(row)
    out = pd.DataFrame(rows)
    if asof is None:
        LOG_SCREEN.info("raw signals on the latest bar: %d/%d names%s", len(out), len(panel),
                        f" · by strategy {out['strategy'].value_counts().to_dict()}" if len(out) else "")
    return out


def apply_hard_filters(cand: pd.DataFrame, cfg: Config, log_gates: bool = True) -> pd.DataFrame:
    if cand.empty:
        return cand
    gates = {
        "turnover": cand["turnover_med20_cr"].ge(cfg.min_median_turnover_cr),
        "atr_band": cand["atr_pct"].between(cfg.min_atr_pct, cfg.max_atr_pct),
        "price_band": cand["close"].between(cfg.price_min, cfg.price_max),
        "stop<entry": cand["stop"] < cand["entry"],
        "stop≤15%": ((cand["entry"] - cand["stop"]) / cand["entry"] * 100).le(15),
    }
    m = pd.Series(True, index=cand.index)
    for g in gates.values():
        m &= g
    if log_gates:
        fails = {k: int((~v).sum()) for k, v in gates.items() if int((~v).sum())}
        LOG_SCREEN.info("hard filters: %d → %d%s", len(cand), int(m.sum()),
                        f" · failures per gate {fails}" if fails else "")
    return cand.loc[m].copy()


def run_screener(cfg: Config, hub: DataHub, universe: pd.DataFrame,
                 refresh: bool = True, panel: Optional[Dict[str, pd.DataFrame]] = None,
                 bench_feat: Optional[pd.DataFrame] = None) -> Dict[str, Any]:
    t0 = time.time()
    if panel is None:
        panel, bench_feat = compute_panel(cfg, hub, universe, refresh)
    if bench_feat is None:
        raise RuntimeError("run_screener(): got a panel but no bench_feat — pass both, or neither")
    br = breadth_pct(panel)
    regime = assess_regime(bench_feat, br, cfg)

    cand = apply_hard_filters(collect_candidates(panel, universe, cfg), cfg)
    scored = score_candidates(cand, cfg) if not cand.empty else cand
    if not scored.empty:
        below = scored[scored["score"] < cfg.min_score]
        if len(below):
            LOG_SCREEN.info("score gate (≥%.0f) removes %d: %s", cfg.min_score, len(below),
                            ", ".join(f"{s}={v:.0f}" for s, v in
                                      zip(below["symbol"].head(8), below["score"].head(8))))
        scored = scored[scored["score"] >= cfg.min_score]
    LOG_SCREEN.info("scan: %d signals after hard filters → %d after score | regime=%s (%s) | %.1fs",
                    len(cand), len(scored), regime.state, regime.note, time.time() - t0)
    if not scored.empty:
        LOG_SCREEN.info("top candidates: %s", ", ".join(
            f"{r.symbol}({r.strategy},{r.score:.0f})" for r in scored.head(8).itertuples()))
    return {"candidates": scored, "regime": regime, "panel": panel,
            "bench": bench_feat, "breadth": br}

07:49:11 | INFO    | engine.cell       | ▶ START  [15] The screener
07:49:11 | INFO    | engine.cell       | ✔ DONE   [15] The screener in 0.0s


## 16 · Trade lifecycle — the state machine

```
                   ┌──────────── rejected / expired ────────────┐
                   │                                            ▼
 PROPOSED ─► PENDING_APPROVAL ─► ENTRY_WORKING ─► OPEN ─► SCALED_OUT ─► CLOSED
     │            (Telegram)      (buy-stop live)   │        (T1 hit,        ▲
     │                                 │            │      stop→breakeven)   │
     └── score gate / risk veto        │            └──► TRAILING ───────────┘
                                       └── unfilled by valid_till ─► EXPIRED
```

Exit logic lives in **one pure function**, `exit_decisions`, used by both the live manager and the
backtester. That is deliberate: the most common way a backtest lies to you is that the live code
and the simulated code drift apart. Here they cannot.

In [112]:
# [16a] Trade lifecycle — states & exit rules
class TState(StrEnum):
    PROPOSED = "PROPOSED"
    PENDING_APPROVAL = "PENDING_APPROVAL"
    ENTRY_WORKING = "ENTRY_WORKING"
    OPEN = "OPEN"
    SCALED_OUT = "SCALED_OUT"
    TRAILING = "TRAILING"
    EXIT_WORKING = "EXIT_WORKING"
    CLOSED = "CLOSED"
    CANCELLED = "CANCELLED"
    EXPIRED = "EXPIRED"
    REJECTED = "REJECTED"


@dataclass
class Position:
    """Live view of one open trade. Mirrors the `trades` table."""
    id: str
    symbol: str
    strategy: str
    qty: int                       # open quantity
    init_qty: int
    avg_entry: float
    init_stop: float
    cur_stop: float
    t1: float
    t2: float
    r: float                       # ₹ risk per share at entry
    entry_date: str
    bars_held: int = 0
    scaled: bool = False
    be_moved: bool = False
    high_water: float = 0.0
    mfe_r: float = 0.0
    mae_r: float = 0.0
    state: str = TState.OPEN
    sector: str = "UNKNOWN"

    def unrealised_r(self, px: float) -> float:
        return (px - self.avg_entry) / self.r if self.r else 0.0


@dataclass
class Action:
    kind: str          # EXIT_ALL | SCALE_OUT | MOVE_STOP
    qty: int
    price: float
    reason: str


def exit_decisions(p: Position, bar: Dict[str, float], atr: float,
                   cfg: Config, regime_state: str = "NEUTRAL") -> List[Action]:
    """Pure. Given a position and one daily bar, return the actions to take, in execution order.

    Intrabar convention (conservative, matches how you'd actually be filled):
      1. A gap through the stop fills at the open, not the stop.
      2. If both the stop and T1 are inside the bar's range, the stop is assumed to hit first.
      3. Trailing stops are recalculated on the close and take effect the *next* session.
    """
    acts: List[Action] = []
    o, h, l, c = bar["open"], bar["high"], bar["low"], bar["close"]

    # --- 1. protective stop -------------------------------------------------
    # If the bar is locked at a lower circuit there is no executable exit at the theoretical stop.
    # Carry the position; the next available bar will realize the gap. This requires a positively
    # identified lock from data/surveillance logic; ordinary gap-down bars are still exited.
    if bool(bar.get("lower_circuit_locked", False)):
        return []
    if o <= p.cur_stop:
        return [Action("EXIT_ALL", p.qty, o, "STOP_GAP")]
    if l <= p.cur_stop:
        return [Action("EXIT_ALL", p.qty, p.cur_stop, "STOP")]

    # --- 2. targets ---------------------------------------------------------
    if not p.scaled and h >= p.t1 and p.qty > 1:
        q = max(1, int(round(p.qty * cfg.t1_exit_fraction)))
        acts.append(Action("SCALE_OUT", q, max(p.t1, o), "T1"))
        acts.append(Action("MOVE_STOP", 0, p.avg_entry, "BREAKEVEN_AFTER_T1"))
    if p.scaled and h >= p.t2:
        return acts + [Action("EXIT_ALL", p.qty, max(p.t2, o), "T2")]

    # --- 3. follow-through, time & regime stops (evaluated on the close) -----
    r_now = p.unrealised_r(c)
    # v2: release dead capital early, but never use this after the trade has demonstrated
    # a right-tail move. Parameters are validated only in walk-forward research.
    if (cfg.use_follow_through_exit and not p.scaled and
            p.bars_held >= cfg.follow_through_bars and
            p.mfe_r < cfg.follow_through_min_r and
            r_now < cfg.follow_through_min_r and
            p.mfe_r < cfg.keep_right_tail_after_r):
        return acts + [Action("EXIT_ALL", p.qty, c, "NO_FOLLOW_THROUGH")]
    if p.bars_held >= cfg.time_stop_bars and r_now < 1.0 and not p.scaled:
        return acts + [Action("EXIT_ALL", p.qty, c, "TIME_STOP")]
    if p.bars_held >= cfg.max_hold_bars:
        return acts + [Action("EXIT_ALL", p.qty, c, "MAX_HOLD")]
    if regime_state == "RISK_OFF" and r_now < 0.5 and p.bars_held >= 3:
        return acts + [Action("EXIT_ALL", p.qty, c, "REGIME_EXIT")]

    # --- 4. trail -----------------------------------------------------------
    new_stop = p.cur_stop
    if r_now >= cfg.breakeven_after_r and not p.be_moved:
        new_stop = max(new_stop, p.avg_entry + 0.05 * p.r)      # BE + a whisker for costs
    if r_now >= cfg.trail_activate_r and np.isfinite(atr) and atr > 0:
        chandelier = max(h, p.high_water, c) - cfg.chandelier_mult * atr
        new_stop = max(new_stop, chandelier)
    new_stop = min(new_stop, c * 0.995)                          # never above the market
    if new_stop > p.cur_stop + 1e-6:
        acts.append(Action("MOVE_STOP", 0, round(new_stop, 2), "TRAIL"))
    return acts

07:49:11 | INFO    | engine.cell       | ▶ START  [16a] Trade lifecycle — states & exit rules
07:49:11 | INFO    | engine.cell       | ✔ DONE   [16a] Trade lifecycle — states & exit rules in 0.0s


In [113]:
# [16b] Trade lifecycle — the engine
class LifecycleEngine:
    """Persists and drives trades from proposal to closure against any BrokerAdapter."""

    def __init__(self, cfg: Config, broker: BrokerAdapter, store: Store, notifier=None):
        self.cfg, self.broker, self.store, self.notifier = cfg, broker, store, notifier
        self.positions: Dict[str, Position] = {}
        self.pending: Dict[str, Dict[str, Any]] = {}     # trade_id -> entry order info
        self._load_state()

    # ---------------- persistence ----------------
    def _load_state(self) -> None:
        df = self.store.query(
            "SELECT * FROM trades WHERE state NOT IN ('CLOSED','CANCELLED','EXPIRED','REJECTED')")
        for _, t in df.iterrows():
            if t["state"] in (TState.OPEN, TState.SCALED_OUT, TState.TRAILING):
                self.positions[t["id"]] = Position(
                    id=t["id"], symbol=t["symbol"], strategy=t["strategy"],
                    qty=int(t["open_qty"] or 0), init_qty=int(t["filled_qty"] or 0),
                    avg_entry=float(t["avg_entry"] or 0), init_stop=float(t["init_stop"] or 0),
                    cur_stop=float(t["cur_stop"] or 0), t1=float(t["t1"] or 0),
                    t2=float(t["t2"] or 0), r=float(t["r_per_share"] or 1),
                    entry_date=str(t["entry_date"]), bars_held=int(t["bars_held"] or 0),
                    scaled=t["state"] == TState.SCALED_OUT, state=t["state"])
        if self.positions:
            LOG_LC.info("restored %d open positions from db: %s", len(self.positions),
                        ", ".join(f"{p.symbol}×{p.qty}@{p.avg_entry:.2f} SL {p.cur_stop:.2f}"
                                  for p in self.positions.values()))
        else:
            LOG_LC.info("no open positions in the db")
        other = df[~df["id"].isin(list(self.positions))] if len(df) else df
        if len(other):
            LOG_LC.info("%d non-terminal trades not restored as positions (states: %s) — pending "
                        "entries are not re-armed after a restart", len(other),
                        other["state"].value_counts().to_dict())

    def _persist(self, t: Dict[str, Any]) -> None:
        t["updated_at"] = now_ist().isoformat()
        self.store.upsert("trades", t)

    def _persist_position(self, p: Position, **extra) -> None:
        self._persist({"id": p.id, "symbol": p.symbol, "strategy": p.strategy, "state": p.state,
                       "side": "LONG", "filled_qty": p.init_qty, "open_qty": p.qty,
                       "avg_entry": p.avg_entry, "init_stop": p.init_stop, "cur_stop": p.cur_stop,
                       "t1": p.t1, "t2": p.t2, "r_per_share": p.r, "bars_held": p.bars_held,
                       "mfe_r": p.mfe_r, "mae_r": p.mae_r, "entry_date": p.entry_date, **extra})

    def _log_order(self, trade_id: str, resp: Dict[str, Any], purpose: str,
                   symbol: str, txn: str, qty: int, order_type: str,
                   price: float = 0, trigger: float = 0) -> str:
        oid = resp.get("order_id", "")
        self.store.upsert("orders", {
            "id": f"ORD-{uuid.uuid4().hex[:10]}", "trade_id": trade_id, "broker_order_id": oid,
            "symbol": symbol, "txn": str(txn), "order_type": str(order_type), "qty": qty,
            "price": price, "trigger_price": trigger, "status": str(resp.get("status", "")),
            "filled_qty": 0, "avg_price": 0.0, "purpose": purpose,
            "created_at": now_ist().isoformat(), "updated_at": now_ist().isoformat(),
            "raw": json.dumps(resp.get("raw", {}), default=str)})
        return oid

    # ---------------- risk gates ----------------
    def portfolio_open_risk(self) -> float:
        return sum(max(p.avg_entry - p.cur_stop, 0) * p.qty for p in self.positions.values())

    def sector_count(self, sector: str) -> int:
        return sum(1 for p in self.positions.values() if p.sector == sector)

    def can_open(self, plan: TradePlan) -> Tuple[bool, str]:
        if len(self.positions) >= self.cfg.max_positions:
            return False, "max positions"
        if any(p.symbol == plan.symbol for p in self.positions.values()):
            return False, "already long"
        if plan.id in self.pending:
            return False, "entry already working"
        if self.sector_count(plan.sector) >= self.cfg.max_sector_positions:
            return False, f"sector cap ({plan.sector})"
        if self.store.get_kv("halted", False):
            return False, "engine halted by kill switch"
        return True, ""

    # ---------------- entry ----------------
    def propose(self, plan: TradePlan) -> str:
        ok, why = self.can_open(plan)
        if not ok:
            LOG_LC.info("proposal %s REJECTED: %s", plan.symbol, why)
        state = TState.PROPOSED if ok else TState.REJECTED
        if ok and self.cfg.require_approval and self.cfg.mode == "LIVE":
            state = TState.PENDING_APPROVAL
        self.store.upsert("signals", plan.to_row())
        self._persist({"id": plan.id, "signal_id": plan.id, "symbol": plan.symbol,
                       "strategy": plan.strategy, "state": state, "side": "LONG",
                       "planned_qty": plan.qty, "filled_qty": 0, "open_qty": 0,
                       "entry_trigger": plan.entry_trigger, "init_stop": plan.init_stop,
                       "cur_stop": plan.init_stop, "t1": plan.t1, "t2": plan.t2,
                       "r_per_share": plan.r_per_share, "realised_pnl": 0.0, "fees": 0.0,
                       "bars_held": 0, "notes": plan.thesis})
        self.store.event("PROPOSE", f"{plan.symbol} {plan.strategy} score={plan.score} -> {state} {why}",
                         symbol=plan.symbol, trade_id=plan.id, payload=plan.features)
        return state

    def submit_entry(self, plan: TradePlan) -> bool:
        ok, why = self.can_open(plan)
        if not ok:
            LOG_LC.info("entry blocked %s: %s", plan.symbol, why)
            return False
        otype = OrdType.SL_M if plan.entry_type == "BUY_STOP" else OrdType.LIMIT
        resp = self.broker.place_order(
            symbol=plan.symbol, txn=Txn.BUY, qty=plan.qty, order_type=otype,
            price=plan.entry_trigger if otype == OrdType.LIMIT else 0.0,
            trigger_price=plan.entry_trigger if otype == OrdType.SL_M else 0.0,
            ref_id=plan.id.replace("SIG-", "E"))
        oid = self._log_order(plan.id, resp, "ENTRY", plan.symbol, Txn.BUY, plan.qty, otype,
                              plan.entry_trigger, plan.entry_trigger)
        self.pending[plan.id] = {"plan": plan, "order_id": oid, "placed": now_ist().isoformat()}
        self._persist({"id": plan.id, "state": TState.ENTRY_WORKING, "symbol": plan.symbol})
        self.store.event("ENTRY_SENT", f"{plan.symbol} {otype} {plan.qty}@{plan.entry_trigger}",
                         symbol=plan.symbol, trade_id=plan.id)
        if self.notifier:
            self.notifier.send(f"📤 Entry working: {plan.symbol} {plan.qty} @ {plan.entry_trigger}")
        return True

    def on_entry_fill(self, trade_id: str, qty: int, price: float, sector: str = "UNKNOWN") -> Position:
        plan: TradePlan = self.pending[trade_id]["plan"]
        r = max(price - plan.init_stop, 0.01)
        p = Position(id=trade_id, symbol=plan.symbol, strategy=plan.strategy, qty=qty, init_qty=qty,
                     avg_entry=price, init_stop=plan.init_stop, cur_stop=plan.init_stop,
                     t1=price + self.cfg.t1_r_multiple * r, t2=price + self.cfg.t2_r_multiple * r,
                     r=r, entry_date=str(now_ist().date()), high_water=price,
                     state=TState.OPEN, sector=sector)
        self.positions[trade_id] = p
        self.pending.pop(trade_id, None)
        # protective stop goes in immediately — never hold a position without one
        resp = self.broker.place_order(symbol=p.symbol, txn=Txn.SELL, qty=qty,
                                       order_type=OrdType.SL_M, trigger_price=p.cur_stop,
                                       ref_id=trade_id.replace("SIG-", "S"))
        self._log_order(trade_id, resp, "STOP", p.symbol, Txn.SELL, qty, OrdType.SL_M, 0, p.cur_stop)
        self._persist_position(p, avg_exit=None, exit_reason=None)
        self.store.event("FILL", f"{p.symbol} entry {qty}@{price:.2f} SL {p.cur_stop:.2f}",
                         symbol=p.symbol, trade_id=trade_id)
        if self.notifier:
            self.notifier.send(f"✅ *FILLED* {p.symbol} {qty} @ ₹{price:,.2f}\n"
                               f"SL ₹{p.cur_stop:,.2f} · T1 ₹{p.t1:,.2f} · T2 ₹{p.t2:,.2f}")
        return p

    def expire_stale_entries(self) -> None:
        for tid, info in list(self.pending.items()):
            plan: TradePlan = info["plan"]
            if str(now_ist().date()) > plan.valid_till:
                try:
                    self.broker.cancel_order(info["order_id"])
                except Exception as e:  # noqa: BLE001
                    LOG_LC.warning("cancel failed %s: %s", tid, e)
                self.pending.pop(tid, None)
                self._persist({"id": tid, "state": TState.EXPIRED, "symbol": plan.symbol})
                self.store.event("EXPIRE", f"{plan.symbol} entry not triggered", symbol=plan.symbol,
                                 trade_id=tid)

    # ---------------- management ----------------
    def apply_actions(self, p: Position, acts: List[Action], costs: bool = True) -> float:
        realised = 0.0
        for a in acts:
            if a.kind == "MOVE_STOP":
                p.cur_stop = a.price
                p.be_moved = p.be_moved or a.reason.startswith("BREAKEVEN")
                p.state = TState.TRAILING if a.reason == "TRAIL" else p.state
                try:
                    self.broker.modify_order(self._stop_order_id(p.id), trigger_price=a.price)
                except Exception as e:  # noqa: BLE001
                    LOG_LC.warning("stop modify failed %s: %s", p.symbol, e)
                self.store.event("MOVE_STOP", f"{p.symbol} stop -> {a.price:.2f} ({a.reason})",
                                 symbol=p.symbol, trade_id=p.id)
            else:
                qty = min(a.qty, p.qty)
                if qty <= 0:
                    continue
                gross = (a.price - p.avg_entry) * qty
                fee = transaction_costs(a.price, qty, Txn.SELL, self.cfg)["total"] if costs else 0.0
                realised += gross - fee
                p.qty -= qty
                if a.kind == "SCALE_OUT":
                    p.scaled = True
                    p.state = TState.SCALED_OUT
                if p.qty == 0:
                    p.state = TState.CLOSED
                self.store.event("EXIT", f"{p.symbol} {a.kind} {qty}@{a.price:.2f} ({a.reason})",
                                 symbol=p.symbol, trade_id=p.id,
                                 payload={"pnl": gross - fee, "r": (a.price - p.avg_entry) / p.r})
                if self.notifier:
                    emoji = "🎯" if a.reason in ("T1", "T2") else "🛑"
                    self.notifier.send(
                        f"{emoji} *{a.kind}* {p.symbol} {qty} @ ₹{a.price:,.2f} ({a.reason})\n"
                        f"P&L ₹{gross - fee:,.0f} · {(a.price - p.avg_entry) / p.r:+.2f}R")
        if p.qty == 0:
            self.positions.pop(p.id, None)
            self._persist_position(p, exit_reason=acts[-1].reason if acts else "",
                                   exit_date=str(now_ist().date()),
                                   avg_exit=acts[-1].price if acts else 0,
                                   realised_pnl=round(realised, 2))
        else:
            self._persist_position(p)
        return realised

    def _stop_order_id(self, trade_id: str) -> str:
        df = self.store.query(
            "SELECT broker_order_id FROM orders WHERE trade_id=? AND purpose='STOP' "
            "ORDER BY created_at DESC LIMIT 1", [trade_id])
        return str(df.iloc[0]["broker_order_id"]) if len(df) else ""

    def manage_all(self, bars: Dict[str, Dict[str, float]], atrs: Dict[str, float],
                   regime_state: str = "NEUTRAL") -> float:
        realised = 0.0
        LOG_LC.info("managing %d open positions · regime %s", len(self.positions), regime_state)
        for p in list(self.positions.values()):
            bar = bars.get(p.symbol)
            if not bar:
                LOG_LC.warning("no bar for %s — position NOT managed today", p.symbol)
                continue
            p.bars_held += 1
            p.high_water = max(p.high_water, bar["high"])
            p.mfe_r = max(p.mfe_r, (bar["high"] - p.avg_entry) / p.r)
            p.mae_r = min(p.mae_r, (bar["low"] - p.avg_entry) / p.r)
            acts = exit_decisions(p, bar, atrs.get(p.symbol, np.nan), self.cfg, regime_state)
            LOG_LC.info("%s bar O%.2f H%.2f L%.2f C%.2f · %+.2fR · stop %.2f · day %d → %s", p.symbol,
                        bar["open"], bar["high"], bar["low"], bar["close"], p.unrealised_r(bar["close"]),
                        p.cur_stop, p.bars_held, ", ".join(f"{a.kind}:{a.reason}" for a in acts) or "hold")
            if acts:
                realised += self.apply_actions(p, acts)
        LOG_LC.info("management done · realised ₹%s · %d still open", f"{realised:,.0f}",
                    len(self.positions))
        return realised

07:49:11 | INFO    | engine.cell       | ▶ START  [16b] Trade lifecycle — the engine
07:49:11 | INFO    | engine.cell       | ✔ DONE   [16b] Trade lifecycle — the engine in 0.0s


## 17 · Backtester

Event-driven over daily bars, portfolio-level, with the same `exit_decisions` and `size_position`
functions the live engine uses. What it models that naive vectorised backtests skip:

- **Signal at close *t*, order worked on *t+1*.** No same-bar entries.
- **Unfilled breakouts expire.** If the trigger is never touched the next day, the idea is dead.
- **Gap-away rejection.** If tomorrow opens more than `max_entry_gap_pct` above the trigger, the
  trade is skipped rather than filled at a fantasy price.
- **Capital and slot contention.** Only `max_positions` at a time; the highest-scoring candidates
  win the slots, exactly as in production.
- **Full costs and size-aware slippage** on both legs.

What it still does *not* model, and you must discount for: survivorship (the universe is today's
listed names), corporate actions, circuit-limit days where you simply cannot exit, and the
possibility that a small cap you own gets moved to ASM/T2T overnight.

In [114]:
# [17] Backtester
@dataclass
class BTFill:
    date: str
    symbol: str
    side: str
    qty: int
    price: float
    fees: float
    reason: str


class Backtester:
    def __init__(self, cfg: Config, panel: Dict[str, pd.DataFrame], universe: pd.DataFrame,
                 bench_feat: pd.DataFrame):
        if not panel:
            raise ValueError("Backtester: the panel is empty")
        self.cfg, self.panel = cfg, panel
        self.bench = bench_feat if bench_feat is not None else pd.DataFrame()
        if self.bench.empty:
            LOG_BT.warning("Backtester: no benchmark — the regime will be NEUTRAL on every day")
        self.meta = universe.set_index("symbol")[["name", "sector"]].to_dict("index")
        self.signals: Dict[str, pd.DataFrame] = {}
        t0 = time.time()
        for s, f in panel.items():
            self.signals[s] = run_strategies(f, cfg)
        n_sig = defaultdict(int)
        for sig in self.signals.values():
            for k, v in sig.loc[sig["signal"], "strategy"].value_counts().items():
                n_sig[k] += int(v)
        LOG_BT.info("Backtester ready: %d symbols · signals precomputed in %.1fs · total signal-days "
                    "by strategy %s", len(panel), time.time() - t0, dict(n_sig))

    # ---------- helpers ----------
    def _candidates_on(self, d: pd.Timestamp) -> pd.DataFrame:
        rows = []
        for s, sig in self.signals.items():
            if d not in sig.index:
                continue
            row = sig.loc[d]
            if not bool(row["signal"]):
                continue
            f = self.panel[s].loc[d]
            if not np.isfinite(f.get("atr", np.nan)) or not np.isfinite(f.get("ema200", np.nan)):
                continue
            r = {"symbol": s, "date": d, "strategy": row["strategy"], "entry": float(row["entry"]),
                 "stop": float(row["stop"]), "n_confirm": int(row["n_confirm"])}
            r.update({c: (bool(f[c]) if isinstance(f[c], (bool, np.bool_)) else float(f[c]))
                      for c in FEATURE_COLS if c in f})
            r.update(self.meta.get(s, {"name": s, "sector": "UNKNOWN"}))
            rows.append(r)
        return pd.DataFrame(rows)

    def _breadth(self, d: pd.Timestamp) -> float:
        vals = [float(f.loc[d, "close"] > f.loc[d, "ema50"])
                for f in self.panel.values()
                if d in f.index and np.isfinite(f.loc[d, "ema50"])]
        return 100 * float(np.mean(vals)) if vals else 50.0

    # ---------- main loop ----------
    def run(self, start: str, end: str, verbose: bool = True) -> Dict[str, Any]:
        cfg = self.cfg
        dates = sorted(set().union(*[set(f.index) for f in self.panel.values()]))
        dates = [d for d in dates if pd.Timestamp(start) <= d <= pd.Timestamp(end)]
        if not dates:
            raise ValueError(f"no bars in the requested window {start} → {end}")
        t_run = time.time()
        progress_every = max(len(dates) // 8, 20)
        if verbose:
            LOG_BT.info("backtest %s → %s: %d sessions · equity ₹%s · %d symbols",
                        pd.Timestamp(dates[0]).date(), pd.Timestamp(dates[-1]).date(), len(dates),
                        f"{cfg.equity:,.0f}", len(self.panel))

        cash = cfg.equity
        book: Dict[str, Position] = {}
        pending: List[Dict[str, Any]] = []
        fills: List[BTFill] = []
        trades: List[Dict[str, Any]] = []
        entry_fee_per_share: Dict[str, float] = {}
        curve: List[Dict[str, Any]] = []
        peak_equity = cfg.equity
        halted = False

        for i_day, d in enumerate(dates, 1):
            bars = {s: self.panel[s].loc[d] for s in self.panel if d in self.panel[s].index}

            # ---- 1. manage open positions on today's bar ----
            breadth = self._breadth(d)
            bf = self.bench.loc[:d] if not self.bench.empty else self.bench
            regime = assess_regime(bf, breadth, cfg, verbose=False) if len(bf) > 200 else \
                Regime("NEUTRAL", breadth, True, True, 0.0, 0.65, cfg.max_new_per_day)

            for p in list(book.values()):
                b = bars.get(p.symbol)
                if b is None:
                    continue
                bar = {k: float(b[k]) for k in ("open", "high", "low", "close")}
                bar["lower_circuit_locked"] = bool(b.get("lower_circuit_locked", False))
                p.bars_held += 1
                p.high_water = max(p.high_water, bar["high"])
                p.mfe_r = max(p.mfe_r, (bar["high"] - p.avg_entry) / p.r)
                p.mae_r = min(p.mae_r, (bar["low"] - p.avg_entry) / p.r)
                for a in exit_decisions(p, bar, float(b["atr"]), cfg, regime.state):
                    if a.kind == "MOVE_STOP":
                        p.cur_stop = a.price
                        p.be_moved = p.be_moved or a.reason.startswith("BREAKEVEN")
                        continue
                    qty = min(a.qty, p.qty)
                    if qty <= 0:
                        continue
                    px = a.price * (1 - slippage_bps(qty, float(b["adv20"]), cfg) / 10_000)
                    fee = transaction_costs(px, qty, Txn.SELL, cfg)["total"]
                    cash += px * qty - fee
                    p.qty -= qty
                    if a.kind == "SCALE_OUT":
                        p.scaled = True
                    fills.append(BTFill(str(d.date()), p.symbol, "SELL", qty, px, fee, a.reason))
                    trades.append({"exit_date": str(d.date()), "symbol": p.symbol,
                                   "strategy": p.strategy, "entry_date": p.entry_date,
                                   "entry": p.avg_entry, "exit": px, "qty": qty,
                                   "pnl": (px - p.avg_entry) * qty - fee - entry_fee_per_share.get(p.id, 0.0) * qty,
                                   "r": ((px - p.avg_entry) * qty - fee - entry_fee_per_share.get(p.id, 0.0) * qty) / max(p.r * qty, 1e-9),
                                   "reason": a.reason, "bars": p.bars_held,
                                   "mfe_r": p.mfe_r, "mae_r": p.mae_r,
                                   "sector": p.sector, "fees": fee + entry_fee_per_share.get(p.id, 0.0) * qty})
                    if p.qty == 0:
                        book.pop(p.id, None)

            # ---- 2. work yesterday's pending entries against today's bar ----
            for pe in pending:
                b = bars.get(pe["symbol"])
                if b is None:
                    continue
                o, h, l = float(b["open"]), float(b["high"]), float(b["low"])
                trg = pe["trigger"]
                px = None
                if pe["type"] == "BUY_STOP":
                    if o > trg * (1 + cfg.max_entry_gap_pct / 100):
                        pass                                   # gapped away — stand down
                    elif h >= trg:
                        px = max(trg, o)
                else:                                          # LIMIT
                    if l <= trg:
                        px = min(trg, o)
                if px is None:
                    continue                                   # single-day validity
                px *= (1 + slippage_bps(pe["qty"], float(b["adv20"]), cfg) / 10_000)
                qty = pe["qty"]
                fee = transaction_costs(px, qty, Txn.BUY, cfg)["total"]
                if px * qty + fee > cash:
                    qty = int((cash - fee) // px)
                if qty <= 0 or len(book) >= cfg.max_positions:
                    continue
                fee = transaction_costs(px, qty, Txn.BUY, cfg)["total"]
                cash -= px * qty + fee
                r = max(px - pe["stop"], 0.01)
                tid = f"BT-{uuid.uuid4().hex[:8]}"
                book[tid] = Position(id=tid, symbol=pe["symbol"], strategy=pe["strategy"],
                                     qty=qty, init_qty=qty, avg_entry=px, init_stop=pe["stop"],
                                     cur_stop=pe["stop"], t1=px + cfg.t1_r_multiple * r,
                                     t2=px + cfg.t2_r_multiple * r, r=r,
                                     entry_date=str(d.date()), high_water=px,
                                     sector=pe["sector"])
                entry_fee_per_share[tid] = fee / max(qty, 1)
                fills.append(BTFill(str(d.date()), pe["symbol"], "BUY", qty, px, fee, "ENTRY"))
            pending = []            # entry orders are day-only: untriggered ideas are dropped

            # ---- 3. equity mark & kill switches ----
            pos_val = sum(p.qty * float(bars[p.symbol]["close"])
                          for p in book.values() if p.symbol in bars)
            equity = cash + pos_val
            peak_equity = max(peak_equity, equity)
            dd = equity / peak_equity - 1
            if dd * 100 <= -cfg.max_drawdown_halt_pct and not halted and verbose:
                LOG_BT.warning("%s: drawdown %.1f%% breached the %.1f%% halt — no new entries",
                               pd.Timestamp(d).date(), dd * 100, cfg.max_drawdown_halt_pct)
            halted = dd * 100 <= -cfg.max_drawdown_halt_pct
            open_risk = sum(max(p.avg_entry - p.cur_stop, 0) * p.qty for p in book.values())
            curve.append({"date": d, "equity": equity, "cash": cash, "positions": len(book),
                          "open_risk_pct": open_risk / max(equity, 1) * 100,
                          "regime": regime.state, "breadth": breadth, "dd": dd * 100})
            if verbose and (i_day % progress_every == 0 or i_day == len(dates)):
                LOG_BT.info("  %s (%d/%d) equity ₹%s · dd %.1f%% · %d open · regime %s · breadth %.0f%% "
                            "· exits so far %d", pd.Timestamp(d).date(), i_day, len(dates),
                            f"{equity:,.0f}", dd * 100, len(book), regime.state, breadth, len(trades))

            # ---- 4. scan tonight's close for tomorrow's orders ----
            slots = cfg.max_positions - len(book)
            if halted or slots <= 0 or regime.max_new <= 0:
                continue
            cand = apply_hard_filters(self._candidates_on(d), cfg, log_gates=False)
            if cand.empty:
                continue
            scored = score_candidates(cand, cfg)
            scored = scored[scored["score"] >= cfg.min_score]
            held = {p.symbol for p in book.values()}
            sec_count = defaultdict(int)
            for p in book.values():
                sec_count[p.sector] += 1

            taken = 0
            for _, row in scored.iterrows():
                if taken >= min(regime.max_new, slots):
                    break
                if row["symbol"] in held or any(x["symbol"] == row["symbol"] for x in pending):
                    continue
                if sec_count[row.get("sector", "UNKNOWN")] >= cfg.max_sector_positions:
                    continue
                sz = size_position(float(row["entry"]), float(row["stop"]), equity, cash,
                                   float(row.get("adv20", 0)), cfg, regime.exposure_mult, open_risk)
                if not sz.ok:
                    continue
                pending.append({"symbol": row["symbol"], "trigger": float(row["entry"]),
                                "stop": float(row["stop"]), "qty": sz.qty,
                                "strategy": row["strategy"], "sector": row.get("sector", "UNKNOWN"),
                                "type": "LIMIT" if row["strategy"] == "MEAN_REVERT" else "BUY_STOP",
                                "score": float(row["score"])})
                open_risk += sz.risk_amount
                sec_count[row.get("sector", "UNKNOWN")] += 1
                taken += 1

        curve_df = pd.DataFrame(curve).set_index("date")
        trades_df = pd.DataFrame(trades)
        if book and verbose:
            LOG_BT.info("%d positions still open at the end (marked to market, not closed): %s",
                        len(book), [p.symbol for p in book.values()])
        if verbose:
            LOG_BT.info("backtest done in %.1fs: %d fills, %d exits, final equity ₹%s",
                        time.time() - t_run, len(fills), len(trades_df),
                        f"{curve_df['equity'].iloc[-1]:,.0f}")
        return {"curve": curve_df, "trades": trades_df,
                "fills": pd.DataFrame([asdict(f) for f in fills]),
                "metrics": performance_metrics(curve_df, trades_df, self.cfg)}

07:49:11 | INFO    | engine.cell       | ▶ START  [17] Backtester
07:49:11 | INFO    | engine.cell       | ✔ DONE   [17] Backtester in 0.0s


### Performance metrics

Expectancy in **R** is the number that matters most here. A small-cap swing system living on
40% hit rate and +0.35R expectancy is perfectly healthy; the same system with expectancy
below ~0.15R will be eaten alive by costs and the occasional untradeable gap.

In [115]:
# [17b] Performance metrics & purged walk-forward
def performance_metrics(curve: pd.DataFrame, trades: pd.DataFrame, cfg: Config) -> Dict[str, Any]:
    if curve.empty:
        return {}
    eq = curve["equity"].astype(float)
    rets = eq.pct_change().dropna()
    years = max((eq.index[-1] - eq.index[0]).days / 365.25, 1e-9)
    cagr = (eq.iloc[-1] / eq.iloc[0]) ** (1 / years) - 1 if eq.iloc[0] > 0 else 0
    vol = rets.std() * math.sqrt(252)
    downside = rets[rets < 0].std() * math.sqrt(252)
    dd = eq / eq.cummax() - 1
    m: Dict[str, Any] = {
        "start": str(pd.Timestamp(eq.index[0]).date()), "end": str(pd.Timestamp(eq.index[-1]).date()),
        "years": round(years, 2), "final_equity": round(float(eq.iloc[-1]), 0),
        "total_return_pct": round(float(eq.iloc[-1] / eq.iloc[0] - 1) * 100, 2),
        "cagr_pct": round(cagr * 100, 2), "vol_pct": round(float(vol) * 100, 2) if np.isfinite(vol) else 0,
        "sharpe": round(float(rets.mean() / rets.std() * math.sqrt(252)), 2) if len(rets) and rets.std() else 0,
        "sortino": round(float(rets.mean() * 252 / downside), 2) if downside and downside > 0 else 0,
        "max_dd_pct": round(float(dd.min()) * 100, 2),
        "calmar": round(float(cagr / abs(dd.min())), 2) if dd.min() < 0 else 0,
        "avg_positions": round(float(curve["positions"].mean()), 2),
        "time_in_market_pct": round(float((curve["positions"] > 0).mean()) * 100, 1),
        "pct_days_risk_off": round(float((curve["regime"] == "RISK_OFF").mean()) * 100, 1),
    }
    if trades is not None and not trades.empty:
        t = trades.copy()
        w, l = t[t["pnl"] > 0], t[t["pnl"] <= 0]
        capital_days = float((t["bars"].clip(lower=1) * (t["entry"] * t["qty"])).sum())
        net_pnl = float(t["pnl"].sum())
        m.update({
            "n_exits": int(len(t)),
            "hit_rate_pct": round(len(w) / len(t) * 100, 1),
            "avg_win_r": round(float(w["r"].mean()), 2) if len(w) else 0,
            "avg_loss_r": round(float(l["r"].mean()), 2) if len(l) else 0,
            "expectancy_r": round(float(t["r"].mean()), 3),
            "profit_factor": round(float(w["pnl"].sum() / abs(l["pnl"].sum())), 2)
                             if len(l) and l["pnl"].sum() != 0 else float("inf"),
            "avg_bars_held": round(float(t["bars"].mean()), 1),
            "median_bars_held": round(float(t["bars"].median()), 1),
            "return_per_capital_day_bps": round(net_pnl / max(capital_days, 1) * 10_000, 3),
            "total_fees": round(float(t["fees"].sum()), 0),
            "best_r": round(float(t["r"].max()), 2),
            "worst_r": round(float(t["r"].min()), 2),
        })
    return m


def by_bucket(trades: pd.DataFrame, col: str) -> pd.DataFrame:
    if trades is None or trades.empty or col not in trades.columns:
        return pd.DataFrame()
    g = trades.groupby(col, dropna=False).agg(
        n=("r", "size"), expectancy_r=("r", "mean"),
        hit_rate=("pnl", lambda s: (s > 0).mean() * 100),
        total_pnl=("pnl", "sum"), avg_bars=("bars", "mean"))
    return g.sort_values("total_pnl", ascending=False).round(2)


def _all_dates(panel: Dict[str, pd.DataFrame]) -> List[pd.Timestamp]:
    return sorted(pd.Timestamp(x) for x in set().union(*[set(f.index) for f in panel.values() if len(f)]))


def _per_symbol_first_eligible(panel: Dict[str, pd.DataFrame], warmup: int) -> Dict[str, pd.Timestamp]:
    out = {}
    for s, f in panel.items():
        if len(f) > warmup:
            out[s] = pd.Timestamp(f.index[warmup])
    return out


def _latest_bar(panel: Dict[str, pd.DataFrame]) -> pd.Timestamp:
    ds = _all_dates(panel)
    if not ds:
        raise ValueError("panel has no dates")
    return ds[-1]


def walk_forward(cfg: Config, panel: Dict[str, pd.DataFrame], universe: pd.DataFrame,
                 bench_feat: pd.DataFrame, folds: Optional[int] = None,
                 variant: str = "ANCHORED", exclude_final_holdout: bool = True) -> pd.DataFrame:
    """Purged/embargoed OOS slices with per-symbol warm-up.

    Rule strategies do not 'train' in a fold, but the boundaries are still explicit so the same
    folds can be reused by parameter selection and meta-labeling. A recent listing becomes eligible
    on its own 251st bar; it cannot delay every other name.
    """
    folds = int(folds or cfg.wf_folds)
    variant = variant.upper()
    assert variant in ("ANCHORED", "ROLLING")
    dates = _all_dates(panel)
    elig = _per_symbol_first_eligible(panel, cfg.warmup_bars)
    if not dates or not elig:
        return pd.DataFrame()
    first_ok = min(elig.values())  # FIX: never max across symbols
    latest = dates[-1]
    requested_end = pd.Timestamp(cfg.bt_end) if cfg.bt_end else latest
    end = min(latest, requested_end)
    if exclude_final_holdout:
        holdout_start = end - pd.DateOffset(months=cfg.holdout_months)
        research_end_candidates = [d for d in dates if d < holdout_start]
        if not research_end_candidates:
            raise ValueError("not enough history before locked holdout")
        end = research_end_candidates[-1]

    start = max(pd.Timestamp(cfg.bt_start), first_ok)
    window = [d for d in dates if start <= d <= end]
    need = folds * cfg.wf_min_test_bars
    if len(window) < need:
        folds = max(2, len(window) // max(cfg.wf_min_test_bars, 1))
    if folds < 2:
        LOG_BT.warning("walk-forward: insufficient sessions (%d)", len(window))
        return pd.DataFrame()

    # First third acts as initial design/training span; remaining dates become disjoint OOS folds.
    split0 = max(cfg.warmup_bars, len(window) // 3)
    oos_dates = window[split0:]
    chunks = [list(x) for x in np.array_split(np.array(oos_dates, dtype="datetime64[ns]"), folds) if len(x)]
    bt = Backtester(cfg, panel, universe, bench_feat)
    rows = []
    for i, raw in enumerate(chunks, 1):
        test = [pd.Timestamp(x) for x in raw]
        test_start, test_end = test[0], test[-1]
        pos_start = window.index(test_start)
        train_end_pos = max(0, pos_start - cfg.wf_embargo_bars - 1)
        train_end = window[train_end_pos]
        if variant == "ANCHORED":
            train_start = window[0]
        else:
            roll = max(cfg.warmup_bars * 2, len(window) // 2)
            train_start = window[max(0, train_end_pos - roll)]
        # Purge: training ends before embargo; test never overlaps it.
        res = bt.run(str(test_start.date()), str(test_end.date()), verbose=False)
        m = res["metrics"]
        rows.append({
            "variant": variant, "fold": i,
            "train_start": str(train_start.date()), "train_end": str(train_end.date()),
            "embargo_bars": cfg.wf_embargo_bars,
            "test_start": str(test_start.date()), "test_end": str(test_end.date()),
            "eligible_symbols_at_test_start": int(sum(d <= test_start for d in elig.values())),
            **{k: m.get(k) for k in (
                "cagr_pct", "max_dd_pct", "sharpe", "expectancy_r", "profit_factor",
                "n_exits", "avg_bars_held", "median_bars_held", "return_per_capital_day_bps")}
        })
    return pd.DataFrame(rows)


def locked_holdout(cfg: Config, panel: Dict[str, pd.DataFrame], universe: pd.DataFrame,
                   bench_feat: pd.DataFrame, unlock: bool = False) -> Dict[str, Any]:
    """Evaluate the final ~12 months only when explicitly unlocked after design is frozen."""
    if not unlock:
        return {"locked": True, "message": "Final holdout not evaluated. Set unlock=True only after freezing design choices."}
    latest = _latest_bar(panel)
    start = latest - pd.DateOffset(months=cfg.holdout_months)
    bt = Backtester(cfg, panel, universe, bench_feat)
    res = bt.run(str(start.date()), str(latest.date()), verbose=False)
    return {"locked": False, **res}


07:49:11 | INFO    | engine.cell       | ▶ START  [17b] Performance metrics & purged walk-forward
07:49:11 | INFO    | engine.cell       | ✔ DONE   [17b] Performance metrics & purged walk-forward in 0.0s


## 18 · Optional layer A — Gemini as a qualitative overlay

**What the LLM is allowed to do:** read the quantitative picture, apply a bounded score adjustment
(±`gemini_max_adjust`, default 10 points), raise named risk flags, and write the human-readable
thesis and daily note.

**What it is not allowed to do:** produce a price, a stop, a quantity, or a trade that the quant
core did not already generate. It cannot promote a rejected name. It cannot widen a stop.

That boundary is the whole design. An LLM is excellent at "this name has run 60% in three weeks
and the volume pattern looks like a distribution top" and catastrophic at "the stop should be
₹247.30". Every failure mode — timeouts, malformed JSON, hallucinated symbols — degrades to
*neutral*, never to a trade.

Wire in a news/announcement feed (corporate actions, results calendar, exchange filings) via the
`context` argument and the flags get considerably sharper.

In [116]:
# [18] Optional layer A — Gemini qualitative overlay
GEMINI_SYSTEM = """You are a risk reviewer on a systematic Indian small-cap swing desk.
You do NOT generate trade ideas. You review ideas the quant model has already produced and flag
qualitative risks the model cannot see.

For each candidate return an object with:
  symbol: str (echo exactly)
  adjust: number in [-10, 10]  -- score adjustment; be conservative, use 0 when unsure
  flags: array of short strings from
    [EXTENDED, THIN_LIQUIDITY, EARNINGS_RISK, NEWS_DRIVEN_SPIKE, LOW_QUALITY_BASE,
     SECTOR_CROWDING, CIRCUIT_RISK, PUMP_PATTERN, NONE]
  rationale: one sentence, under 25 words, specific and non-generic.

Rules:
- Penalise names that are already far extended from the 20-EMA, have deep sloppy bases,
  or show a vertical volume spike typical of an operator-driven move.
- Penalise crowding: if many candidates share a sector, flag SECTOR_CROWDING on the weaker ones.
- Never invent facts you were not given. Never output prices, quantities or stops.
- Return ONLY a JSON array. No markdown fences, no prose.
"""


class GeminiAnalyst:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.client = None
        self.mode = None
        if not (cfg.use_gemini and cfg.gemini_api_key):
            LOG_LLM.info("Gemini off (use_gemini=%s, key %s)", cfg.use_gemini,
                         "set" if cfg.gemini_api_key else "missing")
            return
        try:
            from google import genai  # type: ignore
            self.client = genai.Client(api_key=cfg.gemini_api_key)
            self.mode = "genai"
        except Exception:  # noqa: BLE001
            try:
                import google.generativeai as ggen  # type: ignore
                ggen.configure(api_key=cfg.gemini_api_key)
                self.client = ggen.GenerativeModel(cfg.gemini_model)
                self.mode = "legacy"
            except Exception as e:  # noqa: BLE001
                LOG_LLM.warning("Gemini unavailable, running without it: %s", e)

    @property
    def enabled(self) -> bool:
        return self.client is not None

    def _generate(self, prompt: str, system: str = "", temperature: float = 0.2) -> str:
        if not self.enabled:
            return ""
        t0 = time.time()
        LOG_LLM.info("Gemini request · model=%s · prompt %d chars", self.cfg.gemini_model, len(prompt))
        try:
            if self.mode == "genai":
                r = self.client.models.generate_content(
                    model=self.cfg.gemini_model,
                    contents=prompt,
                    config={"system_instruction": system, "temperature": temperature,
                            "response_mime_type": "application/json" if system else "text/plain"},
                )
                out = getattr(r, "text", "") or ""
            else:
                r = self.client.generate_content(f"{system}\n\n{prompt}")
                out = getattr(r, "text", "") or ""
            LOG_LLM.info("Gemini reply · %d chars in %.1fs", len(out), time.time() - t0)
            return out
        except Exception as e:  # noqa: BLE001
            LOG_LLM.warning("Gemini call failed after %.1fs (continuing without): %s", time.time() - t0, e)
            return ""

    @staticmethod
    def _parse_json(text: str) -> Any:
        if not text:
            return None
        t = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            return json.loads(t)
        except Exception:  # noqa: BLE001
            i, j = t.find("["), t.rfind("]")
            if 0 <= i < j:
                try:
                    return json.loads(t[i:j + 1])
                except Exception:  # noqa: BLE001
                    return None
            return None

    def review(self, cand: pd.DataFrame, regime: Regime,
               context: Optional[Dict[str, str]] = None, top_n: int = 12) -> pd.DataFrame:
        """Returns `cand` with `llm_adjust`, `llm_flags`, `llm_note`, and an adjusted `score`."""
        d = cand.copy()
        d["llm_adjust"] = 0.0
        d["llm_flags"] = ""
        d["llm_note"] = ""
        if not self.enabled or d.empty:
            return d

        cols = ["symbol", "sector", "strategy", "score", "ret_21", "ret_63", "rel_63",
                "dist_52w_high", "ext_atr", "atr_pct", "vol_ratio", "base_depth",
                "turnover_med20_cr", "adx", "rsi14"]
        sub = d.head(top_n)[[c for c in cols if c in d.columns]].round(2)
        payload = {
            "regime": regime.to_dict(),
            "sector_counts": sub["sector"].value_counts().to_dict() if "sector" in sub else {},
            "candidates": sub.to_dict("records"),
            "context": context or {},
        }
        raw = self._generate(json.dumps(payload, default=str), GEMINI_SYSTEM)
        parsed = self._parse_json(raw)
        if not isinstance(parsed, list):
            LOG_LLM.info("Gemini returned nothing usable — proceeding on the quant score alone "
                         "(raw head: %r)", (raw or "")[:160])
            return d

        known = set(d["symbol"])
        for item in parsed:
            try:
                sym = str(item.get("symbol", ""))
                if sym not in known:
                    continue                                   # never trust an invented symbol
                adj = float(item.get("adjust", 0))
                adj = float(np.clip(adj, -self.cfg.gemini_max_adjust, self.cfg.gemini_max_adjust))
                m = d["symbol"] == sym
                d.loc[m, "llm_adjust"] = adj
                d.loc[m, "llm_flags"] = ",".join(map(str, item.get("flags", [])))[:120]
                d.loc[m, "llm_note"] = str(item.get("rationale", ""))[:200]
            except Exception:  # noqa: BLE001
                continue
        d["score_quant"] = d["score"]
        d["score"] = np.clip(d["score"] + d["llm_adjust"], 0, 100)
        adj = d[d["llm_adjust"] != 0]
        LOG_LLM.info("Gemini reviewed %d items · adjusted %d: %s", len(parsed), len(adj),
                     ", ".join(f"{r.symbol}{r.llm_adjust:+.0f}[{r.llm_flags}]" for r in adj.head(8).itertuples()))
        return d.sort_values("score", ascending=False)

    def daily_note(self, regime: Regime, picks: pd.DataFrame, positions: List[Position],
                   metrics: Optional[Dict[str, Any]] = None) -> str:
        if not self.enabled:
            return ""
        payload = {
            "regime": regime.to_dict(),
            "new_ideas": picks[["symbol", "strategy", "score", "sector"]].head(6).to_dict("records")
                         if not picks.empty else [],
            "open_positions": [{"symbol": p.symbol, "bars": p.bars_held,
                                "r": round(p.unrealised_r(p.high_water), 2)} for p in positions],
            "metrics": metrics or {},
        }
        prompt = ("Write a 4-sentence pre-market note for the trader who runs this book. "
                  "Be concrete about regime and book risk. No hype, no disclaimers, no advice to "
                  "buy or sell anything not listed.\n\n" + json.dumps(payload, default=str))
        return self._generate(prompt, "", temperature=0.4).strip()


GEMINI = GeminiAnalyst(CFG)
LOG_LLM.info("Gemini layer: %s", f"ENABLED ({GEMINI.mode}, {CFG.gemini_model})" if GEMINI.enabled else "off")

07:49:11 | INFO    | engine.cell       | ▶ START  [18] Optional layer A — Gemini qualitative overlay
07:49:11 | INFO    | engine.llm        | Gemini layer: ENABLED (genai, gemini-2.5-flash)
07:49:11 | INFO    | engine.cell       | ✔ DONE   [18] Optional layer A — Gemini qualitative overlay in 0.4s


## 19 · Optional layer B — Telegram control surface

Raw Bot API over `requests`; no extra dependency. Three jobs:

1. **Push** — the EOD digest, fill confirmations, stop moves, exits.
2. **Pull** — `/positions`, `/pnl`, `/screen`, `/risk` on demand from your phone.
3. **Approve** — in `LIVE` mode with `require_approval=True`, every entry waits for a tap on an
   inline button before an order is sent. This is the single most valuable safety feature here:
   it keeps a human in the loop without requiring the human to be at a desk at 9:15.

Setup: talk to `@BotFather` → `/newbot` → copy the token → message your bot once → read your
`chat_id` from `https://api.telegram.org/bot<TOKEN>/getUpdates`.

In [117]:
# [19] Optional layer B — Telegram control surface
class TelegramNotifier:
    API = "https://api.telegram.org/bot{token}/{method}"

    def __init__(self, cfg: Config, store: Store):
        self.cfg, self.store = cfg, store
        self.enabled = bool(cfg.use_telegram and cfg.telegram_token and cfg.telegram_chat_id
                            and requests is not None)
        self.approvals: Dict[str, str] = {}          # trade_id -> APPROVED | REJECTED
        if self.enabled:
            LOG_TG.info("Telegram enabled → chat %s", cfg.telegram_chat_id)
        else:
            LOG_TG.info("Telegram off (use_telegram=%s, token %s, chat %s) — messages go to this log",
                        cfg.use_telegram, "set" if cfg.telegram_token else "missing",
                        "set" if cfg.telegram_chat_id else "missing")

    def _call(self, method: str, **payload) -> Optional[dict]:
        if not self.enabled:
            return None
        try:
            r = requests.post(self.API.format(token=self.cfg.telegram_token, method=method),
                              json=payload, timeout=20)
            j = r.json()
            if not j.get("ok", False):
                LOG_TG.warning("telegram %s rejected: %s", method, str(j)[:200])
            elif method != "getUpdates":
                LOG_TG.debug("telegram %s ok", method)
            return j
        except Exception as e:  # noqa: BLE001
            LOG_TG.warning("telegram %s failed: %s", method, e)
            return None

    def send(self, text: str, markup: Optional[dict] = None) -> None:
        if not self.enabled:
            LOG_TG.info("[telegram-off] %s", text.replace("\n", " | ")[:220])
            return
        LOG_TG.info("→ telegram: %s", text.replace("\n", " | ")[:160])
        payload = {"chat_id": self.cfg.telegram_chat_id, "text": text,
                   "parse_mode": "Markdown", "disable_web_page_preview": True}
        if markup:
            payload["reply_markup"] = markup
        self._call("sendMessage", **payload)

    # ---------- composed messages ----------
    def digest(self, regime: Regime, plans: List[TradePlan], positions: List[Position],
               equity: float, note: str = "") -> str:
        icon = {"RISK_ON": "🟢", "NEUTRAL": "🟡", "RISK_OFF": "🔴"}[regime.state]
        lines = [f"*EOD SCAN · {now_ist():%d %b %Y}*",
                 f"{icon} Regime *{regime.state}* — {regime.note}",
                 f"Book ₹{equity:,.0f} · {len(positions)} open · {len(plans)} new ideas", ""]
        if note:
            lines += [f"_{note}_", ""]
        if plans:
            lines.append("*🎯 Tomorrow's orders*")
            lines += [p.telegram() + "\n" for p in plans]
        else:
            lines.append("_No setups cleared the filters. Cash is a position._")
        if positions:
            lines.append("*📊 Open book*")
            for p in positions:
                lines.append(f"• {p.symbol} {p.qty} @ ₹{p.avg_entry:,.2f} · SL ₹{p.cur_stop:,.2f} "
                             f"· {p.bars_held}d · MFE {p.mfe_r:+.1f}R")
        msg = "\n".join(lines)
        self.send(msg)
        return msg

    def ask_approval(self, plan: TradePlan) -> None:
        markup = {"inline_keyboard": [[
            {"text": "✅ Approve", "callback_data": f"ok:{plan.id}"},
            {"text": "❌ Skip", "callback_data": f"no:{plan.id}"},
        ]]}
        self.send("*APPROVAL NEEDED*\n" + plan.telegram(), markup)

    # ---------- command loop ----------
    def poll(self, handlers: Dict[str, Callable[[List[str]], str]], seconds: int = 30) -> None:
        """Long-poll for commands and approval taps. Run in a thread during market hours."""
        if not self.enabled:
            return
        offset = self.store.get_kv("tg_offset", 0)
        deadline = time.time() + seconds
        while time.time() < deadline:
            r = self._call("getUpdates", offset=offset + 1, timeout=10) or {}
            for upd in r.get("result", []):
                offset = max(offset, upd["update_id"])
                if "callback_query" in upd:
                    cq = upd["callback_query"]
                    data = cq.get("data", "")
                    verdict, _, tid = data.partition(":")
                    self.approvals[tid] = "APPROVED" if verdict == "ok" else "REJECTED"
                    self._call("answerCallbackQuery", callback_query_id=cq["id"],
                               text=f"{self.approvals[tid].title()}")
                    self.store.event("APPROVAL", f"{tid} -> {self.approvals[tid]}", trade_id=tid)
                    continue
                text = (upd.get("message") or {}).get("text", "").strip()
                if not text.startswith("/"):
                    continue
                parts = text.split()
                cmd, args = parts[0].lstrip("/").split("@")[0].lower(), parts[1:]
                fn = handlers.get(cmd)
                LOG_TG.info("command /%s %s", cmd, args)
                self.send(fn(args) if fn else f"Unknown command `{cmd}`. Try /help")
            self.store.set_kv("tg_offset", offset)
            time.sleep(1.5)


TG = TelegramNotifier(CFG, STORE)

07:49:11 | INFO    | engine.cell       | ▶ START  [19] Optional layer B — Telegram control surface
07:49:11 | INFO    | engine.telegram   | Telegram enabled → chat 
07:49:11 | INFO    | engine.cell       | ✔ DONE   [19] Optional layer B — Telegram control surface in 0.0s


## 20 · Orchestrator

The daily clock. Everything above is a library; this is the only thing a scheduler needs to call.

| Time (IST) | Job | What happens |
|---|---|---|
| 08:45 | `premarket()` | Refresh token, reconcile broker positions against the DB, submit approved entry orders as AMO/pre-open, re-arm protective stops |
| 09:15–15:30 | `intraday(every 5 min)` | Poll order status, register fills, place the protective stop the instant an entry fills, watch the kill switches |
| 15:20 | `manage_eod()` | Apply `exit_decisions` on near-final prices: T1 scale-outs, trailing stop updates, time stops |
| 15:45 | `eod_scan()` | Full screen → rank → size → plan → Telegram digest (and approval buttons) |
| Weekly | `refresh_universe()` | Rebuild the small-cap universe, re-pull the ASM/GSM exclusions |

On a server: `cron` + `papermill`, or convert this notebook to a module and run it under
`systemd`. Keep the SQLite file on persistent storage; that file *is* your book.

In [118]:
# [20] Orchestrator
class SwingEngine:
    def __init__(self, cfg: Config, hub: DataHub, broker: BrokerAdapter, store: Store,
                 gemini: GeminiAnalyst, tg: TelegramNotifier):
        self.cfg, self.hub, self.broker, self.store = cfg, hub, broker, store
        self.gemini, self.tg = gemini, tg
        self.lc = LifecycleEngine(cfg, broker, store, notifier=tg)
        self.universe: pd.DataFrame = pd.DataFrame()
        self.panel: Dict[str, pd.DataFrame] = {}
        self.bench: Optional[pd.DataFrame] = None
        self.last_plans: List[TradePlan] = []
        self.regime: Optional[Regime] = None

    # ---------- state ----------
    @property
    def equity(self) -> float:
        """Cash plus positions marked to the last price we know about."""
        pos = list(self.lc.positions.values())
        marks: Dict[str, float] = {}
        if pos:
            try:
                marks = self.broker.ltp([p.symbol for p in pos]) or {}
            except Exception:  # noqa: BLE001
                marks = {}
        pos_val = sum(p.qty * (marks.get(p.symbol) or
                               (float(self.panel[p.symbol]["close"].iloc[-1])
                                if p.symbol in self.panel else p.avg_entry))
                      for p in pos)
        cash = getattr(self.broker, "cash", self.cfg.equity - pos_val)
        return float(cash + pos_val)

    def check_kill_switches(self) -> bool:
        eq = self.equity
        peak = float(self.store.get_kv("peak_equity", eq))
        peak = max(peak, eq)
        self.store.set_kv("peak_equity", peak)
        dd = (eq / peak - 1) * 100
        LOG_LC.info("kill-switch check | equity ₹%s · peak ₹%s · dd %.2f%% (halt at -%.1f%%)",
                    f"{eq:,.0f}", f"{peak:,.0f}", dd, self.cfg.max_drawdown_halt_pct)
        if dd <= -self.cfg.max_drawdown_halt_pct:
            self.store.set_kv("halted", True)
            self.tg.send(f"🚨 *HALTED* — drawdown {dd:.1f}% breached the "
                         f"{self.cfg.max_drawdown_halt_pct}% limit. No new entries.")
            return True
        return bool(self.store.get_kv("halted", False))

    # ---------- jobs ----------
    def refresh_universe(self, refresh_data: bool = True) -> pd.DataFrame:
        with log_step("job: refresh_universe", log):
            self.universe = build_universe(self.cfg, self.hub, self.broker, SYNTH, refresh_data)
        return self.universe

    def eod_scan(self, refresh_data: bool = True) -> List[TradePlan]:
        with log_step("job: eod_scan", log):
            return self._eod_scan(refresh_data)

    def _eod_scan(self, refresh_data: bool) -> List[TradePlan]:
        if self.universe.empty:
            log.info("eod_scan: no universe in memory — building it first")
            self.refresh_universe(refresh_data)
        self.panel, self.bench = compute_panel(self.cfg, self.hub, self.universe, refresh_data)
        res = run_screener(self.cfg, self.hub, self.universe, panel=self.panel, bench_feat=self.bench)
        self.regime = res["regime"]
        cand = res["candidates"]

        if self.gemini.enabled and not cand.empty:
            n_before = len(cand)
            cand = self.gemini.review(cand, self.regime)
            cand = cand[cand["score"] >= self.cfg.min_score]
            log.info("eod_scan: after Gemini overlay %d → %d candidates", n_before, len(cand))

        halted = self.check_kill_switches()
        plans: List[TradePlan] = []
        equity = self.equity
        cash = getattr(self.broker, "cash", equity)
        open_risk = self.lc.portfolio_open_risk()
        slots = self.cfg.max_positions - len(self.lc.positions)
        scan_date = str(now_ist().date())
        log.info("eod_scan: equity ₹%s · cash ₹%s · open risk ₹%s · free slots %d · regime %s "
                 "(max %d new) · halted=%s · %d candidates", f"{equity:,.0f}", f"{cash:,.0f}",
                 f"{open_risk:,.0f}", slots, self.regime.state, self.regime.max_new, halted, len(cand))
        if halted:
            log.warning("eod_scan: kill switch engaged — no plans will be built")
        elif slots <= 0:
            log.info("eod_scan: book full (%d/%d) — no plans", len(self.lc.positions), self.cfg.max_positions)
        elif self.regime.max_new <= 0:
            log.info("eod_scan: regime %s allows 0 new entries", self.regime.state)

        if not halted and slots > 0 and not cand.empty:
            for _, row in cand.iterrows():
                if len(plans) >= min(self.regime.max_new, slots):
                    break
                plan = build_trade_plan(row, self.cfg, equity, cash, self.regime, open_risk, scan_date)
                if plan is None:
                    continue
                if row.get("llm_note"):
                    plan.llm = {"adjust": float(row.get("llm_adjust", 0)),
                                "flags": row.get("llm_flags", ""), "note": row.get("llm_note", "")}
                    plan.thesis += f" · LLM: {row['llm_note']}"
                state = self.lc.propose(plan)
                if state in (TState.PROPOSED, TState.PENDING_APPROVAL):
                    plans.append(plan)
                    open_risk += plan.risk_amount
                    if state == TState.PENDING_APPROVAL:
                        self.tg.ask_approval(plan)

        self.last_plans = plans
        log.info("eod_scan: %d plans → %s", len(plans), [p.symbol for p in plans])
        note = self.gemini.daily_note(self.regime, cand, list(self.lc.positions.values())) \
            if self.gemini.enabled else ""
        self.tg.digest(self.regime, plans, list(self.lc.positions.values()), equity, note)
        return plans

    def premarket(self) -> None:
        log.info("job: premarket · %d plans from the last scan · %d pending entries",
                 len(self.last_plans), len(self.lc.pending))
        if self.cfg.mode == "LIVE":
            self.broker.connect()
        self.lc.expire_stale_entries()
        if self.check_kill_switches():
            log.warning("engine halted — no entries submitted")
            return
        sent = 0
        for plan in self.last_plans:
            if self.cfg.require_approval and self.cfg.mode == "LIVE":
                if self.tg.approvals.get(plan.id) != "APPROVED":
                    log.info("skipping %s — not approved (approval=%s)", plan.symbol,
                             self.tg.approvals.get(plan.id, "none"))
                    continue
            sent += int(bool(self.lc.submit_entry(plan)))
        log.info("premarket done · %d entries submitted", sent)

    def intraday(self) -> None:
        """Poll working orders; the instant an entry fills, arm its protective stop."""
        log.info("job: intraday · polling %d working entries", len(self.lc.pending))
        for tid, info in list(self.lc.pending.items()):
            try:
                det = self.broker.order_detail(info["order_id"])
            except Exception as e:  # noqa: BLE001
                log.warning("order poll failed %s: %s", tid, e)
                continue
            log.info("order %s (%s): status=%s filled=%s avg=%s", info["order_id"], info["plan"].symbol,
                     det.get("status"), det.get("filled_qty"), det.get("avg_price"))
            if det.get("status") == OrdStatus.FILLED and det.get("filled_qty", 0) > 0:
                sector = next((p.sector for p in [info["plan"]]), "UNKNOWN")
                self.lc.on_entry_fill(tid, int(det["filled_qty"]), float(det["avg_price"]), sector)
            elif det.get("status") in (OrdStatus.REJECTED, OrdStatus.CANCELLED):
                self.lc.pending.pop(tid, None)
                self.store.event("ENTRY_DEAD", f"{info['plan'].symbol} {det.get('status')}",
                                 trade_id=tid, level="WARN")

    def manage_eod(self) -> float:
        """Apply the exit rule set to today's (near-)final bars."""
        syms = [p.symbol for p in self.lc.positions.values()]
        log.info("job: manage_eod · %d open positions", len(syms))
        if not syms:
            return 0.0
        bars, atrs = {}, {}
        for s in syms:
            f = self.panel.get(s)
            if f is None or f.empty:
                log.info("manage_eod: %s not in the panel — using a live quote as today's bar", s)
                q = self.broker.quote(s)
                if q.get("ltp"):
                    bars[s] = {"open": q.get("open", q["ltp"]), "high": q.get("high", q["ltp"]),
                               "low": q.get("low", q["ltp"]), "close": q["ltp"]}
                    atrs[s] = np.nan
                continue
            last = f.iloc[-1]
            bars[s] = {k: float(last[k]) for k in ("open", "high", "low", "close")}
            atrs[s] = float(last["atr"])
        state = self.regime.state if self.regime else "NEUTRAL"
        return self.lc.manage_all(bars, atrs, state)

    # ---------- telegram command handlers ----------
    def handlers(self) -> Dict[str, Callable[[List[str]], str]]:
        def positions(_):
            ps = list(self.lc.positions.values())
            if not ps:
                return "Flat. No open positions."
            return "\n".join(f"{p.symbol} {p.qty}@₹{p.avg_entry:,.2f} SL ₹{p.cur_stop:,.2f} "
                             f"{p.bars_held}d MFE {p.mfe_r:+.1f}R" for p in ps)

        def risk(_):
            eq = self.equity
            r = self.lc.portfolio_open_risk()
            return (f"Equity ₹{eq:,.0f}\nOpen risk ₹{r:,.0f} ({r/max(eq,1)*100:.2f}%)\n"
                    f"Positions {len(self.lc.positions)}/{self.cfg.max_positions}\n"
                    f"Regime {self.regime.state if self.regime else 'n/a'}\n"
                    f"Halted: {self.store.get_kv('halted', False)}")

        def screen(_):
            plans = self.eod_scan(refresh_data=True)
            return f"Scan complete — {len(plans)} ideas." if plans else "Scan complete — nothing qualified."

        def pnl(_):
            df = self.store.query("SELECT symbol,realised_pnl,exit_reason,exit_date FROM trades "
                                  "WHERE state='CLOSED' ORDER BY exit_date DESC LIMIT 10")
            if df.empty:
                return "No closed trades yet."
            return df.to_string(index=False)

        return {
            "help": lambda _: "/screen /positions /risk /pnl /pause /resume /halt",
            "screen": screen, "positions": positions, "risk": risk, "pnl": pnl,
            "pause": lambda _: (self.store.set_kv("halted", True), "⏸ Paused — no new entries.")[1],
            "resume": lambda _: (self.store.set_kv("halted", False), "▶️ Resumed.")[1],
            "halt": lambda _: (self.store.set_kv("halted", True), "🛑 Halted.")[1],
        }

    # ---------- convenience ----------
    def run_daily_cycle(self, refresh_data: bool = True) -> Dict[str, Any]:
        """One full simulated day, in order. In production these are separate cron jobs."""
        with log_step("daily cycle (premarket → intraday → manage → scan)", log):
            self.premarket()
            self.intraday()
            realised = self.manage_eod()
            plans = self.eod_scan(refresh_data)
        return {"realised": realised, "plans": plans, "regime": self.regime}


ENGINE = SwingEngine(CFG, HUB, BROKER, STORE, GEMINI, TG)
log.info("engine ready | mode=%s | broker=%s | dry_run=%s | gemini=%s | telegram=%s | "
         "open positions=%d", CFG.mode, BROKER.name, CFG.dry_run, GEMINI.enabled, TG.enabled,
         len(ENGINE.lc.positions))

07:49:11 | INFO    | engine.cell       | ▶ START  [20] Orchestrator
07:49:11 | INFO    | engine.lifecycle  | no open positions in the db
07:49:11 | INFO    | engine            | engine ready | mode=LIVE | broker=groww | dry_run=True | gemini=True | telegram=True | open positions=0
07:49:11 | INFO    | engine.cell       | ✔ DONE   [20] Orchestrator in 0.0s


---
# Part II · v2 Research & operations extensions

These cells add the trustworthy-research harness first, then the daily operating path. Experimental edges remain disabled by default until they pass the same OOS gates as the baseline.

## [21] Data quality, deep backfill, point-in-time membership

Research mode backfills roughly nine calendar years with split/bonus-adjusted Yahoo Finance data, persists it in SQLite on Drive, detects suspicious discontinuities, and supports historical membership CSV snapshots when supplied. If snapshots are absent, the report explicitly labels the current-universe approximation as survivorship-biased.

In [119]:
# [21] Research data quality & deep backfill
def config_hash(cfg: Config) -> str:
    raw = json.dumps(asdict(cfg), sort_keys=True, default=str).encode()
    return hashlib.sha256(raw).hexdigest()[:16]

def data_quality_report(panel: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for s, f in panel.items():
        if f.empty:
            continue
        suspect = int(f.get("corp_action_suspect", pd.Series(False, index=f.index)).sum())
        locked = int(f.get("lower_circuit_locked", pd.Series(False, index=f.index)).sum())
        rows.append({"symbol":s, "bars":len(f), "start":f.index.min(), "end":f.index.max(),
                     "corp_action_suspect_days":suspect, "lower_circuit_like_days":locked,
                     "missing_ohlc":int(f[["open","high","low","close"]].isna().any(axis=1).sum())})
    return pd.DataFrame(rows).sort_values(["corp_action_suspect_days","missing_ohlc"], ascending=False)

def deep_backfill_v2(cfg: Config, hub: DataHub, symbols: Sequence[str], years: int = 9) -> pd.DataFrame:
    old = cfg.data_source
    cfg.data_source = "YFINANCE"
    rows = []
    start = now_ist().date() - timedelta(days=int(365.25*years))
    end = now_ist().date()
    try:
        for i, s in enumerate(symbols, 1):
            t0=time.time()
            d=hub._fetch(s, start, end)
            n=hub.store.save_candles(s,d) if d is not None and not d.empty else 0
            rows.append({"symbol":s,"bars":n,"seconds":round(time.time()-t0,2)})
            if i % 25 == 0 or i == len(symbols):
                LOG_DATA.info("deep backfill %d/%d · %s · %d bars", i, len(symbols), s, n)
    finally:
        cfg.data_source = old
    return pd.DataFrame(rows)

def load_point_in_time_membership(cfg: Config) -> pd.DataFrame:
    root=Path(cfg.pit_membership_dir)
    if not root.exists():
        return pd.DataFrame(columns=["date","symbol","source"])
    rows=[]
    for p in sorted(root.glob("*.csv")):
        try:
            d=pd.read_csv(p)
            col="symbol" if "symbol" in d.columns else d.columns[0]
            dt=pd.to_datetime(re.search(r"\d{4}[-_]?\d{2}[-_]?\d{2}",p.stem).group(0).replace("_","-")) if re.search(r"\d{4}[-_]?\d{2}[-_]?\d{2}",p.stem) else pd.NaT
            rows += [{"date":dt,"symbol":str(x).upper().strip(),"source":p.name} for x in d[col].dropna()]
        except Exception as e:
            LOG_UNIV.warning("PIT membership file skipped %s: %s",p.name,e)
    return pd.DataFrame(rows)

def survivorship_note(pit: pd.DataFrame) -> str:
    if pit.empty:
        return ("SURVIVORSHIP WARNING: historical constituent snapshots were not supplied; research uses "
                "today's screened universe as an approximation. This can overstate historical results by "
                "excluding delisted/failed names. Treat the bias as directionally positive and unresolved.")
    return f"Point-in-time membership snapshots loaded: {pit['date'].nunique()} dates, {pit['symbol'].nunique()} symbols."

07:49:11 | INFO    | engine.cell       | ▶ START  [21] Research data quality & deep backfill
07:49:11 | INFO    | engine.cell       | ✔ DONE   [21] Research data quality & deep backfill in 0.0s


## [22] OOS diagnostics and overfitting controls

These functions report MFE/MAE timing, Monte-Carlo drawdown distribution, an approximate Deflated Sharpe adjustment, CSCV-style PBO across pre-declared variants, and year/strategy/sector/exit/holding-period reports.

In [120]:
# [22] Research diagnostics
def trade_diagnostics(trades: pd.DataFrame) -> Dict[str,pd.DataFrame]:
    if trades is None or trades.empty:
        return {}
    t=trades.copy()
    t["hold_bucket"]=pd.cut(t["bars"], [0,3,5,8,12,20,999], right=True)
    return {
        "strategy": by_bucket(t,"strategy"),
        "sector": by_bucket(t,"sector"),
        "exit_reason": by_bucket(t,"reason"),
        "holding_bucket": by_bucket(t,"hold_bucket"),
        "entry_year": by_bucket(t.assign(entry_year=pd.to_datetime(t["entry_date"]).dt.year),"entry_year"),
    }

def monte_carlo_drawdown(trades: pd.DataFrame, sims: int=3000, seed: int=42) -> Dict[str,float]:
    if trades is None or trades.empty:
        return {}
    r=np.asarray(trades["r"].dropna(),dtype=float)
    if len(r)<5: return {}
    rng=np.random.default_rng(seed); dds=[]
    for _ in range(sims):
        x=rng.choice(r,size=len(r),replace=True)
        eq=np.cumsum(x); peak=np.maximum.accumulate(np.r_[0,eq])[1:]
        dds.append(float(np.min(eq-peak)))
    return {"mc_dd_median_R":round(float(np.median(dds)),2),
            "mc_dd_95pct_worst_R":round(float(np.quantile(dds,0.05)),2)}

def deflated_sharpe_approx(sharpe: float, n_returns: int, trials: int=1) -> float:
    # Conservative multiple-testing shrinkage; reported explicitly as an approximation.
    if n_returns<=2: return float("nan")
    penalty=math.sqrt(max(0.0,2*math.log(max(trials,1))/n_returns))*math.sqrt(252)
    return round(float(sharpe-penalty),3)

def cscv_pbo(score_matrix: pd.DataFrame) -> float:
    """CSCV-style PBO estimate. Rows=chronological blocks, cols=predeclared variants, higher is better."""
    x=score_matrix.dropna(axis=0,how="any")
    n=len(x)
    if n<4 or x.shape[1]<2: return float("nan")
    if n%2: x=x.iloc[:-1]; n-=1
    half=n//2; bad=0; total=0
    for train_idx in itertools.combinations(range(n),half):
        test_idx=[i for i in range(n) if i not in train_idx]
        tr=x.iloc[list(train_idx)].mean()
        winner=tr.idxmax()
        te=x.iloc[test_idx].mean().rank(pct=True)
        bad += int(te[winner] <= 0.5); total += 1
        if total>=2000: break
    return bad/max(total,1)

def mfe_mae_summary(trades: pd.DataFrame) -> pd.DataFrame:
    if trades is None or trades.empty: return pd.DataFrame()
    return trades.groupby("reason").agg(n=("r","size"), avg_r=("r","mean"),
        avg_bars=("bars","mean"), avg_mfe_r=("mfe_r","mean"), avg_mae_r=("mae_r","mean")).round(2)

def objective_gate(metrics: Dict[str,Any], cfg: Config, v1_expectancy: float=0.53) -> Tuple[bool,List[str]]:
    fails=[]
    if abs(float(metrics.get("max_dd_pct",0))) > cfg.max_drawdown_halt_pct: fails.append("drawdown")
    if int(metrics.get("n_exits",0)) < cfg.min_oos_trades: fails.append("trade_count")
    if float(metrics.get("expectancy_r",-99)) < v1_expectancy: fails.append("expectancy")
    if float(metrics.get("profit_factor",0)) < cfg.objective_min_profit_factor: fails.append("profit_factor")
    return len(fails)==0, fails

07:49:11 | INFO    | engine.cell       | ▶ START  [22] Research diagnostics
07:49:11 | INFO    | engine.cell       | ✔ DONE   [22] Research diagnostics in 0.0s


## [23] Meta-labeling (optional, filter/rank only)

The model uses only same-date rule-generated candidates and never creates trades, sets prices, or sizes positions. It stays off by default. Triple-barrier labels are built only from future outcomes after the candidate date; walk-forward training must purge/embargo those labels.

In [121]:
# [23] Optional meta-labeling
META_FEATURES=["score","adx","atr_pct","rel_63","ret_63","ret_126","dist_52w_high","vol_ratio","contraction","ext_atr","rsi14"]
def triple_barrier_label(f: pd.DataFrame, dt: pd.Timestamp, entry: float, stop: float, target_r: float=1.5, k:int=12) -> Optional[int]:
    loc=f.index.get_indexer([dt])[0]
    if loc<0 or loc+1>=len(f): return None
    r=entry-stop
    if r<=0:return None
    target=entry+target_r*r
    future=f.iloc[loc+1:loc+1+k]
    for _,b in future.iterrows():
        if float(b["low"])<=stop: return 0
        if float(b["high"])>=target: return 1
    return 0

def fit_meta_labeler(train: pd.DataFrame, seed:int=42):
    from sklearn.ensemble import HistGradientBoostingClassifier
    x=train[META_FEATURES].replace([np.inf,-np.inf],np.nan).fillna(0)
    y=train["label"].astype(int)
    model=HistGradientBoostingClassifier(max_depth=3,max_iter=120,learning_rate=0.05,
                                         l2_regularization=1.0,random_state=seed)
    model.fit(x,y)
    return model

def apply_meta_filter(cand: pd.DataFrame, model, threshold:float=0.55) -> pd.DataFrame:
    if cand.empty or model is None:return cand
    d=cand.copy()
    x=d[META_FEATURES].replace([np.inf,-np.inf],np.nan).fillna(0)
    d["meta_p"]=model.predict_proba(x)[:,1]
    return d[d["meta_p"]>=threshold].sort_values(["meta_p","score"],ascending=False)

07:49:11 | INFO    | engine.cell       | ▶ START  [23] Optional meta-labeling
07:49:11 | INFO    | engine.cell       | ✔ DONE   [23] Optional meta-labeling in 0.0s


## [24] v1-vs-v2 scoreboard and experiment gates

The baseline and v2 are run on identical data/costs. Experimental changes are enabled only when their aggregated OOS rows pass the constraints; otherwise the notebook records them as rejected and keeps baseline behavior.

In [122]:
# [24] Scoreboard / experiment helpers
V1_BASELINE = {"name":"v1 real run 2026-09-11","cagr_pct":10.2,"total_return_pct":9.2,
               "max_dd_pct":-3.1,"sharpe":1.59,"sortino":2.2,"n_exits":78,
               "hit_rate_pct":62.8,"expectancy_r":0.53,"profit_factor":1.8,
               "avg_bars_held":11.2}

def aggregate_oos(wf: pd.DataFrame) -> Dict[str,Any]:
    if wf.empty:return {}
    weighted=lambda c: float(np.average(wf[c].fillna(0),weights=wf["n_exits"].fillna(0).clip(lower=1)))
    return {"n_exits":int(wf["n_exits"].fillna(0).sum()),
            "sharpe":round(weighted("sharpe"),2),
            "expectancy_r":round(weighted("expectancy_r"),3),
            "profit_factor":round(weighted("profit_factor"),2),
            "avg_bars_held":round(weighted("avg_bars_held"),1),
            "median_bars_held":round(weighted("median_bars_held"),1),
            "return_per_capital_day_bps":round(weighted("return_per_capital_day_bps"),3),
            "max_dd_pct":round(float(wf["max_dd_pct"].min()),2),
            "cagr_pct":round(weighted("cagr_pct"),2)}

def v1_v2_scoreboard(wf: pd.DataFrame,cfg:Config) -> pd.DataFrame:
    v2m=aggregate_oos(wf); ok,fails=objective_gate(v2m,cfg)
    rows=[V1_BASELINE,{"name":"v2 OOS aggregate",**v2m,"gate_pass":ok,"gate_failures":",".join(fails)}]
    return pd.DataFrame(rows).set_index("name")

def holding_return_frontier(trades: pd.DataFrame) -> pd.DataFrame:
    if trades is None or trades.empty:return pd.DataFrame()
    rows=[]
    for cap in (5,8,10,12,15,20,30):
        x=trades[trades["bars"]<=cap]
        rows.append({"max_bars_bucket":cap,"trades":len(x),"net_pnl":x["pnl"].sum(),
                     "expectancy_r":x["r"].mean() if len(x) else np.nan,
                     "capital_days":(x["bars"]*x["entry"]*x["qty"]).sum() if len(x) else 0})
    d=pd.DataFrame(rows)
    d["pnl_per_capital_day_bps"]=d["net_pnl"]/d["capital_days"].replace(0,np.nan)*10000
    return d.round(3)

07:49:12 | INFO    | engine.cell       | ▶ START  [24] Scoreboard / experiment helpers
07:49:12 | INFO    | engine.cell       | ✔ DONE   [24] Scoreboard / experiment helpers in 0.0s


## [25] Daily morning run and paper-forward journal

The morning run refreshes the tail, reconciles broker positions (broker truth wins), manages exits, updates regime, and produces clean plans. Live order submission remains behind both `dry_run=True` and approval.

In [123]:
# [25] Morning run / journal
def reconcile_broker_truth(engine: SwingEngine) -> Dict[str,Any]:
    try:
        broker_pos=engine.broker.positions() or []
    except Exception as e:
        engine.store.event("RECONCILE_FAIL",str(e),level="WARN")
        return {"ok":False,"error":str(e)}
    # Adapter-normalized position shapes vary; reconciliation is deliberately non-destructive in PAPER.
    if engine.cfg.mode!="LIVE":
        return {"ok":True,"broker_positions":len(broker_pos),"changed":0}
    bysym={str(x.get("symbol","")).upper():x for x in broker_pos if isinstance(x,dict)}
    changed=0
    for tid,p in list(engine.lc.positions.items()):
        bp=bysym.get(p.symbol.upper())
        if bp is None:
            engine.store.event("RECONCILE_MISSING","Broker has no corresponding open position",
                               level="WARN",symbol=p.symbol,trade_id=tid)
            continue
        qty=int(bp.get("qty",bp.get("quantity",p.qty)) or 0)
        if qty!=p.qty:
            p.qty=qty; changed+=1
            engine.store.event("RECONCILE_QTY",f"broker truth qty={qty}",symbol=p.symbol,trade_id=tid)
    return {"ok":True,"broker_positions":len(broker_pos),"changed":changed}

def log_recommendations(store:Store, plans:Sequence[TradePlan]) -> None:
    for p in plans:
        store.upsert("journal",{"id":p.id,"ts":now_ist().isoformat(),"kind":"RECOMMENDATION",
            "symbol":p.symbol,"strategy":p.strategy,"planned_entry":p.entry_trigger,"actual_entry":None,
            "planned_stop":p.init_stop,"qty":p.qty,"slippage_bps":None,"status":"PLANNED",
            "payload":json.dumps(p.to_row(),default=str)})

def morning_run(engine:SwingEngine, refresh:bool=True) -> pd.DataFrame:
    rec=reconcile_broker_truth(engine)
    LOG_LC.info("reconcile: %s",rec)
    if engine.universe.empty:
        engine.refresh_universe(refresh_data=refresh)
    plans=engine.eod_scan(refresh_data=refresh)
    log_recommendations(engine.store,plans)
    rows=[]
    for p in plans:
        rows.append({"symbol":p.symbol,"strategy":p.strategy,"entry_type":p.entry_type,
                     "trigger":p.entry_trigger,"stop":p.init_stop,"T1":p.t1,"T2":p.t2,
                     "qty":p.qty,"risk_rupees":p.risk_amount,"valid_till":p.valid_till,
                     "reason":p.thesis,"warnings":("WIDE_STOP" if p.stop_pct>6 else "")})
    out=pd.DataFrame(rows)
    if not out.empty: display(out)
    return out

07:49:12 | INFO    | engine.cell       | ▶ START  [25] Morning run / journal
07:49:12 | INFO    | engine.cell       | ✔ DONE   [25] Morning run / journal in 0.0s


## [26] In-notebook tests

These asserts cover lookahead invariance, cost behavior, exit edge cases, circuit lock, sizing caps, and state transitions. SMOKE mode runs them before the backtest.

In [124]:
# [26] Tests
def run_v2_tests(cfg:Config) -> Dict[str,str]:
    results={}
    # cost tests
    assert transaction_costs(500,10,Txn.BUY,cfg)["total"]>0
    assert transaction_costs(500,10,Txn.SELL,cfg)["total"]>0
    results["cost_model"]="PASS"

    # exit edge cases
    p=Position("T","X","TEST",10,10,100,90,90,115,130,10,"2026-01-01")
    assert exit_decisions(p,{"open":85,"high":92,"low":84,"close":90},2,cfg)[0].reason=="STOP_GAP"
    p2=copy.deepcopy(p)
    a=exit_decisions(p2,{"open":100,"high":120,"low":89,"close":110},2,cfg)
    assert a[0].reason=="STOP"  # conservative stop-before-target
    p3=copy.deepcopy(p)
    assert exit_decisions(p3,{"open":90,"high":90,"low":90,"close":90,"lower_circuit_locked":True},2,cfg)==[]
    results["exit_decisions"]="PASS"

    # sizing stop cap
    bad=size_position(100,100*(1-(cfg.max_stop_pct+1)/100),cfg.equity,cfg.equity,1e6,cfg)
    assert not bad.ok and bad.binding_cap=="stop_too_wide"
    results["sizing_caps"]="PASS"

    # no explicit negative shifts in feature/strategy source
    source=FEATURE_SOURCE_FOR_TEST+"\n"+STRATEGY_SOURCE_FOR_TEST
    assert not re.search(r"shift\(\s*-\s*\d+",source)
    results["negative_shift_scan"]="PASS"

    # truncate future: features at t must be identical
    sm=SyntheticMarket(n_symbols=2,years=2,seed=123)
    d=sm.daily(sm.symbols[0]).copy()
    full=compute_features(d)
    t=d.index[-30]
    trunc=compute_features(d.loc[:t])
    common=[c for c in ("ema20","ema50","rsi14","atr","ret_63") if c in full]
    for c in common:
        a0,b0=full.loc[t,c],trunc.loc[t,c]
        if pd.isna(a0) and pd.isna(b0):continue
        assert np.isclose(a0,b0,equal_nan=True)
    results["truncate_lookahead"]="PASS"
    return results

07:49:12 | INFO    | engine.cell       | ▶ START  [26] Tests
07:49:12 | INFO    | engine.cell       | ✔ DONE   [26] Tests in 0.0s


---
# Part II · Run it

`SMOKE` is the default and requires no credentials. `RESEARCH` uses deep cached history and keeps the final holdout locked. `LIVE` performs the morning workflow with safe execution defaults.

In [ ]:
# [R0] Mode dispatcher
def prepare_engine_data(cfg:Config, refresh:bool=True):
    global ENGINE
    if cfg.run_mode=="SMOKE":
        # rebuild globals with the configured synthetic tape so every path is credential-free
        ENGINE.universe=build_universe(cfg,HUB,BROKER,SYNTH,refresh=refresh)
    else:
        ENGINE.universe=build_universe(cfg,HUB,BROKER,SYNTH,refresh=refresh)
    ENGINE.panel, ENGINE.bench = compute_panel(cfg,HUB,ENGINE.universe,refresh=refresh)
    return ENGINE.universe,ENGINE.panel,ENGINE.bench

def run_smoke(cfg:Config=CFG) -> Dict[str,Any]:
    assert cfg.run_mode=="SMOKE"
    tests=run_v2_tests(cfg)
    uni,panel,bench=prepare_engine_data(cfg,refresh=False)
    dates=_all_dates(panel)
    start=dates[max(cfg.warmup_bars, len(dates)//3)]
    end=dates[-1]
    bt=Backtester(cfg,panel,uni,bench)
    res=bt.run(str(start.date()),str(end.date()),verbose=False)
    wf=walk_forward(cfg,panel,uni,bench,folds=min(5,cfg.wf_folds),exclude_final_holdout=False)
    return {"tests":tests,"metrics":res["metrics"],"wf":wf,"trades":res["trades"]}

def run_research(cfg:Config=CFG, unlock_holdout:bool=False) -> Dict[str,Any]:
    assert cfg.run_mode=="RESEARCH"
    uni=build_universe(cfg,HUB,BROKER,SYNTH,refresh=True)
    deep_backfill_v2(cfg,HUB,uni["symbol"].tolist(),years=9)
    panel,bench=compute_panel(cfg,HUB,uni,refresh=False)
    dq=data_quality_report(panel)
    pit=load_point_in_time_membership(cfg)
    print(survivorship_note(pit))
    anchored=walk_forward(cfg,panel,uni,bench,variant="ANCHORED")
    rolling=walk_forward(cfg,panel,uni,bench,variant="ROLLING")
    board=v1_v2_scoreboard(anchored,cfg)
    hold=locked_holdout(cfg,panel,uni,bench,unlock=unlock_holdout)
    return {"universe":uni,"panel":panel,"bench":bench,"data_quality":dq,
            "wf_anchored":anchored,"wf_rolling":rolling,"scoreboard":board,"holdout":hold}

def run_live(cfg:Config=CFG) -> pd.DataFrame:
    assert cfg.run_mode=="LIVE"
    assert cfg.dry_run is True and cfg.require_approval is True
    return morning_run(ENGINE,refresh=True)

RUN_RESULT = None
if CFG.run_mode=="SMOKE":
    RUN_RESULT=run_smoke(CFG)
    print("SMOKE tests:",RUN_RESULT["tests"])
    print("SMOKE metrics:",RUN_RESULT["metrics"])
    display(RUN_RESULT["wf"])
elif CFG.run_mode=="RESEARCH":
    RUN_RESULT=run_research(CFG,unlock_holdout=False)
    display(RUN_RESULT["scoreboard"])
    display(RUN_RESULT["wf_anchored"])
else:
    RUN_RESULT=run_live(CFG)

07:49:12 | INFO    | engine.cell       | ▶ START  [R0] Mode dispatcher
07:49:12 | WARNING | engine.events     | [RECONCILE_FAIL] The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
07:49:12 | INFO    | engine.lifecycle  | reconcile: {'ok': False, 'error': 'The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().'}
07:49:12 | INFO    | engine            | → job: refresh_universe ...
07:49:12 | INFO    | engine.universe   | building universe | source=GROWW prefilter=QUOTE
07:49:14 | INFO    | engine.broker     | instrument master: 136779 rows in 2.2s
07:49:14 | INFO    | engine.broker     |   by segment: {'FNO': 83259, 'COMMODITY': 40805, 'CASH': 12715}
07:49:14 | INFO    | engine.broker     |   by instrument_type: {'PE': 61555, 'CE': 61541, 'EQ': 12684, 'FUT': 968, 'IDX': 31}
07:49:15 | INFO    | engine.universe   |   gate exchange == NSE              136779 → 108399
07:49:15 | INFO    | engine.univer

---
# Part III · Going live

Before changing `dry_run`, verify current Groww SDK method names/response shapes, fee schedules, NSE surveillance sources and any applicable SEBI/broker retail-algo requirements. The adapter remains the only broker-specific layer; uncertain external behavior should be marked `# VERIFY` rather than guessed.

## CHANGELOG — v1 → v2

- Fixed global warm-up bug: symbol eligibility is now per-symbol; recent listings no longer collapse walk-forward.
- `bt_end` defaults to the latest available bar.
- Research history target increased to ~9 calendar years, cache-first and resumable on Drive.
- Yahoo research downloads use adjusted OHLC to reduce split/bonus discontinuities; suspicious >35% single-day moves are flagged.
- Added lower-circuit-like lock handling so exits are carried instead of filled at an impossible stop.
- Fixed trade-level net P&L accounting to include entry-side as well as exit-side fees.
- Capital changed to ₹50,000; max tolerated drawdown to 15%; natural stop distance hard-capped at 8%.
- Added early failure-to-follow-through exit while preserving a right-tail path.
- Replaced the 3-step exposure multiplier with a continuous breadth/trend/drawdown/volatility dial while retaining compatibility labels.
- Added optional 52-week momentum and episodic-pivot edges, disabled until OOS validation passes.
- Added purged/embargoed anchored and rolling walk-forward, locked final holdout, Monte Carlo DD, DSR approximation, CSCV/PBO helper, objective gates and v1-v2 scoreboard.
- Added optional gradient-boosted meta-labeling scaffold restricted to filtering/ranking existing rule-based candidates.
- Added paper-forward recommendation journal and broker-truth reconciliation.
- Added SMOKE / RESEARCH / LIVE top-level modes and deterministic in-notebook tests.

## Known limitations

- True point-in-time Nifty Smallcap/Microcap membership depends on historical constituent snapshots. If those are not supplied, the notebook explicitly reports survivorship bias instead of pretending it is solved.
- Circuit-lock inference from OHLC is only a fallback. For production use, merge official daily price-band/surveillance files; regulatory/source endpoints can change and should be verified.
- The Deflated Sharpe function is a conservative approximation, while CSCV/PBO is meaningful only when fed a fixed pre-declared matrix of genuinely out-of-sample variant scores.
- The optional meta-label model is off by default and must be trained only inside purged walk-forward folds.
- Broker SDK response shapes, live fee rates, tax schedules and regulatory requirements can change; the notebook does not silently invent them.